# Colab CPU — tạo hàng đợi embedding 4 phiên Colab, mỗi phiên 1 GPU

Chạy một lần trên Colab CPU. Kế hoạch mới giữ riêng task của Kaggle acc 2
đang chạy trong plan 63b6fe57cbf4; phần của các acc đã nhập và task chưa
được giao sẽ sang 4 Colab worker. Notebook chỉ đọc manifest/key index đã
lưu trên Drive để lập lịch; không encode, không đọc lại corpus.

Trước khi chạy, thêm Colab Secret `KAGGLE_JSON`, bật Notebook access,
rồi Run all. Có thể điền handle Outputs acc 2 nếu phiên đã lưu xong.
Kế hoạch và file thực thi khóa SHA được ghi trên Drive. Không chạy lại cell
này sau khi worker đã bắt đầu để giữ nguyên task assignment.

In [ ]:
#@title 1. Colab CPU — kết nối Drive và tải mã khóa
import copy
import importlib
import json
import os
from pathlib import Path
import re
import subprocess
import sys
if not Path('/var/colab/hostname').is_file():
    raise RuntimeError('Notebook chuyển dữ liệu này chạy trên Colab CPU. Kaggle dùng 04_new_data_account_0 hoặc 1.')
print('COLAB PLAN: dùng manifest frozen trên Drive; không export corpus hay nạp model.', flush=True)
KAGGLE_ACCOUNTS = 4 #@param {type:"integer"}
MIGRATE_UNSTARTED_LEGACY = True #@param {type:"boolean"}
KAGGLE_NOTEBOOK_OUTPUT = "" #@param {type:"string"}
# Để trống để xuất gói. Sau GPU: owner/notebook[,owner2/notebook2] để nhập Outputs.
from google.colab import drive, auth, userdata
drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/VietMedBridge')
relative = 'data_staging/btc-maximize-71be0e12-v1'
if not (PROJECT/relative/'delta_candidates.json').is_file():
    PROJECT = Path('/content/drive/.shortcut-targets-by-id/1uKKHYEcMPodnJIscKPi6jsoNBJvyhFGK/VietMedBridge')
PROJECT = PROJECT.resolve()
lock = {'repo_url':'https://github.com/Platypus27-coder/VietMedBridge.git','commit':'bbf09ec01d542a5b06a4c2beec245571abeb3515'}
REPO_URL = 'https://github.com/Platypus27-coder/VietMedBridge.git'
if lock.get('repo_url') != REPO_URL or not re.fullmatch('[0-9a-f]{40}', lock.get('commit', '')):
    raise ValueError('Intake producer code lock không hợp lệ.')
try:
    credentials = json.loads(userdata.get('KAGGLE_JSON'))
    owner, api_key = credentials['username'], credentials['key']
    if not re.fullmatch('[A-Za-z0-9_-]+', owner) or not api_key:
        raise ValueError()
except Exception:
    raise RuntimeError('Thêm Colab Secret KAGGLE_JSON chứa nội dung kaggle.json và bật Notebook access.') from None
os.environ['KAGGLE_USERNAME'], os.environ['KAGGLE_KEY'] = owner, api_key
del credentials, api_key
CHECKOUT = Path('/content/VietMedBridge-full-kaggle')
WORK_DIR = Path('/content/vmb_full_kaggle_cpu')
WORK_DIR.mkdir(parents=True, exist_ok=True)
if not (CHECKOUT/'.git').is_dir():
    subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(CHECKOUT)], check=True)
if subprocess.check_output(['git', '-C', str(CHECKOUT), 'remote', 'get-url', 'origin'], text=True).strip() != REPO_URL:
    raise ValueError('Checkout thuộc repo khác.')
subprocess.run(['git', '-C', str(CHECKOUT), 'fetch', '--depth', '1', 'origin', lock['commit']], check=True)
subprocess.run(['git', '-C', str(CHECKOUT), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', 'numpy', 'kaggle>=2.2.2', 'kagglehub==1.0.3'], cwd=CHECKOUT, check=True)
sys.path[:0] = [str(WORK_DIR), str(CHECKOUT/'src')]
importlib.invalidate_caches()
auth.authenticate_user()

In [ ]:
#@title 2. Kết nối manifest frozen và cache đã có — CPU, không mở vector payload
SOURCES = json.loads('{"new_data_kaggle.py": "\\"\\"\\"Sparse, resumable Kaggle handoff for the isolated frozen new-data batch.\\n\\nCPU selection opens each frozen part once, saves its draft references and only\\nexports uncached texts. Neither export nor import changes the submission pointer.\\n\\"\\"\\"\\nfrom pathlib import Path\\nfrom types import SimpleNamespace\\nimport argparse\\nimport copy\\nimport json\\nimport os\\nimport re\\nimport signal\\nimport subprocess\\nimport sys\\nimport time\\n\\nimport numpy as np\\nimport pyarrow as pa\\nimport pyarrow.parquet as pq\\n\\nfrom vietmedbridge.artifacts import atomic_json, digest_json, publish_file, read_json, sha256_file, verify_file, utc_now\\nfrom vietmedbridge.content_embeddings import ContentBlocks, checked, seal, key\\nfrom vietmedbridge.embeddings import validate_vectors\\nfrom vietmedbridge.portable_embeddings import kaggle_assignment, validate_encoder, relative_path\\nfrom vietmedbridge.scale_vectors import bound\\ntry:\\n    from .new_data_embeddings import NewDataEmbeddings, LazyEncoder, embedding_contract, FAMILIES\\n    from .new_data_cache_lookup import ResumableContentBlocks\\nexcept ImportError:\\n    from new_data_embeddings import NewDataEmbeddings, LazyEncoder, embedding_contract, FAMILIES\\n    from new_data_cache_lookup import ResumableContentBlocks\\n\\nSOURCE_FILES = (\'new_data_kaggle.py\', \'new_data_embeddings.py\', \'new_data_cache_lookup.py\', \'new_data_drive_api.py\')\\nTASK_ROWS = 512\\n\\n\\ndef lookup(blocks, keys):\\n    known = {}\\n    keys = list(dict.fromkeys(keys))\\n    for start in range(0, len(keys), 900):\\n        selected = keys[start:start+900]\\n        rows = blocks.con.execute(\'SELECT key,path,sha,row FROM vectors WHERE key IN (\'+\\n                                  \',\'.join(\'?\' for _ in selected)+\')\', selected)\\n        known.update({k: {\'path\': p, \'sha256\': s, \'row\': r} for k, p, s, r in rows})\\n    return known\\n\\n\\ndef copied(source, target, checksum):\\n    if Path(target).is_file():\\n        verify_file(target, checksum)\\n    elif publish_file(source, target) != checksum:\\n        raise ValueError(\'Artifact changed during copy.\')\\n\\n\\nclass NewDataKaggle:\\n    def __init__(self, project, work, identities, *, accounts=2, gpu_count=2, batch_loader=None):\\n        self.assignment = kaggle_assignment(accounts=accounts, account_id=0, gpu_count=gpu_count)\\n        self.view = NewDataEmbeddings(project, work, identities, batch_loader=batch_loader)\\n        self.work = Path(work)\\n        self.team = {\'accounts\': accounts, \'gpu_count\': gpu_count}\\n        self.root = self.view.output / f\'kaggle-{accounts}x{gpu_count}\'\\n        self.identity = {\'schema\': \'new-data-selection-v1\', \'embedding_binding\': self.view.binding, \'team\': self.team}\\n        self.binding = digest_json(self.identity)\\n        contract = self.root / \'contract.json\'\\n        if contract.exists() and checked(read_json(contract)) != seal(self.identity):\\n            raise ValueError(\'Selection producer/team changed.\')\\n        atomic_json(contract, seal(self.identity))\\n        self.sources = {s[\'ordinal\']: s for s in self.view.sources}\\n\\n    def journals(self, family):\\n        result = {}\\n        for build in dict.fromkeys(s[\'build\'][\'build_run\'] for s in self.view.sources):\\n            path = self.root / \'selection\' / family / build / \'progress.json\'\\n            if not path.exists():\\n                continue\\n            saved = checked(read_json(path))\\n            if saved.get(\'binding\') != self.binding or saved.get(\'family\') != family:\\n                raise ValueError(\'Selection checkpoint belongs to another batch/model/team.\')\\n            for ordinal, record in saved[\'parts\'].items():\\n                source = self.sources[int(ordinal)]\\n                if (source[\'build\'][\'build_run\'] != build or record[\'input_sha256\'] != source[\'part\'][\'sha256\']\\n                        or record[\'ordinal\'] != int(ordinal) or record[\'rows\'] != source[\'part\'][\'rows\']):\\n                    raise ValueError(\'Selection checkpoint input binding changed.\')\\n                result[int(ordinal)] = record\\n        return result\\n\\n    def _save(self, family, source, selected):\\n        build = source[\'build\'][\'build_run\']\\n        record = {\'binding\': self.binding, \'family\': family, \'parts\': {\\n            str(k): v for k, v in selected.items() if self.sources[k][\'build\'][\'build_run\'] == build}}\\n        atomic_json(self.root / \'selection\' / family / build / \'progress.json\', seal(record))\\n\\n    def _map(self, family, source, refs):\\n        root, identity = self.view._view(family, source)\\n        config = root / \'config.json\'\\n        if config.exists() and read_json(config) != identity:\\n            raise ValueError(\'Embedding map producer changed.\')\\n        atomic_json(config, identity)\\n        part = source[\'part\']\\n        target = root / f\\"map-{part[\'start\']:012d}.parquet\\"\\n        local = self.work / \'map.parquet\'\\n        pq.write_table(pa.Table.from_pylist(refs), local, compression=\'zstd\')\\n        record = seal({\'signature\': digest_json(identity), \'start\': part[\'start\'], \'rows\': part[\'rows\'],\\n                       \'input_sha256\': part[\'sha256\'], \'path\': target.name, \'sha256\': publish_file(local, target)})\\n        atomic_json(root / f\\"map-{part[\'start\']:012d}.done.json\\", record)\\n        return record\\n\\n    def select(self, *, receipt_api=None, blocks_factory=ResumableContentBlocks):\\n        \\"\\"\\"Save draft refs and missing-text tasks before committing each input part.\\"\\"\\"\\n        for family in FAMILIES:\\n            selected = self.journals(family)\\n            complete = self.view._journals(family)\\n            pending = [s for s in self.view.sources if s[\'ordinal\'] not in selected and self.view._key(s) not in complete]\\n            print(f\'SELECTION {family}: {len(selected)} selection parts saved | {len(complete)}/{len(self.sources)} maps complete; {len(pending)} new\', flush=True)\\n            if not pending:\\n                continue\\n            expected = self.view.identities[family]\\n            blocks = blocks_factory(self.view.data, SimpleNamespace(identity=expected[\'encoder\'], dimension=expected[\'dimension\']),\\n                                    self.work / family, receipt_api=receipt_api, progress=print)\\n            try:\\n                for source in pending:\\n                    part, ordinal = source[\'part\'], source[\'ordinal\']\\n                    path = bound(self.view.project, source[\'build\'][\'build_dir\']) / \'index_inputs\' / part[\'path\']\\n                    local_input = self.work / \'current-input.parquet\'\\n                    if publish_file(path, local_input) != part[\'sha256\']:\\n                        raise ValueError(\'Frozen input checksum mismatch during staging.\')\\n                    rows = pq.read_table(local_input, columns=[\'id\', \'text\']).to_pylist()\\n                    if len(rows) != part[\'rows\'] or any(r[\'id\'] != key(r[\'text\']) for r in rows):\\n                        raise ValueError(\'Frozen input row/hash mismatch.\')\\n                    known = lookup(blocks, [r[\'id\'] for r in rows])\\n                    draft = [{\'key\': r[\'id\'], **known.get(r[\'id\'], {\'path\': None, \'sha256\': None, \'row\': None})} for r in rows]\\n                    directory = self.root / \'selection\' / family / source[\'build\'][\'build_run\']\\n                    local = self.work / \'selection.parquet\'\\n                    schema = pa.schema([(\'key\', pa.string()), (\'path\', pa.string()), (\'sha256\', pa.string()), (\'row\', pa.int64())])\\n                    pq.write_table(pa.Table.from_pylist(draft, schema=schema), local, compression=\'zstd\')\\n                    draft_path = directory / f\'draft-{ordinal:06d}.parquet\'\\n                    draft_entry = {\'path\': draft_path.relative_to(self.root).as_posix(), \'sha256\': publish_file(local, draft_path)}\\n                    tasks, missing_rows = [], []\\n                    missing = [r for r in rows if r[\'id\'] not in known]\\n                    target = directory / f\'missing-{ordinal:06d}.parquet\'\\n                    for worker in range(self.assignment[\'workers\']):\\n                        group = [r for r in missing if int(r[\'id\'][:8], 16) % self.assignment[\'workers\'] == worker]\\n                        for offset in range(0, len(group), TASK_ROWS):\\n                            values = group[offset:offset+TASK_ROWS]\\n                            task_id = digest_json([self.binding, family, ordinal, worker, offset])\\n                            tasks.append({\'selection_id\': task_id, \'family\': family, \'ordinal\': ordinal, \'worker\': worker,\\n                                \'path\': target.relative_to(self.root).as_posix(), \'offset\': len(missing_rows),\\n                                \'rows\': len(values), \'keys_digest\': digest_json(sorted(r[\'id\'] for r in values))})\\n                            missing_rows.extend(values)\\n                    if missing_rows:\\n                        pq.write_table(pa.Table.from_pylist(missing_rows), local, compression=\'zstd\')\\n                        checksum = publish_file(local, target)\\n                        for task in tasks:\\n                            task[\'sha256\'] = checksum\\n                    selected[ordinal] = {\'ordinal\': ordinal, \'input_sha256\': part[\'sha256\'], \'rows\': part[\'rows\'],\\n                                         \'cached_rows\': len(rows)-len(missing), \'draft\': draft_entry, \'tasks\': tasks}\\n                    if not missing and self.view._key(source) not in complete:\\n                        complete[self.view._key(source)] = self._map(family, source, [known[r[\'id\']] for r in rows])\\n                        self.view._save(family, complete)\\n                    self._save(family, source, selected)\\n                    print(f\'SELECTION_CHECKPOINT {family} {len(selected)}/{len(self.sources)} | cached {len(rows)-len(missing)}, missing {len(missing)}\', flush=True)\\n                self.view._seal_views(family, complete)\\n            finally:\\n                blocks.close()\\n        return {f: self.journals(f) for f in FAMILIES}\\n\\n    def imported(self):\\n        done = set()\\n        for path in (self.root / \'imports\').glob(\'*.json\'):\\n            saved = checked(read_json(path))\\n            if saved.get(\'binding\') != self.binding:\\n                raise ValueError(\'Imported output binding changed.\')\\n            done.update(saved[\'selection_ids\'])\\n        return done\\n\\n    def complete_maps(self, family, ordinals, blocks):\\n        selected, completed = self.journals(family), self.view._journals(family)\\n        for ordinal in sorted(ordinals):\\n            source = self.sources[ordinal]\\n            if self.view._key(source) in completed:\\n                continue\\n            entry = selected[ordinal][\'draft\']\\n            path = bound(self.root, entry[\'path\'])\\n            verify_file(path, entry[\'sha256\'])\\n            draft = pq.read_table(path).to_pylist()\\n            if len(draft) != source[\'part\'][\'rows\']:\\n                raise ValueError(\'Draft map row coverage changed.\')\\n            missing = [r[\'key\'] for r in draft if r[\'path\'] is None]\\n            known = lookup(blocks, missing)\\n            if all(k in known for k in missing):\\n                refs = [{k: r[k] for k in (\'path\', \'sha256\', \'row\')} if r[\'path\'] is not None else known[r[\'key\']] for r in draft]\\n                completed[self.view._key(source)] = self._map(family, source, refs)\\n                self.view._save(family, completed)\\n        self.view._seal_views(family, completed)\\n\\n    def export(self, checkout, *, config, runtime, code_commit, receipt_api=None,\\n               blocks_factory=ResumableContentBlocks, output_bytes=16_000_000_000):\\n        active = self.root/\'active_round.json\'\\n        if active.exists():\\n            saved = checked(read_json(active))\\n            if saved[\'binding\'] != self.binding:\\n                raise ValueError(\'Active Kaggle round binding changed.\')\\n            imported = {r[\'account_id\'] for p in (self.root/\'imports\').glob(\'*.json\')\\n                        if (r := checked(read_json(p))).get(\'job\') == saved[\'job\'] and \'account_id\' in r}\\n            if imported != set(range(self.team[\'accounts\'])):\\n                return None, {\'state\': \'WAITING_FOR_KAGGLE_OUTPUTS\', \'dataset_handle\': saved[\'dataset_handle\'],\\n                    \'imported_accounts\': sorted(imported), \'required_accounts\': list(range(self.team[\'accounts\'])),\\n                    \'instruction\': \'Use the existing dataset; import saved Outputs of the current round before starting another round.\'}\\n        selected = self.select(receipt_api=receipt_api, blocks_factory=blocks_factory)\\n        if type(output_bytes) is not int or not 0 < output_bytes <= 16_000_000_000:\\n            raise ValueError(\'Output budget must be between 1 and 16 GB per account.\')\\n        if not re.fullmatch(\'[0-9a-f]{40}\', code_commit):\\n            raise ValueError(\'Pin the full native producer commit.\')\\n        done = self.imported()\\n        used, tasks, remaining = [0]*self.team[\'accounts\'], [], {f: 0 for f in FAMILIES}\\n        assets = self.work/\'export-assets\'\\n        assets.mkdir(parents=True, exist_ok=True)\\n        for family in FAMILIES:\\n            pending = [task for entry in selected[family].values() for task in entry[\'tasks\'] if task[\'selection_id\'] not in done]\\n            remaining[family] = len(pending)\\n            if not pending:\\n                continue\\n            expected = self.view.identities[family]\\n            blocks = blocks_factory(self.view.data, SimpleNamespace(identity=expected[\'encoder\'], dimension=expected[\'dimension\']),\\n                                    self.work/family, receipt_api=receipt_api, progress=print)\\n            cache_only, affected = [], set()\\n            last_path, last_rows = None, None\\n            try:\\n                for task in pending:\\n                    account = task[\'worker\'] // self.team[\'gpu_count\']\\n                    estimate = task[\'rows\'] * (expected[\'dimension\']*4+1024)\\n                    if estimate > output_bytes:\\n                        raise ValueError(\'One task exceeds the round output budget.\')\\n                    if used[account]+estimate > output_bytes:\\n                        continue\\n                    path = bound(self.root, task[\'path\'])\\n                    if path != last_path:\\n                        verify_file(path, task[\'sha256\'])\\n                        last_rows = pq.read_table(path, columns=[\'id\', \'text\']).to_pylist()\\n                        last_path = path\\n                    rows = last_rows[task[\'offset\']:task[\'offset\']+task[\'rows\']]\\n                    if len(rows) != task[\'rows\'] or digest_json(sorted(r[\'id\'] for r in rows)) != task[\'keys_digest\']:\\n                        raise ValueError(\'Saved missing-text task changed.\')\\n                    known = lookup(blocks, [r[\'id\'] for r in rows])\\n                    rows = [r for r in rows if r[\'id\'] not in known]\\n                    if not rows:\\n                        cache_only.append(task[\'selection_id\'])\\n                        affected.add(task[\'ordinal\'])\\n                        remaining[family] -= 1\\n                        continue\\n                    estimate = len(rows) * (expected[\'dimension\']*4+1024)\\n                    used[account] += estimate\\n                    asset = \'task-\'+task[\'selection_id\']+\'.parquet\'\\n                    pq.write_table(pa.Table.from_pylist(rows), assets/asset, compression=\'zstd\')\\n                    tasks.append({**task, \'id\': task[\'selection_id\'], \'asset\': asset, \'rows\': len(rows),\\n                        \'sha256\': sha256_file(assets/asset), \'keys_digest\': digest_json(sorted(r[\'id\'] for r in rows)),\\n                        \'estimated_output_bytes\': estimate})\\n                if cache_only:\\n                    self.complete_maps(family, affected, blocks)\\n                    atomic_json(self.root/\'imports\'/(\'cache-\'+digest_json(cache_only)+\'.json\'), seal({\\n                        \'binding\': self.binding, \'selection_ids\': cache_only, \'state\': \'REUSED_NEWLY_AVAILABLE_CACHE\'}))\\n            finally:\\n                blocks.close()\\n        if not tasks:\\n            return None, {\'state\': \'NO_PENDING_KAGGLE_TASKS\', \'coverage\': self.coverage()}\\n        model_code = {n: sha256_file(Path(checkout)/\'src/vietmedbridge\'/n) for n in (\'retrieval_models.py\', \'qwen_models.py\')}\\n        scripts = Path(__file__).resolve().parent\\n        workers = {n: sha256_file(scripts/n) for n in SOURCE_FILES}\\n        frozen_sources = {source[\'build\'][\'build_run\']: source[\'build\'] for source in self.view.sources}\\n        job = seal({\'schema\': \'new-data-kaggle-v1\', \'selection_binding\': self.binding, \'batch\': self.view.batch[\'manifest_sha256\'],\\n            \'source_batch_dir\': self.view.root.relative_to(self.view.project).as_posix(),\\n            \'coverage_scope\': {k: self.view.batch[k] for k in (\'official_ids\',\'source_builds\',\'source_documents_sum\',\'input_rows_before_cross_source_dedup\',\'scope_manifest_sha256\') if k in self.view.batch},\\n            \'frozen_sources\': [{\'build_run\': b[\'build_run\'], \'candidate_manifest_sha256\': b[\'candidate_manifest_sha256\'],\\n                \'index_inputs_manifest_sha256\': b[\'index_inputs_manifest_sha256\'], \'unique_dense_inputs\': b[\'unique_dense_inputs\'],\\n                \'inherited_complete_models\': sorted(b.get(\'completed_views\', {}))} for b in frozen_sources.values()],\\n            \'scheduled_tasks\': len(tasks), \'unscheduled_tasks\': sum(remaining.values())-len(tasks),\\n            \'all_remaining_tasks_scheduled\': sum(remaining.values()) == len(tasks),\\n            \'team\': self.team, \'code_commit\': code_commit, \'repo_url\': \'https://github.com/Platypus27-coder/VietMedBridge.git\',\\n            \'models\': self.view.identities, \'config\': config, \'runtime\': runtime, \'model_code\': model_code,\\n            \'worker_sources\': workers, \'tasks\': tasks, \'remaining_tasks_before_round\': remaining,\\n            \'scheduled_bytes_per_account\': used, \'scope\': self.view.batch.get(\'scope\',\'Frozen corpus child embedding tasks only; no submission inference.\')})\\n        target = self.work / \'exports\' / job[\'manifest_sha256\'][:16]\\n        target.mkdir(parents=True, exist_ok=True)\\n        for i, task in enumerate(tasks, 1):\\n            copied(assets/task[\'asset\'], target/task[\'asset\'], task[\'sha256\'])\\n            if i % 100 == 0:\\n                print(f\'KAGGLE_EXPORT {i}/{len(tasks)} missing-text tasks\', flush=True)\\n        for name, checksum in workers.items():\\n            copied(scripts/name, target/name, checksum)\\n        atomic_json(target/\'job.json\', job)\\n        atomic_json(self.root/\'exports\'/(\'job-\'+job[\'manifest_sha256\']+\'.json\'), job)\\n        if \'official_ids\' in self.view.batch:\\n            self.coverage()\\n        return target, job\\n\\n    def coverage(self):\\n        coverage = {f: {\'completed_parts\': len(self.view._journals(f)), \'total_parts\': len(self.sources)} for f in FAMILIES}\\n        if \'official_ids\' in self.view.batch:\\n            complete = all(v[\'completed_parts\'] == v[\'total_parts\'] for v in coverage.values())\\n            atomic_json(self.root/\'full_embedding_status.json\', seal({\\n                \'state\': \'FULL_DATA_EMBEDDINGS_COMPLETE\' if complete else \'FULL_DATA_EMBEDDINGS_IN_PROGRESS\',\\n                \'official_ids_in_scope\': self.view.batch[\'official_ids\'], \'batch\': self.view.batch[\'manifest_sha256\'],\\n                \'coverage\': coverage, \'payload_verification\': \'SHA256_ON_USE\', \'updated_at\': utc_now()}))\\n        return coverage\\n\\n    def assigned_output_tasks(self, job, result):\\n        assignment = kaggle_assignment(accounts=job[\'team\'][\'accounts\'], account_id=result[\'account_id\'],\\n                                       gpu_count=job[\'team\'][\'gpu_count\'])\\n        return {t[\'id\'] for t in job[\'tasks\'] if t[\'worker\'] in assignment[\'worker_ids\']}\\n\\n    def install(self, output, *, receipt_api=None, blocks_factory=ResumableContentBlocks, cached_keys=None):\\n        output = Path(output)\\n        job = checked(read_json(output/\'job.json\'))\\n        saved_job = checked(read_json(self.root/\'exports\'/(\'job-\'+job[\'manifest_sha256\']+\'.json\')))\\n        result = checked(read_json(output/\'new-data-result.json\'))\\n        if job != saved_job or job[\'selection_binding\'] != self.binding or job[\'models\'] != self.view.identities or result[\'job\'] != job[\'manifest_sha256\']:\\n            raise ValueError(\'Kaggle output differs from the frozen export/model contract.\')\\n        prior_path = self.root/\'imports\'/(result[\'manifest_sha256\']+\'.json\')\\n        if prior_path.exists():\\n            prior = checked(read_json(prior_path))\\n            if (prior.get(\'binding\') != self.binding or prior.get(\'job\') != job[\'manifest_sha256\']\\n                    or prior.get(\'result\') != result[\'manifest_sha256\']):\\n                raise ValueError(\'Committed import receipt binding changed.\')\\n            return seal({**{k:v for k,v in prior.items() if k != \'manifest_sha256\'},\\n                         \'coverage\': self.coverage(), \'payloads_reopened\': False})\\n        allowed = self.assigned_output_tasks(job, result)\\n        tasks = {t[\'id\']: t for t in job[\'tasks\']}\\n        expected_keys, affected, completed_ids = {}, {f: set() for f in FAMILIES}, []\\n        for receipt in result[\'tasks\']:\\n            task = tasks[receipt[\'task\']]\\n            if receipt[\'task\'] not in allowed or receipt[\'task\'] in completed_ids:\\n                raise ValueError(\'Output contains duplicate/unassigned tasks.\')\\n            keys = receipt[\'keys\']\\n            if len(keys) != task[\'rows\'] or len(set(keys)) != len(keys) or digest_json(sorted(keys)) != task[\'keys_digest\']:\\n                raise ValueError(\'Output task key coverage changed.\')\\n            expected_keys.setdefault(task[\'family\'], set()).update(keys)\\n            completed_ids.append(receipt[\'task\'])\\n            affected[task[\'family\']].add(task[\'ordinal\'])\\n        # Validate all output files before publishing any coverage receipt.\\n        blocks_by_family = {f: [] for f in FAMILIES}\\n        available = {f: set() for f in FAMILIES}\\n        # A full-input execution plan may deliberately omit vectors already on\\n        # Drive. Never trust that claim from the GPU result: verify its keys in\\n        # the exact paid producer\'s CPU lookup before allowing task coverage.\\n        for family, values in (cached_keys or {}).items():\\n            keys = set(values) & expected_keys.get(family, set())\\n            if family not in FAMILIES:\\n                raise ValueError(\'Unexpected cached vector family.\')\\n            if not keys:\\n                continue\\n            expected = self.view.identities[family]\\n            cache = blocks_factory(self.view.data, SimpleNamespace(identity=expected[\'encoder\'], dimension=expected[\'dimension\']),\\n                                   self.work/family, receipt_api=receipt_api, progress=print)\\n            try:\\n                if set(lookup(cache, list(keys))) != keys:\\n                    raise ValueError(\'Execution plan cached vectors are not available in the verified Drive producer.\')\\n                available[family].update(keys)\\n            finally:\\n                cache.close()\\n        for block in result[\'blocks\']:\\n            checked(block)\\n            family = next((f for f in FAMILIES if block[\'identity\'] == {\\n                **self.view.identities[f], \'format\': \'content-float32-l2-v1\'}), None)\\n            if family is None or not set(block[\'keys\']) <= expected_keys.get(family, set()) or len(block[\'keys\']) != block[\'rows\']:\\n                raise ValueError(\'Unexpected vector block producer/keys.\')\\n            path = bound(output, block[\'path\'])\\n            verify_file(path, block[\'sha256\'])\\n            validate_vectors(np.load(path, mmap_mode=\'r\', allow_pickle=False), block[\'rows\'], block[\'identity\'][\'dimension\'])\\n            if path.parent.relative_to(output).as_posix() != \'model_cache/\'+digest_json(block[\'identity\'])[:24]:\\n                raise ValueError(\'Output vector path/producer namespace changed.\')\\n            blocks_by_family[family].append(block)\\n            available[family].update(block[\'keys\'])\\n        if any(not keys <= available[f] for f, keys in expected_keys.items()):\\n            raise ValueError(\'Output tasks are missing saved vector payloads.\')\\n        for family in FAMILIES:\\n            if not affected[family]:\\n                continue\\n            expected = self.view.identities[family]\\n            blocks = blocks_factory(self.view.data, SimpleNamespace(identity=expected[\'encoder\'], dimension=expected[\'dimension\']),\\n                                    self.work/family, receipt_api=receipt_api, progress=print)\\n            try:\\n                for block in blocks_by_family[family]:\\n                    path = bound(self.view.data, block[\'path\'])\\n                    if path.exists():\\n                        verify_file(path, block[\'sha256\'])\\n                    else:\\n                        copied(bound(output, block[\'path\']), path, block[\'sha256\'])\\n                    blocks.save_block(block[\'keys\'], path, block[\'sha256\'])\\n                self.complete_maps(family, affected[family], blocks)\\n                if getattr(blocks, \'_snapshot_dirty\', False):\\n                    blocks._snapshot()\\n            finally:\\n                blocks.close()\\n        report = seal({\'state\': \'NEW_DATA_KAGGLE_IMPORTED\', \'binding\': self.binding, \'job\': job[\'manifest_sha256\'],\\n            \'result\': result[\'manifest_sha256\'], \'account_id\': result[\'account_id\'], \'selection_ids\': completed_ids,\\n            \'coverage\': self.coverage(), \'updated_at\': utc_now()})\\n        atomic_json(self.root/\'imports\'/(result[\'manifest_sha256\']+\'.json\'), report)\\n        return report\\n\\n\\ndef validate_job(directory, checkout=None):\\n    directory = Path(directory)\\n    job = checked(read_json(directory/\'job.json\'))\\n    if (job.get(\'schema\') != \'new-data-kaggle-v1\' or job.get(\'repo_url\') != \'https://github.com/Platypus27-coder/VietMedBridge.git\'\\n            or not re.fullmatch(\'[0-9a-f]{40}\', job.get(\'code_commit\', \'\'))):\\n        raise ValueError(\'Invalid new-data Kaggle job.\')\\n    kaggle_assignment(accounts=job[\'team\'][\'accounts\'], account_id=0, gpu_count=job[\'team\'][\'gpu_count\'])\\n    if set(job[\'worker_sources\']) != set(SOURCE_FILES) or set(job[\'models\']) != set(FAMILIES):\\n        raise ValueError(\'Missing model/worker producer contract.\')\\n    for name, checksum in job[\'worker_sources\'].items():\\n        verify_file(directory/name, checksum)\\n    seen = set()\\n    for task in job[\'tasks\']:\\n        if (task[\'id\'] in seen or task[\'id\'] != task[\'selection_id\'] or task[\'family\'] not in FAMILIES\\n                or type(task[\'worker\']) is not int or not 0 <= task[\'worker\'] < job[\'team\'][\'accounts\']*job[\'team\'][\'gpu_count\']\\n                or type(task[\'rows\']) is not int or not 0 < task[\'rows\'] <= TASK_ROWS\\n                or relative_path(task[\'asset\']) != \'task-\'+task[\'id\']+\'.parquet\'):\\n            raise ValueError(\'Invalid sparse task assignment.\')\\n        seen.add(task[\'id\'])\\n    if checkout is not None:\\n        for name, checksum in job[\'model_code\'].items():\\n            if name not in (\'retrieval_models.py\', \'qwen_models.py\'):\\n                raise ValueError(\'Unexpected model code file.\')\\n            verify_file(Path(checkout)/\'src/vietmedbridge\'/name, checksum)\\n        for family in FAMILIES:\\n            validate_encoder(job, family, job[\'models\'][family][\'encoder\'], job[\'models\'][family][\'dimension\'])\\n    return job\\n\\n\\ndef download_outputs(api, handle, work, *, project=None, import_roots=(), allow_failed=False,\\n                     manifest_path=None, expected_result=None, on_manifest=None):\\n    \\"\\"\\"Page through the new-data manifest first, before downloading vectors.\\"\\"\\"\\n    if on_manifest is not None and (not callable(on_manifest) or manifest_path is None):\\n        raise ValueError(\'Manifest observer requires a callable and pinned manifest path.\')\\n    from vietmedbridge.kaggle_outputs import _download_pages, output_handles\\n    output_handles(handle)\\n    transport = None\\n    if manifest_path is not None:\\n        try:\\n            from .kaggle_output_transfer import SavedNotebookOutputs, retry_kaggle\\n        except ImportError:\\n            from kaggle_output_transfer import SavedNotebookOutputs, retry_kaggle\\n        status = retry_kaggle(lambda: api.kernels_status(handle))\\n    else:\\n        status = api.kernels_status(handle)\\n    status = status.get(\'status\') if isinstance(status, dict) else status.status\\n    status = getattr(status, \'name\', str(status)).upper()\\n    failed = status in {\'ERROR\', \'FAILED\', \'CANCELLED\', \'CANCELED\'}\\n    if status != \'COMPLETE\' and not (allow_failed and failed):\\n        raise RuntimeError(f\'{handle}: status={status}; wait for the saved version to finish.\')\\n    if failed:\\n        print(f\'KAGGLE_SESSION_{status}: inspecting saved checkpoint Outputs only; \'\\n              \'GPU work is not restarted and completion is not assumed.\', flush=True)\\n    base = Path(work)/\'downloads\'/digest_json(handle)[:16]\\n    probe = base/(\'probe-\'+str(time.time_ns()))\\n    probe.mkdir(parents=True, exist_ok=True)\\n    if manifest_path is not None:\\n        transport = SavedNotebookOutputs(api, handle, work, project=project)\\n        paths = [transport.manifest(manifest_path)]\\n    else:\\n        paths = _download_pages(api, handle, probe, file_pattern=r\'(^|/)new-data-result[.]json$\', force=True)\\n    manifests = [p for p in paths if p.name == \'new-data-result.json\']\\n    if len(manifests) != 1:\\n        raise RuntimeError(f\'{handle}: expected one new-data-result.json, found {len(manifests)}; choose the saved new-data worker version.\')\\n    manifest = checked(read_json(manifests[0]))\\n    if expected_result is not None:\\n        expected = dict(expected_result)\\n        actual = {**manifest, \'completed_tasks\': len(manifest.get(\'tasks\', []))}\\n        if (not expected or set(expected)-{\'manifest_sha256\', \'job\', \'execution_plan\', \'account_id\', \'completed_tasks\'}\\n                or any(actual.get(name) != value for name, value in expected.items())):\\n            raise ValueError(\'Saved Outputs differ from the expected completed GPU run; no vector download started.\')\\n    if on_manifest is not None:\\n        # Optional diagnostics use the same pinned saved version, after the\\n        # expected-run guard and before any vector transfer or import shortcut.\\n        on_manifest(transport, copy.deepcopy(manifest))\\n    relative = Path(manifest_path) if transport is not None else manifests[0].relative_to(probe)\\n    if project is not None:\\n        from_parent = Path(project)/\'data_staging\'/\'btc-new-71be0e12-20261019-v1\'/\'freeze_batch_v1\'/\'embeddings_v1\'\\n        roots=[from_parent,*[bound(project,p) for p in import_roots]]\\n        receipts=[p for root in roots for p in root.glob(\'kaggle-*/imports/\'+manifest[\'manifest_sha256\']+\'.json\')]\\n        for receipt in receipts:\\n            prior = checked(read_json(receipt))\\n            if prior.get(\'result\') != manifest[\'manifest_sha256\'] or prior.get(\'job\') != manifest[\'job\']:\\n                raise ValueError(\'Previously imported output receipt changed.\')\\n            job = checked(read_json(receipt.parent.parent/\'exports\'/(\'job-\'+manifest[\'job\']+\'.json\')))\\n            atomic_json(manifests[0].parent/\'job.json\', job)\\n            print(\'OUTPUT_ALREADY_IMPORTED: sealed receipt reused; no vector download\', flush=True)\\n            return manifests[0].parent\\n    destination = base/manifest[\'manifest_sha256\'][:16]\\n    destination.mkdir(parents=True, exist_ok=True)\\n    if transport is not None:\\n        target = transport.all(manifest_path, manifest)\\n        paths = [target]\\n    else:\\n        paths = _download_pages(api, handle, destination, force=False, quiet=False)\\n        target = destination/relative\\n    if target not in paths or checked(read_json(target)) != manifest:\\n        raise ValueError(\'Saved Kaggle Outputs changed after preflight; select a fixed version.\')\\n    return target.parent\\n\\n\\ndef upload_job(flow, target, job, owner, *, activation_filename=\'active_round.json\', dataset_prefix=None):\\n    \\"\\"\\"A unique private dataset, with an upload pointer retained on Drive.\\"\\"\\"\\n    import kagglehub\\n    if not re.fullmatch(\'[A-Za-z0-9_-]+\', owner):\\n        raise ValueError(\'Invalid Kaggle owner.\')\\n    pointer = flow.root/\'uploads\'/(owner+\'-\'+job[\'manifest_sha256\']+\'.json\')\\n    if pointer.exists():\\n        saved = read_json(pointer)\\n        if saved[\'job\'] != job[\'manifest_sha256\']:\\n            raise ValueError(\'Upload pointer belongs to another job.\')\\n        handle = saved[\'dataset_handle\']\\n    else:\\n        prefix = dataset_prefix or (\'/vmb-full-\' if job.get(\'coverage_scope\', {}).get(\'official_ids\') else \'/vmb-new-\')\\n        if not re.fullmatch(r\'/[a-z0-9-]+-\', prefix):\\n            raise ValueError(\'Invalid dataset prefix.\')\\n        handle = owner+prefix+job[\'manifest_sha256\'][:12]+\'-\'+str(time.time_ns())\\n        atomic_json(pointer, {\'job\': job[\'manifest_sha256\'], \'dataset_handle\': handle, \'state\': \'UPLOAD_PLANNED\'})\\n        saved = read_json(pointer)\\n    if saved[\'state\'] == \'UPLOAD_PLANNED\':\\n        # Deterministic pointer survives a failed upload; never choose a second handle.\\n        kagglehub.dataset_upload(handle, str(target))\\n        atomic_json(pointer, {\'job\': job[\'manifest_sha256\'], \'dataset_handle\': handle, \'state\': \'VERIFY_PENDING\'})\\n    remote = None\\n    for attempt in range(6):\\n        try:\\n            remote = Path(kagglehub.dataset_download(handle, path=\'job.json\', force_download=True))\\n            break\\n        except Exception:\\n            if attempt == 5:\\n                raise RuntimeError(\'Dataset is processing or inaccessible; rerun to verify the same upload.\') from None\\n            time.sleep(min(30, 5*(attempt+1)))\\n    if read_json(remote) != job:\\n        raise ValueError(\'Uploaded Kaggle job header differs; do not start GPU workers.\')\\n    atomic_json(pointer, {\'job\': job[\'manifest_sha256\'], \'dataset_handle\': handle, \'state\': \'VERIFIED\'})\\n    if Path(activation_filename).name != activation_filename or not activation_filename.endswith(\'.json\'):\\n        raise ValueError(\'Invalid dataset activation filename.\')\\n    atomic_json(flow.root/activation_filename, seal({\'binding\': flow.binding, \'job\': job[\'manifest_sha256\'], \'dataset_handle\': handle}))\\n    print(\'NEW_DATASET_READY:\', \'https://www.kaggle.com/datasets/\'+handle, flush=True)\\n    print(\'Team:\', job[\'team\'], \'| scheduled missing-text tasks:\', len(job[\'tasks\']), flush=True)\\n    return handle\\n\\n\\ndef run_worker(directory, output, work, *, worker_id, factories, deadline, batch_sizes=None):\\n    \\"\\"\\"Each GPU owns every occurrence of a given text hash, avoiding duplicate encoding.\\"\\"\\"\\n    directory, output, work = Path(directory), Path(output), Path(work)\\n    job = validate_job(directory)\\n    if type(worker_id) is not int or not 0 <= worker_id < job[\'team\'][\'accounts\']*job[\'team\'][\'gpu_count\']:\\n        raise ValueError(\'Worker ID is outside this job.\')\\n    output.mkdir(parents=True, exist_ok=True)\\n    if (output/\'job.json\').exists() and read_json(output/\'job.json\') != job:\\n        raise ValueError(\'Working output contains another job. Save its Outputs; start a fresh Kaggle version.\')\\n    atomic_json(output/\'job.json\', job)\\n    journal = output/f\'worker-{worker_id}.json\'\\n    completed = []\\n    if journal.exists():\\n        saved = checked(read_json(journal))\\n        if saved[\'job\'] != job[\'manifest_sha256\'] or saved[\'worker_id\'] != worker_id:\\n            raise ValueError(\'Worker journal belongs to another job/worker.\')\\n        completed = saved[\'tasks\']\\n    done = {r[\'task\'] for r in completed}\\n    if len(done) != len(completed) or not done <= {t[\'id\'] for t in job[\'tasks\'] if t[\'worker\'] == worker_id}:\\n        raise ValueError(\'Duplicate local task receipt.\')\\n    stop_reason = \'ASSIGNMENT_COMPLETE\'\\n    for family in FAMILIES:\\n        assigned = [t for t in job[\'tasks\'] if t[\'worker\'] == worker_id and t[\'family\'] == family and t[\'id\'] not in done]\\n        print(f\'GPU worker {worker_id}: {family}, {len(assigned)} missing-text tasks\', flush=True)\\n        if not assigned or time.monotonic() >= deadline:\\n            if assigned:\\n                stop_reason = \'SESSION_TIME_LIMIT\'\\n            continue\\n        lazy = LazyEncoder(job[\'models\'][family], factories[family])\\n        blocks = ContentBlocks(output, lazy, work/family)\\n        try:\\n            for task in assigned:\\n                if time.monotonic() >= deadline:\\n                    stop_reason = \'SESSION_TIME_LIMIT\'\\n                    break\\n                path = directory/task[\'asset\']\\n                verify_file(path, task[\'sha256\'])\\n                rows = pq.read_table(path, columns=[\'id\', \'text\']).to_pylist()\\n                keys = [r[\'id\'] for r in rows]\\n                workers = job[\'team\'][\'accounts\']*job[\'team\'][\'gpu_count\']\\n                if (len(rows) != task[\'rows\'] or digest_json(sorted(keys)) != task[\'keys_digest\']\\n                        or any(k != key(r[\'text\']) or int(k[:8], 16) % workers != worker_id for k, r in zip(keys, rows))):\\n                    raise ValueError(\'Task payload/hash/ownership changed.\')\\n                blocks.refs([r[\'text\'] for r in rows], batch_size=(batch_sizes or {\'bge\': 32, \'qwen\': 16})[family],\\n                            encode_missing=True, encode_block_rows=TASK_ROWS)\\n                completed.append({\'task\': task[\'id\'], \'keys\': keys})\\n                atomic_json(journal, seal({\'job\': job[\'manifest_sha256\'], \'worker_id\': worker_id, \'tasks\': completed}))\\n                print(f\'KAGGLE_CHECKPOINT worker {worker_id} | saved tasks {len(completed)} | {family} | /kaggle/working\', flush=True)\\n        finally:\\n            blocks.close()\\n            lazy.close()\\n    return seal({\'job\': job[\'manifest_sha256\'], \'worker_id\': worker_id, \'tasks\': completed, \'stop_reason\': stop_reason})\\n\\n\\ndef finalize(directory, output, account_id):\\n    job = validate_job(directory)\\n    output = Path(output)\\n    assignment = kaggle_assignment(accounts=job[\'team\'][\'accounts\'], account_id=account_id, gpu_count=job[\'team\'][\'gpu_count\'])\\n    root = output/\'result\'\\n    root.mkdir(parents=True, exist_ok=True)\\n    tasks = []\\n    for worker in assignment[\'worker_ids\']:\\n        path = root/f\'worker-{worker}.json\'\\n        if path.exists():\\n            saved = checked(read_json(path))\\n            if saved[\'job\'] != job[\'manifest_sha256\'] or saved[\'worker_id\'] != worker:\\n                raise ValueError(\'Worker journal belongs to another job/assignment.\')\\n            tasks.extend(saved[\'tasks\'])\\n    task_family = {t[\'id\']: t[\'family\'] for t in job[\'tasks\']}\\n    committed = {f: {k for task in tasks if task_family[task[\'task\']] == f for k in task[\'keys\']} for f in FAMILIES}\\n    blocks = {}\\n    for path in root.glob(\'model_cache/*/*.done.json\'):\\n        block = checked(read_json(path))\\n        family = next((f for f in FAMILIES if block[\'identity\'] == {**job[\'models\'][f], \'format\': \'content-float32-l2-v1\'}), None)\\n        if family is not None and set(block[\'keys\']) <= committed[family]:\\n            blocks[block[\'path\']] = block\\n    atomic_json(root/\'job.json\', job)\\n    result = seal({\'schema\': \'new-data-kaggle-result-v1\', \'job\': job[\'manifest_sha256\'],\\n        \'account_id\': account_id, \'tasks\': tasks, \'blocks\': list(blocks.values()), \'updated_at\': utc_now(),\\n        \'state\': \'ROUND_COMPLETE\' if len(tasks) == sum(t[\'worker\'] in assignment[\'worker_ids\'] for t in job[\'tasks\']) else \'CHECKPOINTED_PARTIAL\'})\\n    atomic_json(root/\'new-data-result.json\', result)\\n    return result\\n\\n\\ndef launch(directory, checkout, output, work, *, account_id=0, hours=10.0):\\n    \\"\\"\\"Fresh CUDA subprocesses, one model per T4; leave time to save Kaggle Outputs.\\"\\"\\"\\n    import torch\\n    from vietmedbridge.portable_embeddings import embedding_runtime\\n    job = validate_job(directory, checkout)\\n    if embedding_runtime() != job[\'runtime\'] or not torch.cuda.is_available() or torch.cuda.device_count() != job[\'team\'][\'gpu_count\']:\\n        raise RuntimeError(\'CUDA/runtime mismatch: select T4 x2 and use the locked CUDA packages.\')\\n    assignment = kaggle_assignment(accounts=job[\'team\'][\'accounts\'], account_id=account_id, gpu_count=job[\'team\'][\'gpu_count\'])\\n    if not 0 < hours <= 10.5:\\n        raise ValueError(\'Use a save deadline up to 10.5 hours, before the provider hard stop.\')\\n    deadline = time.monotonic()+hours*3600-180\\n    processes = []\\n    try:\\n        for local_gpu, worker in enumerate(assignment[\'worker_ids\']):\\n            env = os.environ.copy()\\n            env[\'CUDA_VISIBLE_DEVICES\'] = str(local_gpu)\\n            env[\'PYTHONPATH\'] = str(Path(checkout)/\'src\')+os.pathsep+str(Path(directory))\\n            command = [sys.executable, \'-u\', str(Path(directory)/\'new_data_kaggle.py\'), \'worker\', \'--job\', str(directory),\\n                       \'--output\', str(Path(output)/\'result\'), \'--work\', str(Path(work)/f\'worker-{worker}\'),\\n                       \'--worker-id\', str(worker), \'--deadline\', str(deadline)]\\n            processes.append(subprocess.Popen(command, env=env))\\n        while any(p.poll() is None for p in processes):\\n            if any(p.poll() not in (None, 0) for p in processes):\\n                raise RuntimeError(\'A GPU worker failed; outputs of completed tasks remain available.\')\\n            time.sleep(2)\\n        result = finalize(directory, output, account_id)\\n        if any(p.returncode != 0 for p in processes):\\n            raise RuntimeError(\'A GPU worker failed; inspect the preceding traceback.\')\\n        return result\\n    finally:\\n        for process in processes:\\n            if process.poll() is None:\\n                process.terminate()\\n        for process in processes:\\n            if process.poll() is None:\\n                try:\\n                    process.wait(timeout=30)\\n                except subprocess.TimeoutExpired:\\n                    process.kill()\\n                    process.wait(timeout=10)\\n        # Even an exception publishes a partial result for completed tasks.\\n        finalize(directory, output, account_id)\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser()\\n    parser.add_argument(\'mode\', choices=[\'worker\', \'launch\'])\\n    parser.add_argument(\'--job\', required=True)\\n    parser.add_argument(\'--output\', required=True)\\n    parser.add_argument(\'--work\', required=True)\\n    parser.add_argument(\'--checkout\')\\n    parser.add_argument(\'--worker-id\', type=int)\\n    parser.add_argument(\'--deadline\', type=float)\\n    parser.add_argument(\'--account-id\', type=int, default=0)\\n    parser.add_argument(\'--hours\', type=float, default=10.0)\\n    args = parser.parse_args()\\n    if args.mode == \'launch\':\\n        result = launch(args.job, args.checkout, args.output, args.work, account_id=args.account_id, hours=args.hours)\\n    else:\\n        import torch\\n        from vietmedbridge.portable_embeddings import embedding_runtime\\n        from vietmedbridge.retrieval_models import TorchDenseEncoder\\n        from vietmedbridge.qwen_models import TorchQwenEncoder\\n        job = validate_job(args.job)\\n        if not torch.cuda.is_available() or embedding_runtime() != job[\'runtime\']:\\n            raise RuntimeError(\'GPU worker runtime differs from the paid vector producer.\')\\n        result = run_worker(args.job, args.output, args.work, worker_id=args.worker_id, deadline=args.deadline,\\n            factories={\'bge\': lambda: TorchDenseEncoder(job[\'config\'][\'dense\']), \'qwen\': lambda: TorchQwenEncoder(job[\'config\'][\'second_dense\'])})\\n    print(json.dumps(result, ensure_ascii=False, indent=2), flush=True)\\n\\n\\nif __name__ == \'__main__\':\\n    main()\\n", "new_data_embeddings.py": "\\"\\"\\"Frozen-input and embedding-map contracts for the isolated ten-source batch.\\n\\nNative model code and content-cache keys stay unchanged. Only new input parts\\nare opened. A worker journal replaces repeated scans of old map payloads.\\n\\"\\"\\"\\nfrom pathlib import Path\\n\\n\\nfrom vietmedbridge.artifacts import (atomic_json, digest_json, publish_file,\\n    read_json, sha256_file, utc_now, verify_file)\\nfrom vietmedbridge.content_embeddings import checked, key, seal\\nfrom vietmedbridge.portable_embeddings import resources_relative, validate_encoder, validate_runtime_lock\\nfrom vietmedbridge.scale_vectors import bound\\n\\ntry:\\n    from .new_data_cache_lookup import ResumableContentBlocks\\nexcept ImportError:\\n    from new_data_cache_lookup import ResumableContentBlocks\\n\\nPARENT = \\"btc-new-71be0e12-20261019-v1\\"\\nFAMILIES = (\\"bge\\", \\"qwen\\")\\n\\n\\ndef load_new_batch(project):\\n    project = Path(project).resolve()\\n    root = project / \\"data_staging\\" / PARENT / \\"freeze_batch_v1\\"\\n    batch, contract = checked(read_json(root / \\"freeze_batch.json\\")), checked(read_json(root / \\"contract.json\\"))\\n    if (batch.get(\\"state\\") != \\"NEW_DATA_INPUTS_READY\\" or batch.get(\\"completed_builds\\") != 10\\n            or batch.get(\\"total_builds\\") != 10 or batch[\\"contract_sha256\\"] != contract[\\"manifest_sha256\\"]\\n            or len(batch[\\"builds\\"]) != 10 or len(contract[\\"sources\\"]) != 10):\\n        raise ValueError(\\"The complete ten-source frozen input handoff is required.\\")\\n    expected = {f\\"new-{i:05d}\\" for i in range(9)} | {\\"retained-71be0e12\\"}\\n    if {b[\\"build_run\\"] for b in batch[\\"builds\\"]} != expected:\\n        raise ValueError(\\"New-data batch source coverage changed.\\")\\n    sources, ordinal = [], 0\\n    for item, source in zip(batch[\\"builds\\"], contract[\\"sources\\"]):\\n        checked(item)\\n        if (item[\\"state\\"] != \\"COMPLETE\\" or item[\\"contract_sha256\\"] != contract[\\"manifest_sha256\\"]\\n                or any(item[k] != source[k] for k in (\\"build_run\\", \\"build_dir\\", \\"snapshot_sha256\\", \\"counts\\"))):\\n            raise ValueError(\\"Frozen source receipt differs from the batch contract.\\")\\n        build = bound(project, item[\\"build_dir\\"])\\n        inputs = checked(read_json(build / \\"index_inputs/units.json\\"))\\n        if (inputs.get(\\"state\\") != \\"COMPLETE\\" or inputs[\\"manifest_sha256\\"] != item[\\"index_inputs_manifest_sha256\\"]\\n                or inputs[\\"candidate_manifest_sha256\\"] != item[\\"candidate_manifest_sha256\\"]\\n                or inputs[\\"input_count\\"] != item[\\"unique_dense_inputs\\"] or len(inputs[\\"parts\\"]) != item[\\"input_parts\\"]):\\n            raise ValueError(\\"Frozen input manifest binding changed.\\")\\n        cursor = 0\\n        for part in inputs[\\"parts\\"]:\\n            if (part[\\"start\\"] != cursor or part[\\"rows\\"] != min(4096, inputs[\\"input_count\\"]-cursor)\\n                    or part[\\"signature\\"] != inputs[\\"signature\\"] or not part[\\"path\\"].endswith(\\".parquet\\")):\\n                raise ValueError(\\"Frozen input part range changed.\\")\\n            bound(build / \\"index_inputs\\", part[\\"path\\"])\\n            sources.append({\\"ordinal\\": ordinal, \\"build\\": item, \\"inputs\\": inputs, \\"part\\": part})\\n            cursor += part[\\"rows\\"]\\n            ordinal += 1\\n        if cursor != inputs[\\"input_count\\"]:\\n            raise ValueError(\\"Frozen input coverage changed.\\")\\n    if len(sources) != batch[\\"input_parts\\"]:\\n        raise ValueError(\\"Batch part coverage changed.\\")\\n    return root, batch, sources\\n\\n\\ndef embedding_contract(data_root, checkout):\\n    \\"\\"\\"Metadata-only: bind exactly to the existing paid BGE/Qwen producers.\\"\\"\\"\\n    data_root, checkout = Path(data_root), Path(checkout)\\n    runtime_lock = checked(read_json(data_root / \\"retrieval_embedding_runtime_lock.json\\"))\\n    runtime = runtime_lock[\\"runtime\\"]\\n    validate_runtime_lock(runtime)\\n    config = read_json(checkout / \\"configs/retrieval_full.json\\")\\n    active = read_json(data_root / \\"active_data_candidate.json\\")\\n    base = data_root / resources_relative(active, config)\\n    for name, checksum in runtime_lock[\\"model_code\\"].items():\\n        if name not in (\\"retrieval_models.py\\", \\"qwen_models.py\\") or sha256_file(checkout / \\"src/vietmedbridge\\" / name) != checksum:\\n            raise ValueError(\\"Pinned model producer differs from the existing vector cache.\\")\\n    job = {\\"config\\": config, \\"runtime\\": runtime, \\"model_code\\": runtime_lock[\\"model_code\\"]}\\n    identities = {}\\n    for family in FAMILIES:\\n        saved = read_json(base / family / \\"config.json\\")\\n        validate_encoder(job, family, saved[\\"encoder\\"], saved[\\"dimension\\"])\\n        if saved.get(\\"format\\") != \\"content-reference-v1\\":\\n            raise ValueError(\\"Existing embedding producer format changed.\\")\\n        encoder = saved[\\"encoder\\"]\\n        count = encoder.get(\\"parameters_before_quantization\\", encoder.get(\\"parameters\\"))\\n        if type(count) is not int or not 0 < count <= 15_000_000_000:\\n            raise ValueError(\\"Embedding model parameter budget is not verified.\\")\\n        identities[family] = {\\"encoder\\": encoder, \\"dimension\\": saved[\\"dimension\\"]}\\n    return config, runtime, identities, active[\\"candidate_manifest_sha256\\"]\\n\\n\\nclass LazyEncoder:\\n    \\"\\"\\"No weights until a cache lookup finds at least one missing text.\\"\\"\\"\\n    def __init__(self, expected, factory):\\n        self.identity, self.dimension = expected[\\"encoder\\"], expected[\\"dimension\\"]\\n        self.factory, self.owner, self.encoder = factory, None, None\\n\\n    def encode(self, texts, *, batch_size):\\n        if self.encoder is None:\\n            self.owner = self.factory()\\n            self.encoder = self.owner.for_role(\\"corpus\\") if hasattr(self.owner, \\"for_role\\") else self.owner\\n            if self.encoder.identity != self.identity or self.encoder.dimension != self.dimension:\\n                self.close()\\n                raise ValueError(\\"GPU model identity differs from the paid content cache; no new cache namespace created.\\")\\n            from vietmedbridge.qwen_execution import configure_embedding_execution\\n            configure_embedding_execution(self.encoder)\\n        return self.encoder.encode(texts, batch_size=batch_size)\\n\\n    def close(self):\\n        if self.owner is not None:\\n            self.owner.close()\\n            self.owner = self.encoder = None\\n\\n\\nclass NewDataEmbeddings:\\n    def __init__(self, project, work_dir, identities, *, team_size=1, worker_id=0, batch_loader=None):\\n        if type(team_size) is not int or team_size < 1 or type(worker_id) is not int or not 0 <= worker_id < team_size:\\n            raise ValueError(\\"Invalid GPU team/worker assignment.\\")\\n        self.project, self.work = Path(project).resolve(), Path(work_dir).resolve()\\n        self.root, self.batch, self.sources = (batch_loader or load_new_batch)(self.project)\\n        self.data = self.project / \\"data\\"\\n        self.output = self.root / \\"embeddings_v1\\"\\n        self.identities, self.team_size, self.worker_id = identities, team_size, worker_id\\n        self.work.mkdir(parents=True, exist_ok=True)\\n        self.phase = \\"manifest preflight\\"\\n        binding = {\\"schema_version\\": 1, \\"batch\\": self.batch[\\"manifest_sha256\\"], \\"models\\": identities,\\n                   \\"format\\": \\"NEW_DATA_CONTENT_EMBEDDINGS_V1\\"}\\n        path = self.output / \\"contract.json\\"\\n        if path.is_file() and checked(read_json(path)) != seal(binding):\\n            raise ValueError(\\"New-data embedding contract changed; retain its checkpoints.\\")\\n        atomic_json(path, seal(binding))\\n        self.binding = digest_json(binding)\\n\\n    def _key(self, source):\\n        return source[\\"build\\"][\\"build_run\\"] + \\":\\" + str(source[\\"part\\"][\\"start\\"])\\n\\n    def _view(self, family, source):\\n        inputs = source[\\"inputs\\"]\\n        expected = self.identities[family]\\n        identity = {\\"input_manifest_sha256\\": inputs[\\"manifest_sha256\\"], \\"input_count\\": inputs[\\"input_count\\"],\\n                    \\"encoder\\": expected[\\"encoder\\"], \\"dimension\\": expected[\\"dimension\\"], \\"format\\": \\"content-reference-v1\\"}\\n        inherited = source[\'build\'].get(\'completed_views\', {}).get(family)\\n        root = bound(self.project, inherited[\'path\']) if inherited else self.output / family / source[\'build\'][\'build_run\']\\n        return root, identity\\n\\n    def _valid_part(self, family, source, saved):\\n        checked(saved)\\n        _, identity = self._view(family, source)\\n        part = source[\\"part\\"]\\n        expected = {\\"signature\\": digest_json(identity), \\"start\\": part[\\"start\\"], \\"rows\\": part[\\"rows\\"],\\n                    \\"input_sha256\\": part[\\"sha256\\"], \\"path\\": f\\"map-{part[\'start\']:012d}.parquet\\"}\\n        if any(saved.get(k) != v for k, v in expected.items()):\\n            raise ValueError(\\"Embedding part receipt differs from frozen input/model policy.\\")\\n        return saved\\n\\n    def _journals(self, family):\\n        expected = {self._key(s): s for s in self.sources}\\n        merged = {}\\n        groups = {}\\n        for source in self.sources:\\n            if family in source[\'build\'].get(\'completed_views\', {}):\\n                groups.setdefault(source[\'build\'][\'build_run\'], []).append(source)\\n        for sources in groups.values():\\n            root, identity = self._view(family, sources[0])\\n            reference = sources[0][\'build\'][\'completed_views\'][family]\\n            saved = checked(read_json(root / \'embeddings.json\'))\\n            if (saved.get(\'state\') != \'COMPLETE\' or saved[\'manifest_sha256\'] != reference[\'manifest_sha256\']\\n                    or any(saved.get(k) != v for k, v in identity.items()) or len(saved[\'parts\']) != len(sources)):\\n                raise ValueError(\'Inherited complete embedding view changed.\')\\n            for source, record in zip(sources, saved[\'parts\']):\\n                merged[self._key(source)] = self._valid_part(family, source, record)\\n        directory = self.output / family / \\"progress\\"\\n        for path in sorted(directory.glob(\\"worker-*.json\\")):\\n            saved = checked(read_json(path))\\n            if saved.get(\\"binding\\") != self.binding or saved.get(\\"family\\") != family:\\n                raise ValueError(\\"GPU worker journal belongs to another batch/model.\\")\\n            for name, record in saved[\\"completed\\"].items():\\n                if name not in expected:\\n                    raise ValueError(\\"GPU journal contains an unknown frozen input part.\\")\\n                self._valid_part(family, expected[name], record)\\n                if name in merged and merged[name] != record:\\n                    raise ValueError(\\"GPU workers published conflicting immutable map receipts.\\")\\n                merged[name] = record\\n        return merged\\n\\n    def _save(self, family, completed):\\n        payload = {\\"binding\\": self.binding, \\"family\\": family, \\"team_size\\": self.team_size,\\n                   \\"worker_id\\": self.worker_id, \\"completed\\": completed, \\"updated_at\\": utc_now()}\\n        atomic_json(self.output / family / \\"progress\\" / f\\"worker-{self.worker_id}-of-{self.team_size}.json\\", seal(payload))\\n\\n    def _seal_views(self, family, completed):\\n        grouped = {}\\n        for source in self.sources:\\n            grouped.setdefault(source[\\"build\\"][\\"build_run\\"], []).append(source)\\n        for sources in grouped.values():\\n            root, identity = self._view(family, sources[0])\\n            if family in sources[0][\'build\'].get(\'completed_views\', {}):\\n                continue  # Existing complete views are read-only; never rewrite the submit namespace.\\n            if all(self._key(s) in completed for s in sources):\\n                path = root / \\"embeddings.json\\"\\n                value = seal(identity | {\\"signature\\": digest_json(identity), \\"state\\": \\"COMPLETE\\",\\n                    \\"parts\\": [completed[self._key(s)] for s in sources]})\\n                if path.is_file() and checked(read_json(path)) != value:\\n                    raise ValueError(\\"Completed embedding view changed.\\")\\n                if not path.is_file():\\n                    atomic_json(path, value)\\n", "new_data_cache_lookup.py": "\\"\\"\\"Released resumable receipt lookup; model producers are unchanged.\\"\\"\\"\\nfrom concurrent.futures import ThreadPoolExecutor\\nimport gzip\\nfrom pathlib import Path, PurePosixPath\\nimport shutil\\nimport sqlite3\\nimport time\\nfrom vietmedbridge.artifacts import (atomic_json, digest_json, local_workspace, publish_file,\\n    read_json, sha256_file, utc_now, verify_file)\\nfrom vietmedbridge.content_embeddings import ContentBlocks as NativeContentBlocks, checked, seal\\nfrom vietmedbridge.scale_vectors import bound\\ntry:\\n    from .new_data_drive_api import LookupPublishUnavailable\\nexcept ImportError:\\n    from new_data_drive_api import LookupPublishUnavailable\\n\\n\\nclass ResumableContentBlocks(NativeContentBlocks):\\n    \\"\\"\\"Retain validated receipt lookups on local disk across Drive recovery.\\n\\n    The native content key, encoder identity and vector payload validation stay\\n    unchanged. A compressed lookup is an optimization, never a coverage receipt.\\n    \\"\\"\\"\\n    def __init__(self, data_root, encoder, work_dir, *, progress=print, receipt_api=None, inventory=True):\\n        self.receipt_api = receipt_api\\n        if receipt_api is not None and Path(data_root).absolute() not in (receipt_api.requested_root, receipt_api.data_root):\\n            raise ValueError(\'Receipt API root differs from the embedding data root.\')\\n        self.root = receipt_api.data_root if receipt_api is not None else Path(data_root).resolve()\\n        self.encoder = encoder\\n        self.identity = {\'encoder\': encoder.identity, \'dimension\': encoder.dimension,\\n            \'format\': \'content-float32-l2-v1\'}\\n        self.directory = self.root / \'model_cache\' / digest_json(self.identity)[:24]\\n        if receipt_api is None:\\n            self.directory.mkdir(parents=True, exist_ok=True)\\n        self.progress = progress\\n        self.binding = {\'schema_version\': 1, \'data_root\': str(self.root), \'identity\': self.identity}\\n        self.lookup = Path(work_dir) / \'receipt_lookups\' / digest_json(self.binding)[:16]\\n        self.lookup.mkdir(parents=True, exist_ok=True)\\n        self.database = self.lookup / \'lookup.sqlite\'\\n        self.snapshot_binding = (dict(schema_version=2, data_folder_id=receipt_api.folder_id, identity=self.identity)\\n            if receipt_api is not None and hasattr(receipt_api, \'folder_id\') else self.binding)\\n        self.remote = self.root / \'retrieval/content_lookups\' / digest_json(self.snapshot_binding)[:16]\\n        self.legacy_remote = self.root / \'retrieval/content_lookups\' / digest_json(self.binding)[:16]\\n        self.temporary = local_workspace(work_dir)\\n        self.local = Path(self.temporary.__enter__())\\n        self.con = None\\n        self._snapshot_dirty = False\\n        self._snapshot_busy = False\\n        self._next_snapshot_count = 256\\n        self._receipt_count = 0\\n        try:\\n            if not self.database.exists():\\n                if not inventory:\\n                    raise FileNotFoundError(\'No local lookup to save; no receipt scan or model inference was started.\')\\n                self._restore()\\n            self.con = sqlite3.connect(self.database)\\n            self.con.execute(\'PRAGMA cache_size=-32768\')\\n            self.con.execute(\'PRAGMA synchronous=FULL\')\\n            self.con.execute(\'CREATE TABLE IF NOT EXISTS metadata(value TEXT NOT NULL)\')\\n            row = self.con.execute(\'SELECT value FROM metadata\').fetchone()\\n            signature = digest_json(self.binding)\\n            if row is not None and row[0] != signature:\\n                raise ValueError(\'Receipt lookup model/root binding changed.\')\\n            if row is None:\\n                self.con.execute(\'INSERT INTO metadata VALUES (?)\', [signature])\\n            self.con.execute(\'CREATE TABLE IF NOT EXISTS vectors(key TEXT PRIMARY KEY,path TEXT,sha TEXT,row INTEGER)\')\\n            self.con.execute(\'CREATE TABLE IF NOT EXISTS receipts(name TEXT PRIMARY KEY,digest TEXT NOT NULL)\')\\n            self.con.commit()\\n            known = {row[0] for row in self.con.execute(\'SELECT name FROM receipts\')}\\n            self._receipt_count = len(known)\\n            self._next_snapshot_count = max(256, len(known)*2)\\n            if not inventory:\\n                self._snapshot_dirty = True\\n                self._snapshot()\\n                return\\n            self.progress(f\'BGE receipt inventory: retained {len(known):,} validated receipts; listing new names\')\\n            markers = (receipt_api.receipts(self.identity) if receipt_api is not None\\n                else sorted(self.directory.glob(\'*.done.json\')))\\n            pending = [path for path in markers if path.name not in known]\\n            self.progress(f\'BGE receipt inventory: {len(markers)-len(pending):,}/{len(markers):,} reused\'\\n                f\' | {len(pending):,} unread | {\\"API\\" if receipt_api is not None else \\"mount\\"} transport | commit each receipt\')\\n            self._read_pending(pending, len(markers)-len(pending), len(markers))\\n            self.progress(f\'BGE receipt inventory ready: {len(markers):,} receipts | native vector SHA checked on use\')\\n            if receipt_api is not None:\\n                if pending:\\n                    self._snapshot_dirty = True\\n                if hasattr(receipt_api, \'publish_lookup\') and not self._snapshot_dirty:\\n                    saved = receipt_api.relative_json((self.remote / \'current.json\').relative_to(self.root).as_posix())\\n                    if saved is None:\\n                        self._snapshot_dirty = True\\n                    else:\\n                        checked(saved)\\n                        if saved.get(\'binding\') != self.snapshot_binding:\\n                            raise ValueError(\'Published lookup producer/folder changed.\')\\n                        self._snapshot_dirty = saved[\'receipts\'] < self._receipt_count\\n                if self._snapshot_dirty:\\n                    self._snapshot()\\n                self.progress(\'BGE API receipt lookup ready; no mount I/O for inventory\')\\n            elif pending or not (self.remote / \'current.json\').is_file():\\n                self._snapshot()\\n        except BaseException:\\n            self.close()\\n            raise\\n\\n    def _read_pending(self, pending, reused, total):\\n        # Keep at most two reads in flight: pool.map queued the whole directory\\n        # and discarded its temporary SQLite lookup after any mount failure.\\n        readers = 4 if self.receipt_api is not None else 2\\n        pool = ThreadPoolExecutor(max_workers=readers)\\n        iterator, futures = iter(pending), []\\n        count, last_log = 0, time.monotonic()\\n        def refill():\\n            while len(futures) < readers:\\n                path = next(iterator, None)\\n                if path is None:\\n                    break\\n                futures.append((path, pool.submit(lambda p: checked(\\n                    p.read_json() if self.receipt_api is not None else read_json(p)), path)))\\n        try:\\n            refill()\\n            while futures:\\n                # Register in native filename order, preserving the first\\n                # existing vector reference when blocks contain overlapping text.\\n                path, future = futures.pop(0)\\n                block = future.result()\\n                if path.name != \'block-\' + digest_json(block[\'keys\']) + \'.done.json\':\\n                    raise ValueError(\'Content receipt name/key binding changed.\')\\n                self.register(block)\\n                count += 1\\n                if count % 32 == 0 or time.monotonic()-last_log >= 15 or count == len(pending):\\n                    self.progress(f\'BGE receipt inventory: {reused+count:,}/{total:,}\'\\n                        f\' | reused {reused:,}, newly indexed {count:,} | local checkpoint committed\')\\n                    last_log = time.monotonic()\\n                refill()\\n        except BaseException:\\n            pool.shutdown(wait=False, cancel_futures=True)\\n            raise\\n        else:\\n            pool.shutdown(wait=True)\\n\\n    def register(self, block):\\n        checked(block)\\n        marker = \'block-\' + digest_json(block[\'keys\']) + \'.done.json\'\\n        old = self.con.execute(\'SELECT digest FROM receipts WHERE name=?\', [marker]).fetchone()\\n        if old is not None and old[0] != block[\'manifest_sha256\']:\\n            raise ValueError(\'Immutable content receipt changed in the retained lookup.\')\\n        try:\\n            # Keys and their receipt commit together; interrupted reads never\\n            # advance the inventory or erase an earlier committed prefix.\\n            if self.receipt_api is None:\\n                super().register(block)\\n            else:\\n                if block[\'identity\'] != self.identity or len(block[\'keys\']) != block[\'rows\']:\\n                    raise ValueError(\'Content vector producer/key mismatch.\')\\n                relative = PurePosixPath(block[\'path\'])\\n                if relative.is_absolute() or \'..\' in relative.parts or \'\\\\\\\\\' in block[\'path\'] or not relative.parts:\\n                    raise ValueError(\'API content reference escapes its data root.\')\\n                # No filesystem resolution through FUSE during API inventory.\\n                # Native vector readers still resolve the actual path and check\\n                # its SHA256/shape before consuming the vector payload.\\n                self.con.executemany(\'INSERT OR IGNORE INTO vectors VALUES (?,?,?,?)\',\\n                    [(key,block[\'path\'],block[\'sha256\'],i) for i,key in enumerate(block[\'keys\'])])\\n            self.con.execute(\'INSERT OR IGNORE INTO receipts VALUES (?,?)\', [marker, block[\'manifest_sha256\']])\\n            self.con.commit()\\n            if old is None:\\n                self._receipt_count += 1\\n                self._snapshot_dirty = True\\n        except BaseException:\\n            self.con.rollback()\\n            raise\\n        if (self.receipt_api is not None and self._receipt_count >= self._next_snapshot_count\\n            and hasattr(self.receipt_api, \'publish_lookup\')):\\n            self._next_snapshot_count = max(256, self._receipt_count*2)\\n            self._snapshot()\\n\\n    def _restore(self):\\n        remote = self.remote\\n        marker = remote / \'current.json\'\\n        if self.receipt_api is None:\\n            if not marker.is_file():\\n                return\\n            saved = checked(read_json(marker))\\n        else:\\n            value = self.receipt_api.relative_json(marker.relative_to(self.root).as_posix())\\n            if value is None and self.legacy_remote != self.remote:\\n                remote = self.legacy_remote\\n                value = self.receipt_api.relative_json((remote / \'current.json\').relative_to(self.root).as_posix())\\n            if value is None:\\n                return\\n            saved = checked(value)\\n        if saved.get(\'binding\') != (self.snapshot_binding if remote == self.remote else self.binding):\\n            raise ValueError(\'Published receipt lookup producer/root changed.\')\\n        relative = PurePosixPath(saved[\'path\'])\\n        if relative.is_absolute() or len(relative.parts) != 1 or \'\\\\\\\\\' in saved[\'path\']:\\n            raise ValueError(\'Published receipt snapshot path escapes its directory.\')\\n        self.progress(\'BGE receipt inventory: restore one compressed lookup; no old receipt JSON scan\')\\n        local = self.local / \'lookup.sqlite.gz\'\\n        if self.receipt_api is None:\\n            shutil.copyfile(bound(remote, saved[\'path\']), local)\\n        else:\\n            source = self.receipt_api.relative((remote / saved[\'path\']).relative_to(self.root).as_posix())\\n            if source is None:\\n                raise ValueError(\'Published API receipt lookup payload is missing.\')\\n            self.receipt_api.download(source, local, expected_sha256=saved[\'sha256\'])\\n        verify_file(local, saved[\'sha256\'])\\n        stage = self.lookup / \'restore.sqlite\'\\n        try:\\n            with gzip.open(local, \'rb\') as src, stage.open(\'wb\') as dst:\\n                shutil.copyfileobj(src, dst, length=1024*1024)\\n            verify_file(stage, saved[\'sqlite_sha256\'])\\n            # A shared folder has the same identity on another account even when\\n            # Colab exposes it through a different MyDrive/shortcut path.\\n            old_binding = saved.get(\'lookup_binding\', saved[\'binding\'])\\n            restored = sqlite3.connect(stage)\\n            try:\\n                rows = restored.execute(\'SELECT value FROM metadata\').fetchall()\\n                if rows != [(digest_json(old_binding),)] or old_binding.get(\'identity\') != self.identity:\\n                    raise ValueError(\'Restored lookup SQLite binding changed.\')\\n                if restored.execute(\'SELECT count(*) FROM receipts\').fetchone()[0] != saved[\'receipts\']:\\n                    raise ValueError(\'Restored lookup receipt count changed.\')\\n                if old_binding != self.binding:\\n                    restored.execute(\'UPDATE metadata SET value=?\', [digest_json(self.binding)])\\n                    restored.commit()\\n            finally:\\n                restored.close()\\n            stage.replace(self.database)\\n        finally:\\n            stage.unlink(missing_ok=True)\\n\\n    def _snapshot(self):\\n        if self._snapshot_busy:\\n            return\\n        if self.receipt_api is not None and not hasattr(self.receipt_api, \'publish_lookup\'):\\n            return  # Compatibility with older read-only transports; local prefix remains usable.\\n        self._snapshot_busy = True\\n        self.progress(\'BGE receipt inventory: save compressed lookup for future runtimes\')\\n        copy = self.local / \'lookup-backup.sqlite\'\\n        payload = self.local / \'lookup.sqlite.gz\'\\n        try:\\n            backup = sqlite3.connect(copy)\\n            try:\\n                self.con.backup(backup)\\n            finally:\\n                backup.close()\\n            checksum = sha256_file(copy)\\n            with copy.open(\'rb\') as src, gzip.open(payload, \'wb\', compresslevel=1) as dst:\\n                shutil.copyfileobj(src, dst, length=1024*1024)\\n            target = self.remote / (\'lookup-\' + sha256_file(payload) + \'.sqlite.gz\')\\n            record = seal({\'binding\': self.snapshot_binding, \'lookup_binding\': self.binding, \'path\': target.name,\\n                \'sha256\': sha256_file(payload), \'sqlite_sha256\': checksum,\\n                \'receipts\': self.con.execute(\'SELECT count(*) FROM receipts\').fetchone()[0],\\n                \'updated_at\': utc_now(),\\n                \'scope\': \'Receipt lookup only; vector payloads remain subject to native SHA/shape checks\'})\\n            if self.receipt_api is None:\\n                publish_file(payload, target)\\n                atomic_json(self.remote / \'current.json\', record)\\n            else:\\n                self.receipt_api.publish_lookup((self.remote / \'current.json\').relative_to(self.root).as_posix(),\\n                    payload, record)\\n            self._snapshot_dirty = False\\n            self.progress(f\\"BGE receipt lookup SAVED_ON_DRIVE: {record[\'receipts\']:,} receipts; next runtime restores this lookup\\")\\n        except (OSError, LookupPublishUnavailable) as error:\\n            # Optional acceleration only: the local committed database survives,\\n            # and native reads/writes still surface actual mount/space failures.\\n            self.progress(f\'BGE receipt lookup snapshot deferred: {type(error).__name__}; local checkpoint retained\')\\n        finally:\\n            self._snapshot_busy = False\\n            copy.unlink(missing_ok=True)\\n            payload.unlink(missing_ok=True)\\n\\n    def close(self):\\n        if self.con is not None:\\n            try:\\n                if self.receipt_api is not None and self._snapshot_dirty:\\n                    self._snapshot()\\n            finally:\\n                self.con.close()\\n                self.con = None\\n        self.temporary.__exit__(None, None, None)\\n", "new_data_drive_api.py": "\\"\\"\\"API receipt reads and narrowly scoped lookup checkpoints, without Drive FUSE.\\"\\"\\"\\nimport copy\\nimport hashlib\\nimport json\\nfrom pathlib import Path, PurePosixPath\\nimport re\\nimport threading\\nimport time\\n\\n\\nclass LookupPublishUnavailable(RuntimeError):\\n    \\"\\"\\"An optional lookup upload failed; paid vector receipts remain untouched.\\"\\"\\"\\n\\n\\nclass RemoteReceipt:\\n    def __init__(self, source, metadata):\\n        self.source, self.metadata = source, metadata\\n        self.name = metadata[\'name\']\\n\\n    def read_json(self):\\n        return self.source.read_json(self.metadata)\\n\\n\\nclass DriveReceiptSource:\\n    endpoint = \'https://www.googleapis.com/drive/v3/files\'\\n\\n    def __init__(self, session_factory, data_root, canonical_root, *,\\n                 data_folder_id=\'1Ez1S9jk6FO_puxC8r2vPSc0LJbuOkkdj\', report=print):\\n        if not re.fullmatch(r\'[A-Za-z0-9_-]+\', data_folder_id):\\n            raise ValueError(\'Invalid Drive data folder ID.\')\\n        self.session_factory, self.sessions = session_factory, threading.local()\\n        self.requested_root, self.data_root = Path(data_root).absolute(), Path(canonical_root)\\n        self.folder_id, self.report, self.folders = data_folder_id, report, {}\\n        self.payload_directories = {}\\n\\n    def directory_files(self, relative_directory, *, refresh=False):\\n        \\"\\"\\"One paginated metadata listing, rather than one lookup per payload.\\"\\"\\"\\n        relative = PurePosixPath(relative_directory)\\n        if relative.is_absolute() or \'..\' in relative.parts or \'\\\\\\\\\' in relative_directory or not relative.parts:\\n            raise ValueError(\'Invalid payload directory.\')\\n        if not refresh and relative_directory in self.payload_directories:\\n            return self.payload_directories[relative_directory]\\n        parent = self.folder_id\\n        for part in relative.parts:\\n            if (parent, part) not in self.folders:\\n                value = self.child(parent, part, folder=True)\\n                if value is None:\\n                    return {}\\n                self.folders[parent, part] = value[\'id\']\\n            parent = self.folders[parent, part]\\n        result = {}\\n        for value in self._list(f\\"\'{parent}\' in parents and trashed=false\\"):\\n            if value[\'name\'] in result:\\n                raise ValueError(\'Duplicate payload names in Drive directory.\')\\n            result[value[\'name\']] = value\\n        self.payload_directories[relative_directory] = result\\n        return result\\n\\n    def payload_metadata(self, relative_path):\\n        relative = PurePosixPath(relative_path)\\n        if relative.is_absolute() or \'..\' in relative.parts or \'\\\\\\\\\' in relative_path or len(relative.parts) < 2:\\n            raise ValueError(\'Invalid immutable payload path.\')\\n        directory = relative.parent.as_posix()\\n        result = self.directory_files(directory).get(relative.name)\\n        if result is None:\\n            result = self.directory_files(directory, refresh=True).get(relative.name)\\n        if result is None or \'size\' not in result:\\n            raise ValueError(\'Expected immutable payload is missing from Drive.\')\\n        return result\\n\\n    def _session(self):\\n        if not hasattr(self.sessions, \'session\'):\\n            self.sessions.session = self.session_factory()\\n        return self.sessions.session\\n\\n    def _get(self, url, params, *, stream=False):\\n        import requests\\n        session = self._session()\\n        for attempt in range(4):\\n            try:\\n                response = session.get(url, params=params, stream=stream, timeout=(10,60))\\n            except requests.RequestException:\\n                if attempt == 3:\\n                    raise\\n            else:\\n                if response.status_code not in (429,500,502,503,504) or attempt == 3:\\n                    response.raise_for_status()\\n                    return response\\n                response.close()\\n            self.report(f\'Drive receipt API: transient request failure; retry {attempt+1}/3\')\\n            time.sleep(2**attempt)\\n\\n    def _list(self, query):\\n        token, seen = None, set()\\n        while True:\\n            params = {\'q\': query, \'pageSize\': 1000, \'supportsAllDrives\': \'true\',\\n                \'includeItemsFromAllDrives\': \'true\',\\n                \'fields\': \'nextPageToken,incompleteSearch,files(id,name,mimeType,size,md5Checksum)\'}\\n            if token:\\n                params[\'pageToken\'] = token\\n            with self._get(self.endpoint, params) as response:\\n                page = response.json()\\n            if page.get(\'incompleteSearch\'):\\n                raise ValueError(\'Drive receipt listing is incomplete; refusing partial inventory.\')\\n            yield from page.get(\'files\', [])\\n            token = page.get(\'nextPageToken\')\\n            if not token:\\n                return\\n            if token in seen:\\n                raise ValueError(\'Drive receipt pagination repeated; inventory is incomplete.\')\\n            seen.add(token)\\n\\n    def child(self, parent, name, *, folder=False):\\n        safe = name.replace(\'\\\\\\\\\', \'\\\\\\\\\\\\\\\\\').replace(\\"\'\\", \\"\\\\\\\\\'\\")\\n        query = f\\"\'{parent}\' in parents and trashed=false and name=\'{safe}\'\\"\\n        if folder:\\n            query += \\" and mimeType=\'application/vnd.google-apps.folder\'\\"\\n        values = list(self._list(query))\\n        if len(values) > 1:\\n            raise ValueError(\'Duplicate Drive path names; refusing ambiguous receipt source.\')\\n        return values[0] if values else None\\n\\n    def relative(self, name):\\n        relative = PurePosixPath(name)\\n        if relative.is_absolute() or \'..\' in relative.parts or \'\\\\\\\\\' in name or not relative.parts:\\n            raise ValueError(\'Invalid relative Drive receipt path.\')\\n        parent = self.folder_id\\n        for part in relative.parts[:-1]:\\n            key = parent, part\\n            if key not in self.folders:\\n                value = self.child(parent, part, folder=True)\\n                if value is None:\\n                    return None\\n                self.folders[key] = value[\'id\']\\n            parent = self.folders[key]\\n        return self.child(parent, relative.name)\\n\\n    def download(self, metadata, target, *, expected_sha256=None, max_bytes=None):\\n        size = int(metadata[\'size\'])\\n        if max_bytes is not None and size > max_bytes:\\n            raise ValueError(\'Receipt exceeds the bounded JSON size.\')\\n        sha, md5, count = hashlib.sha256(), hashlib.md5(), 0\\n        with self._get(self.endpoint+\'/\'+metadata[\'id\'], {\'alt\':\'media\',\'supportsAllDrives\':\'true\'}, stream=True) as response:\\n            with Path(target).open(\'wb\') as output:\\n                for chunk in response.iter_content(chunk_size=1024*1024):\\n                    count += len(chunk)\\n                    if count > size or (max_bytes is not None and count > max_bytes):\\n                        raise ValueError(\'Drive receipt payload exceeds its declared size.\')\\n                    sha.update(chunk)\\n                    md5.update(chunk)\\n                    output.write(chunk)\\n        if (count != size or (metadata.get(\'md5Checksum\') and md5.hexdigest() != metadata[\'md5Checksum\'])\\n            or (expected_sha256 and sha.hexdigest() != expected_sha256)):\\n            raise ValueError(\'Drive receipt size/checksum mismatch.\')\\n\\n    def read_json(self, metadata):\\n        import tempfile\\n        with tempfile.TemporaryDirectory(prefix=\'vmb-api-json-\') as directory:\\n            path = Path(directory) / \'receipt.json\'\\n            self.download(metadata, path, max_bytes=16*1024*1024)\\n            return json.loads(path.read_text(encoding=\'utf-8\'))\\n\\n    def relative_json(self, path):\\n        metadata = self.relative(path)\\n        return None if metadata is None else self.read_json(metadata)\\n\\n    def verify_candidate(self, expected):\\n        active = self.relative_json(\'active_data_candidate.json\')\\n        if active is None or active.get(\'candidate_manifest_sha256\') != expected:\\n            raise ValueError(\'Drive API data folder differs from the expected frozen candidate.\')\\n        candidate = self.relative_json(\'processed/\'+active[\'build_run\']+\'/\'+active[\'candidate_name\'])\\n        from vietmedbridge.artifacts import digest_json\\n        if (candidate is None or candidate.get(\'candidate_manifest_sha256\') != expected\\n            or digest_json({k:v for k,v in candidate.items() if k != \'candidate_manifest_sha256\'}) != expected):\\n            raise ValueError(\'Drive API candidate digest changed.\')\\n        self.active, self.candidate = active, candidate\\n\\n    def receipts(self, identity):\\n        from vietmedbridge.artifacts import digest_json\\n        # Resolve the folder itself without relying on any receipt being present.\\n        cache = self.child(self.folder_id, \'model_cache\', folder=True)\\n        folder = self.child(cache[\'id\'], digest_json(identity)[:24], folder=True) if cache else None\\n        if folder is None:\\n            return []\\n        result, names, payloads = [], set(), {}\\n        # Drive name-contains is prefix matching, not a suffix predicate. List\\n        # the complete producer folder then filter receipt names locally.\\n        query = f\\"\'{folder[\'id\']}\' in parents and trashed=false\\"\\n        for value in self._list(query):\\n            if value[\'name\'] in payloads:\\n                raise ValueError(\'Duplicate names in content cache directory.\')\\n            payloads[value[\'name\']] = value\\n            if not value[\'name\'].endswith(\'.done.json\'):\\n                continue\\n            if (not re.fullmatch(r\'block-[0-9a-f]{64}\\\\.done\\\\.json\', value[\'name\'])\\n                or value[\'name\'] in names or \'size\' not in value):\\n                raise ValueError(\'Invalid or duplicate API receipt filename.\')\\n            names.add(value[\'name\'])\\n            result.append(RemoteReceipt(self, value))\\n        self.payload_directories[\'model_cache/\'+digest_json(identity)[:24]] = payloads\\n        self.report(f\'Drive receipt API: complete listing, {len(result):,} receipts; no mount reads\')\\n        return sorted(result, key=lambda value: value.name)\\n\\n    def publish_cache_checkpoint(self, relative_marker, payload, marker):\\n        \\"\\"\\"Only mapping bundles and exact ANN training samples may be published here.\\"\\"\\"\\n        from vietmedbridge.artifacts import digest_json\\n        binding = marker.get(\'binding\', {})\\n        source = binding.get(\'source_manifest_sha256\', \'\')\\n        kind = binding.get(\'kind\')\\n        if kind == \'mapping-bundle-v1\':\\n            directory, prefix, suffix = \'retrieval/mapping_bundles/\'+source, \'maps-\', \'.tar.gz\'\\n        elif kind == \'index-sample-v1\' and re.fullmatch(r\'[0-9a-f]{64}\', binding.get(\'positions_sha256\', \'\')):\\n            directory = \'retrieval/index_samples/\'+source+\'/\'+binding[\'positions_sha256\']\\n            prefix, suffix = \'sample-\', \'.npy\'\\n        else:\\n            raise ValueError(\'Unsupported cache checkpoint kind.\')\\n        if (binding.get(\'data_folder_id\') != self.folder_id or binding.get(\'schema_version\') != 1\\n            or not re.fullmatch(r\'[0-9a-f]{64}\', source)\\n            or relative_marker != directory+\'/current.json\'\\n            or not re.fullmatch(r\'[0-9a-f]{64}\', marker.get(\'sha256\', \'\'))\\n            or marker.get(\'path\') != prefix+marker[\'sha256\']+suffix\\n            or marker.get(\'manifest_sha256\') != digest_json({k:v for k,v in marker.items() if k != \'manifest_sha256\'})):\\n            raise ValueError(\'Cache checkpoint escapes its declared source/folder binding.\')\\n        sha, md5 = hashlib.sha256(), hashlib.md5()\\n        with Path(payload).open(\'rb\') as stream:\\n            for chunk in iter(lambda: stream.read(1024*1024), b\'\'):\\n                sha.update(chunk); md5.update(chunk)\\n        if sha.hexdigest() != marker[\'sha256\']:\\n            raise ValueError(\'Cache checkpoint payload changed before upload.\')\\n        try:\\n            return self._publish_lookup_files(directory, Path(payload), marker, md5.hexdigest())\\n        except ValueError:\\n            raise\\n        except Exception as error:\\n            raise LookupPublishUnavailable(type(error).__name__) from error\\n\\n    def publish_lookup(self, relative_marker, payload, marker):\\n        \\"\\"\\"Upload a checksum-addressed lookup, then its pointer. Never write vector/source files.\\"\\"\\"\\n        from vietmedbridge.artifacts import digest_json\\n        body = {k:v for k,v in marker.items() if k != \'manifest_sha256\'}\\n        binding = marker.get(\'binding\', {})\\n        expected_directory = \'retrieval/content_lookups/\' + digest_json(binding)[:16]\\n        if (relative_marker != expected_directory + \'/current.json\'\\n            or binding.get(\'schema_version\') != 2 or binding.get(\'data_folder_id\') != self.folder_id\\n            or marker.get(\'manifest_sha256\') != digest_json(body)\\n            or marker.get(\'path\') != \'lookup-\' + marker.get(\'sha256\', \'\') + \'.sqlite.gz\'\\n            or not re.fullmatch(r\'lookup-[0-9a-f]{64}\\\\.sqlite\\\\.gz\', marker.get(\'path\', \'\'))):\\n            raise ValueError(\'Lookup upload is outside its bound checkpoint directory.\')\\n        sha, md5 = hashlib.sha256(), hashlib.md5()\\n        with Path(payload).open(\'rb\') as stream:\\n            for chunk in iter(lambda: stream.read(1024*1024), b\'\'):\\n                sha.update(chunk)\\n                md5.update(chunk)\\n        if sha.hexdigest() != marker[\'sha256\']:\\n            raise ValueError(\'Lookup payload changed before upload.\')\\n        try:\\n            return self._publish_lookup_files(expected_directory, Path(payload), marker, md5.hexdigest())\\n        except ValueError:\\n            raise\\n        except Exception as error:\\n            # Do not include provider messages/URLs: resumable upload URLs are private.\\n            raise LookupPublishUnavailable(type(error).__name__) from error\\n\\n    def _publish_lookup_files(self, directory, payload, marker, md5):\\n        from vietmedbridge.artifacts import digest_json\\n        from googleapiclient.discovery import build\\n        from googleapiclient.http import MediaFileUpload, MediaIoBaseUpload\\n        import io\\n        service = build(\'drive\', \'v3\', credentials=copy.copy(self._session().credentials), cache_discovery=False)\\n        files = service.files()\\n        fields = \'id,name,mimeType,size,md5Checksum\'\\n\\n        def new_id():\\n            return files.generateIds(count=1, space=\'drive\', type=\'files\').execute(num_retries=3)[\'ids\'][0]\\n\\n        def ensure_folder(parent, name):\\n            value = self.child(parent, name, folder=True)\\n            if value is None:\\n                identifier = new_id()\\n                try:\\n                    value = files.create(body={\'id\':identifier, \'name\':name, \'parents\':[parent],\\n                        \'mimeType\':\'application/vnd.google-apps.folder\'}, fields=fields,\\n                        supportsAllDrives=True).execute(num_retries=3)\\n                except Exception:\\n                    # A lost response may follow a successful create. Reconcile the\\n                    # preallocated ID; never retry a folder with a different ID.\\n                    value = files.get(fileId=identifier, fields=fields, supportsAllDrives=True).execute(num_retries=3)\\n                    if value.get(\'name\') != name or value.get(\'mimeType\') != \'application/vnd.google-apps.folder\':\\n                        raise ValueError(\'Created lookup folder identity changed.\')\\n            self.folders[parent, name] = value[\'id\']\\n            return value[\'id\']\\n\\n        parent = self.folder_id\\n        for name in PurePosixPath(directory).parts:\\n            parent = ensure_folder(parent, name)\\n        current = self.child(parent, \'current.json\')\\n        if current is not None:\\n            existing = self.read_json(current)\\n            if (existing.get(\'manifest_sha256\') != digest_json(\\n                    {k:v for k,v in existing.items() if k != \'manifest_sha256\'})\\n                or existing.get(\'binding\') != marker[\'binding\']):\\n                raise ValueError(\'Existing lookup checkpoint binding/digest changed.\')\\n            count_key = \'receipts\' if \'receipts\' in marker else \'parts\'\\n            if existing[count_key] > marker[count_key]:\\n                return existing  # An older local prefix must not replace a newer saved lookup.\\n\\n        def upload(media, name, existing=None):\\n            if existing is None:\\n                request = files.create(body={\'id\':new_id(), \'name\':name, \'parents\':[parent]},\\n                    media_body=media, fields=fields, supportsAllDrives=True)\\n            else:\\n                request = files.update(fileId=existing[\'id\'], media_body=media,\\n                    fields=fields, supportsAllDrives=True)\\n            result = None\\n            while result is None:\\n                _, result = request.next_chunk(num_retries=3)\\n            # Query provider metadata independently before publishing a pointer.\\n            return files.get(fileId=result[\'id\'], fields=fields, supportsAllDrives=True).execute(num_retries=3)\\n\\n        saved = self.child(parent, marker[\'path\'])\\n        if saved is None:\\n            mime = \'application/gzip\' if marker[\'path\'].endswith(\'.gz\') else \'application/octet-stream\'\\n            saved = upload(MediaFileUpload(str(payload), mimetype=mime,\\n                chunksize=8*1024*1024, resumable=True), marker[\'path\'])\\n        if int(saved.get(\'size\', -1)) != payload.stat().st_size or saved.get(\'md5Checksum\') != md5:\\n            raise ValueError(\'Uploaded lookup payload size/checksum mismatch; pointer not advanced.\')\\n        data = json.dumps(marker, ensure_ascii=False, sort_keys=True).encode(\'utf-8\')\\n        saved_marker = upload(MediaIoBaseUpload(io.BytesIO(data), mimetype=\'application/json\',\\n            chunksize=256*1024, resumable=True), \'current.json\', current)\\n        if int(saved_marker.get(\'size\', -1)) != len(data) or saved_marker.get(\'md5Checksum\') != hashlib.md5(data).hexdigest():\\n            raise ValueError(\'Uploaded lookup pointer size/checksum mismatch.\')\\n        return marker\\n\\n\\ndef authenticate_drive_receipts(data_root, expected_candidate_sha256):\\n    \\"\\"\\"Authenticate on the main Colab thread; never print/save credential tokens.\\"\\"\\"\\n    from google.colab import auth\\n    import google.auth\\n    from google.auth.transport.requests import AuthorizedSession\\n    root = Path(data_root).resolve()\\n    print(\'Drive receipts: authenticate API; receipts read-only, lookup checkpoints saved in retrieval/content_lookups\', flush=True)\\n    auth.authenticate_user()\\n    credentials, _ = google.auth.default()\\n    client = DriveReceiptSource(lambda: AuthorizedSession(copy.copy(credentials)), data_root, root)\\n    client.verify_candidate(expected_candidate_sha256)\\n    print(\'Drive receipts: API source verified against the same frozen candidate\', flush=True)\\n    return client\\n"}')

for name, source in SOURCES.items():
    (WORK_DIR/name).write_text(source, encoding='utf-8')
for name in SOURCES:
    sys.modules.pop(Path(name).stem, None)
for module_name in ('new_data_kaggle','new_data_embeddings','new_data_cache_lookup','new_data_drive_api'):
    sys.modules.pop(module_name,None)
importlib.invalidate_caches()
import new_data_kaggle as pipeline
from new_data_embeddings import embedding_contract
from new_data_drive_api import DriveReceiptSource
import google.auth
from google.auth.transport.requests import AuthorizedSession
credentials, _ = google.auth.default()
config, runtime, identities, anchor = embedding_contract(PROJECT/'data', CHECKOUT)
data = (PROJECT/'data').resolve()
api_source = DriveReceiptSource(lambda: AuthorizedSession(copy.copy(credentials)), data, data)
api_source.verify_candidate(anchor)
print('Runtime GPU khóa từ cache cũ:', runtime, flush=True)


(WORK_DIR/'recovery_kaggle_batch.py').write_text('"""Read frozen recovery manifests; share the existing sparse Kaggle worker/importer."""\nfrom pathlib import Path\nimport re\n\nfrom vietmedbridge.artifacts import atomic_json, digest_json, read_json\nfrom vietmedbridge.content_embeddings import checked, seal\nfrom vietmedbridge.scale_vectors import bound\n\nCAMPAIGN = \'data_staging/btc-maximize-71be0e12-v1\'\n\n\ndef load_recovery_batch(project, batch_dir=None):\n    project = Path(project).resolve()\n    parent = project / CAMPAIGN\n    if batch_dir is None:\n        frozen = checked(read_json(parent / \'delta_candidates.json\'))\n        root = parent / \'embedding_batches\' / frozen[\'manifest_sha256\'][:16]\n        pointer = root / \'frozen_deltas.json\'\n        if pointer.is_file():\n            if checked(read_json(pointer)) != frozen:\n                raise ValueError(\'Recovery handoff already exists with different manifests.\')\n        else:\n            atomic_json(pointer, frozen)\n    else:\n        root = bound(project, batch_dir)\n        if root.parent != parent / \'embedding_batches\' or not re.fullmatch(\'[0-9a-f]{16}\', root.name):\n            raise ValueError(\'Unexpected recovery embedding batch directory.\')\n        frozen = checked(read_json(root / \'frozen_deltas.json\'))\n        if root.name != frozen[\'manifest_sha256\'][:16]:\n            raise ValueError(\'Recovery handoff directory checksum mismatch.\')\n    if frozen.get(\'submission_candidate_changed\') is not False or not frozen.get(\'builds\'):\n        raise ValueError(\'A nonempty immutable recovery handoff is required.\')\n    sources, builds, names = [], [], set()\n    for item in frozen[\'builds\']:\n        name = item[\'build_run\']\n        if (item.get(\'state\') != \'DELTA_READY_FOR_EMBEDDING\'\n                or not isinstance(item.get(\'data_root\'), str) or item[\'data_root\'].replace(\'\\\\\', \'/\') != CAMPAIGN\n                or not re.fullmatch(r\'(metadata-text-v1|raw-text-v1|raw-ready-[0-9a-f]{16})\', name)\n                or name in names or item.get(\'submission_candidate_changed\') is not False):\n            raise ValueError(\'Recovery source coverage or producer scope changed.\')\n        names.add(name)\n        build = parent / \'processed\' / name\n        expected_name = \'candidate-\' + item[\'candidate_manifest_sha256\'][:16] + \'.json\'\n        if item[\'candidate_name\'] != expected_name:\n            raise ValueError(\'Unexpected frozen candidate path.\')\n        candidate = read_json(build / expected_name)\n        expected = {k: v for k, v in candidate.items() if k != \'candidate_manifest_sha256\'}\n        if (digest_json(expected) != item[\'candidate_manifest_sha256\']\n                or candidate.get(\'candidate_manifest_sha256\') != item[\'candidate_manifest_sha256\']\n                or candidate.get(\'state\') != \'FROZEN_CANDIDATE\'\n                or candidate.get(\'snapshot_sha256\') != item[\'snapshot_sha256\']\n                or candidate.get(\'counts\', {}).get(\'documents\') != item[\'documents\']\n                or not candidate.get(\'golden\', {}).get(\'passed\')\n                or candidate.get(\'health\', {}).get(\'integrity\', {}).get(\'official_membership\') != \'VERIFIED\'\n                or not candidate.get(\'health\', {}).get(\'integrity\', {}).get(\'passed\')):\n            raise ValueError(\'Recovery candidate is not the validated frozen snapshot.\')\n        inputs = checked(read_json(build / \'index_inputs/units.json\'))\n        if (inputs.get(\'state\') != \'COMPLETE\' or inputs[\'manifest_sha256\'] != item[\'index_inputs_manifest_sha256\']\n                or inputs[\'candidate_manifest_sha256\'] != item[\'candidate_manifest_sha256\']\n                or inputs[\'input_count\'] != item[\'unique_dense_inputs\']):\n            raise ValueError(\'Recovery model-input binding changed.\')\n        record = {**item, \'build_dir\': build.relative_to(project).as_posix()}\n        cursor = 0\n        for part in inputs[\'parts\']:\n            if (part[\'start\'] != cursor or part[\'rows\'] != min(4096, inputs[\'input_count\']-cursor)\n                    or part[\'signature\'] != inputs[\'signature\']):\n                raise ValueError(\'Recovery input part coverage changed.\')\n            bound(build / \'index_inputs\', part[\'path\'])\n            sources.append({\'ordinal\': len(sources), \'build\': record, \'inputs\': inputs, \'part\': part})\n            cursor += part[\'rows\']\n        if cursor != inputs[\'input_count\']:\n            raise ValueError(\'Recovery model-input range is incomplete.\')\n        builds.append(record)\n    batch = seal({\'state\': \'RECOVERY_INPUTS_READY\', \'frozen_deltas\': frozen[\'manifest_sha256\'],\n                  \'builds\': builds, \'documents\': sum(b[\'documents\'] for b in builds),\n                  \'input_parts\': len(sources),\n                  \'input_rows_before_cross_batch_dedup\': sum(b[\'unique_dense_inputs\'] for b in builds),\n                  \'scope\': \'Frozen recovery deltas only; corpus embedding coverage is separate.\'})\n    return root, batch, sources\n',encoding='utf-8')
sys.modules.pop('recovery_kaggle_batch',None)

(WORK_DIR/'full_btc_embeddings.py').write_text('"""All validated sources, with a durable audit of current BTC ID coverage.\n\nNo crawl, extraction, chunking, catalog rebuild or vector payload scan. Legacy\nvalidated sources are retained as requested; current BTC IDs have a separate\nmembership manifest so document-row totals cannot masquerade as URL coverage.\n"""\nimport gzip\nimport hashlib\nfrom pathlib import Path\nimport shutil\nimport sqlite3\nimport tempfile\nimport time\n\nimport pyarrow as pa\nimport pyarrow.parquet as pq\n\nfrom vietmedbridge.artifacts import atomic_json, digest_json, publish_file, read_json, utc_now, verify_file\nfrom vietmedbridge.content_embeddings import checked, seal\nfrom vietmedbridge.portable_embeddings import resources_relative\nfrom vietmedbridge.scale_vectors import bound\n\ntry:\n    from .new_data_embeddings import load_new_batch\n    from .recovery_kaggle_batch import load_recovery_batch\nexcept ImportError:\n    from new_data_embeddings import load_new_batch\n    from recovery_kaggle_batch import load_recovery_batch\n\nPARENT = \'data_staging/btc-full-embeddings-71be0e12-v1\'\nCOLLECTION = \'data_staging/btc-all-ai-guru-71be0e12-v1\'\nMAXIMIZE = \'data_staging/btc-maximize-71be0e12-v1\'\nNEW = \'data_staging/btc-new-71be0e12-20261019-v1\'\nMEMBERS = pa.schema([(\'official_id\', pa.int64()), (\'url\', pa.string()),\n                     (\'source\', pa.string()), (\'source_doc_id\', pa.int64())])\n\n\ndef _source(project, run, build_dir, candidate_name, candidate_sha, *, mapping=None, alias_sha=None, views=None):\n    build = bound(project, build_dir)\n    candidate = read_json(bound(build, candidate_name))\n    if (candidate.get(\'candidate_manifest_sha256\') != candidate_sha\n            or digest_json({k: v for k, v in candidate.items() if k != \'candidate_manifest_sha256\'}) != candidate_sha\n            or candidate.get(\'state\') != \'FROZEN_CANDIDATE\'\n            or not candidate.get(\'golden\', {}).get(\'passed\')\n            or not candidate.get(\'health\', {}).get(\'integrity\', {}).get(\'passed\')\n            or candidate[\'health\'][\'integrity\'].get(\'official_membership\') != \'VERIFIED\'):\n        raise ValueError(\'Unvalidated source in full-data handoff: \'+run)\n    if alias_sha is not None and candidate[\'health\'][\'files\'][\'canonical_aliases\'][\'sha256\'] != alias_sha:\n        raise ValueError(\'Baseline alias mapping does not belong to the frozen candidate.\')\n    inputs = checked(read_json(build / \'index_inputs/units.json\'))\n    if inputs.get(\'state\') != \'COMPLETE\' or inputs[\'candidate_manifest_sha256\'] != candidate_sha:\n        raise ValueError(\'Complete native model inputs are required: \'+run)\n    record = {\'build_run\': run, \'build_dir\': build.relative_to(project).as_posix(),\n              \'candidate_name\': candidate_name, \'candidate_manifest_sha256\': candidate_sha,\n              \'snapshot_sha256\': candidate[\'snapshot_sha256\'], \'documents\': candidate[\'counts\'][\'documents\'],\n              \'index_inputs_manifest_sha256\': inputs[\'manifest_sha256\'], \'unique_dense_inputs\': inputs[\'input_count\'],\n              \'mapping\': mapping, \'mapping_sha256\': candidate[\'health\'][\'files\'][\'canonical_aliases\'][\'sha256\'] if mapping else None,\n              \'completed_views\': {}}\n    for family, directory in (views or {}).items():\n        path = directory / \'embeddings.json\'\n        if path.is_file():\n            saved = checked(read_json(path))\n            if saved.get(\'state\') == \'COMPLETE\':\n                if saved.get(\'input_manifest_sha256\') != inputs[\'manifest_sha256\'] or saved.get(\'input_count\') != inputs[\'input_count\']:\n                    raise ValueError(\'Inherited model view belongs to another input manifest.\')\n                record[\'completed_views\'][family] = {\'path\': directory.relative_to(project).as_posix(),\n                    \'manifest_sha256\': saved[\'manifest_sha256\']}\n    return record\n\n\ndef inventory(project, checkout):\n    """Metadata only: baseline + ten frozen intake builds + ALL frozen recovery deltas."""\n    project, checkout = Path(project).resolve(), Path(checkout)\n    scope = read_json(project / NEW / \'full_intake_scope.json\')\n    baseline = scope[\'base_plan\'][\'baseline\']\n    active = read_json(project / \'data/active_data_candidate.json\')\n    if active[\'candidate_manifest_sha256\'] != baseline[\'candidate_manifest_sha256\']:\n        raise ValueError(\'Active candidate differs from the audited baseline; retain both before changing scope.\')\n    resources = project / \'data\' / resources_relative(active, read_json(checkout / \'configs/retrieval_full.json\'))\n    records = [_source(project, active[\'build_run\'], \'data/processed/\'+active[\'build_run\'], active[\'candidate_name\'],\n        active[\'candidate_manifest_sha256\'], mapping=NEW+\'/references/baseline_aliases.parquet\', alias_sha=baseline[\'sha256\'],\n        views={family: resources / family for family in (\'bge\', \'qwen\')})]\n    new_root, new_batch, _ = load_new_batch(project)\n    for item in new_batch[\'builds\']:\n        build = bound(project, item[\'build_dir\'])\n        candidate = read_json(bound(build, item[\'candidate_name\']))\n        alias = candidate[\'health\'][\'files\'][\'canonical_aliases\']\n        records.append(_source(project, item[\'build_run\'], item[\'build_dir\'], item[\'candidate_name\'],\n            item[\'candidate_manifest_sha256\'], mapping=bound(build, alias[\'path\']).relative_to(project).as_posix(),\n            views={family: new_root / \'embeddings_v1\' / family / item[\'build_run\'] for family in (\'bge\', \'qwen\')}))\n    recovery_root, recovery_batch, _ = load_recovery_batch(project)\n    for item in recovery_batch[\'builds\']:\n        records.append(_source(project, item[\'build_run\'], item[\'build_dir\'], item[\'candidate_name\'],\n            item[\'candidate_manifest_sha256\'], views={family: recovery_root / \'embeddings_v1\' / family / item[\'build_run\']\n                                                    for family in (\'bge\', \'qwen\')}))\n    if len({r[\'build_run\'] for r in records}) != len(records):\n        raise ValueError(\'Duplicate build names in full-data handoff.\')\n    report = checked(read_json(project / COLLECTION / \'collection_report.json\'))\n    if report[\'archive_errors\'] or report[\'counts_provisional\']:\n        raise ValueError(\'Resolve collection coverage before full-data export.\')\n    return {\'workflow\': \'FULL_VALIDATED_BTC_EMBEDDINGS_V1\', \'sources\': records,\n            \'collection_report\': report[\'manifest_sha256\'], \'expected_reused\': report[\'counts_by_official_id\'][\'REUSED_VALIDATED_CONTENT\'],\n            \'official_total\': report[\'official_ids\'],\n            \'collection_pointer\': checked(read_json(project / COLLECTION / \'checkpoint.json\')),\n            \'maximize_pointer\': checked(read_json(project / MAXIMIZE / \'checkpoint.json\')),\n            \'scope\': \'All frozen validated data, including retained legacy content; exact current BTC membership audited separately.\'}\n\n\ndef _restore(root, pointer, work):\n    """Restore one metadata SQL snapshot in Colab scratch, never on the user\'s PC."""\n    work.mkdir(parents=True, exist_ok=True)\n    target = work / (pointer[\'sqlite_sha256\']+\'.sqlite\')\n    marker = target.with_suffix(\'.verified.json\')\n    if target.is_file() and marker.is_file():\n        proof = read_json(marker)\n        if proof.get(\'sha256\') == pointer[\'sqlite_sha256\'] and proof.get(\'bytes\') == target.stat().st_size:\n            return target\n    print(\'FULL_SCOPE restore metadata SQL:\', root.name, \'| no TAR/corpus/vector reads\', flush=True)\n    with tempfile.TemporaryDirectory(dir=work) as temporary:\n        compressed, database = Path(temporary)/\'sql.gz\', Path(temporary)/\'sql.sqlite\'\n        sha, copied, logged = hashlib.sha256(), 0, time.monotonic()\n        with bound(root, pointer[\'path\']).open(\'rb\') as incoming, compressed.open(\'wb\') as out:\n            for block in iter(lambda: incoming.read(1024**2), b\'\'):\n                sha.update(block); out.write(block); copied += len(block)\n                if time.monotonic()-logged >= 15:\n                    print(\'FULL_SCOPE metadata transfer:\', root.name, \'|\', round(copied/1024**2, 1), \'MiB\', flush=True)\n                    logged = time.monotonic()\n        if sha.hexdigest() != pointer[\'sha256\']:\n            raise ValueError(\'Metadata SQL snapshot changed during restore.\')\n        with gzip.open(compressed, \'rb\') as incoming, database.open(\'wb\') as out:\n            shutil.copyfileobj(incoming, out, 1024**2)\n        verify_file(database, pointer[\'sqlite_sha256\'])\n        database.replace(target)\n    atomic_json(marker, {\'sha256\': pointer[\'sqlite_sha256\'], \'bytes\': target.stat().st_size})\n    return target\n\n\ndef prepare_full_scope(project, checkout, work, *, minimum_ids=802028):\n    project, work = Path(project).resolve(), Path(work)\n    parent = project / PARENT\n    active_path = parent / \'active_scope.json\'\n    if active_path.is_file():\n        active = checked(read_json(active_path))\n        root = bound(project, active[\'root\'])\n        contract = checked(read_json(root / \'source_contract.json\'))\n        if contract[\'manifest_sha256\'] != active[\'contract\']:\n            raise ValueError(\'Pinned full-data scope changed.\')\n    else:\n        contract = seal(inventory(project, checkout))\n        root = parent / contract[\'manifest_sha256\'][:16]\n        atomic_json(root / \'source_contract.json\', contract)\n        atomic_json(active_path, seal({\'root\': root.relative_to(project).as_posix(), \'contract\': contract[\'manifest_sha256\']}))\n    if root.parent != parent:\n        raise ValueError(\'Unexpected full-scope output directory.\')\n    ready = root / \'scope.json\'\n    if ready.is_file():\n        report = checked(read_json(ready))\n        if report.get(\'state\') != \'FULL_SCOPE_READY\' or report[\'contract\'] != contract[\'manifest_sha256\'] or report[\'official_ids\'] < minimum_ids:\n            raise ValueError(\'Full-data scope is incomplete or below the requested BTC coverage.\')\n        print(\'FULL_SCOPE_REUSED:\', report[\'official_ids\'], \'BTC IDs | no SQL or corpus replay\', flush=True)\n        return root, report\n    collection = _restore(project / COLLECTION, contract[\'collection_pointer\'], work)\n    maximum = _restore(project / MAXIMIZE, contract[\'maximize_pointer\'], work)\n    con = sqlite3.connect(collection.resolve().as_uri()+\'?mode=ro\', uri=True)\n    try:\n        con.execute(\'ATTACH DATABASE ? AS maximum\', [maximum.resolve().as_uri()+\'?mode=ro\'])\n        con.execute(\'PRAGMA query_only=ON\'); con.execute(\'PRAGMA cache_size=-32768\'); con.execute(\'PRAGMA temp_store=FILE\')\n        for database, key in ((\'main\', \'collection_pointer\'), (\'maximum\', \'maximize_pointer\')):\n            if con.execute(\'SELECT value FROM \'+database+\'.contract\').fetchone() != (digest_json(contract[key][\'binding\']),):\n                raise ValueError(\'Metadata SQL has a different source binding.\')\n        reused = con.execute(\'SELECT count(*) FROM official o JOIN content c ON o.fetch=c.fetch WHERE c.existing IS NOT NULL\').fetchone()[0]\n        added = con.execute("SELECT count(*) FROM maximum.aliases a JOIN maximum.todo t USING(fetch) WHERE t.state=\'CHUNKED\'").fetchone()[0]\n        overlap = con.execute("""SELECT count(*) FROM maximum.aliases a JOIN maximum.todo t USING(fetch)\n            JOIN official o ON o.id=a.id JOIN content c ON o.fetch=c.fetch\n            WHERE t.state=\'CHUNKED\' AND (c.existing IS NOT NULL OR o.url!=a.url)""").fetchone()[0]\n        expected = reused+added\n        if reused != contract[\'expected_reused\'] or overlap or expected < minimum_ids:\n            raise ValueError(f\'Full scope audit failed: reused={reused}, added={added}, overlap={overlap}; required >= {minimum_ids}.\')\n        mappings = {r[\'mapping\']: r[\'build_run\'] for r in contract[\'sources\'] if r[\'mapping\']}\n        proofs = dict(con.execute("SELECT json_extract(receipt,\'$.mapping\'),json_extract(receipt,\'$.sha256\') FROM completed WHERE kind=\'existing\'"))\n        references = {r[\'mapping\']: r[\'mapping_sha256\'] for r in contract[\'sources\'] if r[\'mapping\']}\n        for label, in con.execute("SELECT DISTINCT json_extract(existing,\'$.mapping\') FROM content WHERE existing IS NOT NULL"):\n            if label not in mappings or proofs.get(label) != references[label]:\n                raise ValueError(\'Validated alias provenance is missing from full scope: \'+str(label))\n        names = {r[\'build_run\'] for r in contract[\'sources\']}\n        phase = """CASE WHEN coalesce(json_extract(t.candidate,\'$.recovery_root\'),\'\')!=\'maximize\'\n            THEN \'metadata-text-v1\' ELSE coalesce((SELECT m.phase FROM maximum.ready_members m\n                WHERE m.fetch=a.fetch LIMIT 1),\'raw-text-v1\') END"""\n        sql = """SELECT o.id,o.url,\'existing\',json_extract(c.existing,\'$.mapping\'),json_extract(c.existing,\'$.original_doc_id\')\n            FROM official o JOIN content c ON o.fetch=c.fetch WHERE c.existing IS NOT NULL AND o.id>?\n            UNION ALL SELECT a.id,a.url,\'delta\',"""+phase+""",a.id FROM maximum.aliases a JOIN maximum.todo t USING(fetch)\n            JOIN official o ON o.id=a.id AND o.url=a.url JOIN content c ON o.fetch=c.fetch\n            WHERE t.state=\'CHUNKED\' AND c.existing IS NULL AND a.id>? ORDER BY 1"""\n        progress_path = root / \'membership_progress.json\'\n        progress = checked(read_json(progress_path)) if progress_path.is_file() else seal({\n            \'contract\': contract[\'manifest_sha256\'], \'last_id\': -1, \'rows\': 0, \'parts\': [], \'by_source\': {}})\n        if progress[\'contract\'] != contract[\'manifest_sha256\']:\n            raise ValueError(\'Membership checkpoint belongs to another scope.\')\n        cursor = con.execute(sql, [progress[\'last_id\'], progress[\'last_id\']])\n        while rows := cursor.fetchmany(8192):\n            values = []\n            for doc_id, url, kind, label, original in rows:\n                source = mappings.get(label) if kind == \'existing\' else label\n                if source not in names or original is None:\n                    raise ValueError(\'BTC ID lacks a frozen source: \'+str(doc_id)+\' | \'+str(label))\n                values.append({\'official_id\': doc_id, \'url\': url, \'source\': source, \'source_doc_id\': original})\n                progress[\'by_source\'][source] = progress[\'by_source\'].get(source, 0)+1\n            with tempfile.TemporaryDirectory(dir=work) as temporary:\n                path = Path(temporary)/\'membership.parquet\'\n                pq.write_table(pa.Table.from_pylist(values, schema=MEMBERS), path, compression=\'zstd\')\n                relative = f"membership/part-{progress[\'rows\']:09d}.parquet"\n                checksum = publish_file(path, root / relative)\n            progress[\'parts\'].append({\'path\': relative, \'sha256\': checksum, \'rows\': len(values)})\n            progress.update(last_id=rows[-1][0], rows=progress[\'rows\']+len(values))\n            progress = seal({k: v for k, v in progress.items() if k != \'manifest_sha256\'})\n            atomic_json(progress_path, progress)\n            print(\'FULL_SCOPE membership:\', f"{progress[\'rows\']:,}/{expected:,}", \'| checkpoint saved; no document/vector reads\', flush=True)\n        if progress[\'rows\'] != expected or sum(progress[\'by_source\'].values()) != expected:\n            raise ValueError(\'Full-data membership is incomplete; no GPU export is allowed.\')\n        for source in contract[\'sources\']:\n            if not source[\'mapping\'] and progress[\'by_source\'].get(source[\'build_run\'], 0) != source[\'documents\']:\n                raise ValueError(\'Frozen recovery documents are missing from BTC membership: \'+source[\'build_run\'])\n        result = seal({\'state\': \'FULL_SCOPE_READY\', \'contract\': contract[\'manifest_sha256\'],\n            \'official_ids\': expected, \'official_total\': contract[\'official_total\'], \'reused_ids\': reused, \'new_chunked_ids\': added,\n            \'source_builds\': len(contract[\'sources\']), \'source_documents_sum\': sum(r[\'documents\'] for r in contract[\'sources\']),\n            \'input_rows_before_cross_source_dedup\': sum(r[\'unique_dense_inputs\'] for r in contract[\'sources\']),\n            \'membership\': progress, \'scope\': contract[\'scope\'], \'submission_candidate_changed\': False, \'updated_at\': utc_now()})\n        atomic_json(ready, result)\n        print(\'FULL_SCOPE_READY:\', expected, \'BTC IDs |\', result[\'source_builds\'], \'frozen sources\', flush=True)\n        return root, result\n    finally:\n        con.close()\n\n\ndef load_full_batch(project, batch_dir, *, minimum_ids=802028):\n    project = Path(project).resolve()\n    root = bound(project, batch_dir)\n    if root.parent != project / PARENT:\n        raise ValueError(\'Unexpected full-data batch directory.\')\n    contract, report = checked(read_json(root / \'source_contract.json\')), checked(read_json(root / \'scope.json\'))\n    if (report.get(\'state\') != \'FULL_SCOPE_READY\' or report[\'contract\'] != contract[\'manifest_sha256\']\n            or report[\'official_ids\'] < minimum_ids):\n        raise ValueError(\'Full-data BTC membership has not been verified; no partial fallback.\')\n    sources = []\n    for record in contract[\'sources\']:\n        build = bound(project, record[\'build_dir\'])\n        inputs = checked(read_json(build / \'index_inputs/units.json\'))\n        if (inputs[\'state\'] != \'COMPLETE\' or inputs[\'manifest_sha256\'] != record[\'index_inputs_manifest_sha256\']\n                or inputs[\'candidate_manifest_sha256\'] != record[\'candidate_manifest_sha256\']\n                or inputs[\'input_count\'] != record[\'unique_dense_inputs\']):\n            raise ValueError(\'Full-data frozen model inputs changed.\')\n        cursor = 0\n        for part in inputs[\'parts\']:\n            if part[\'start\'] != cursor or part[\'rows\'] != min(4096, inputs[\'input_count\']-cursor) or part[\'signature\'] != inputs[\'signature\']:\n                raise ValueError(\'Full-data input part coverage changed.\')\n            bound(build / \'index_inputs\', part[\'path\'])\n            sources.append({\'ordinal\': len(sources), \'build\': record, \'inputs\': inputs, \'part\': part})\n            cursor += part[\'rows\']\n        if cursor != inputs[\'input_count\']:\n            raise ValueError(\'Full-data input range is incomplete.\')\n    return root, seal({\'state\': \'FULL_VALIDATED_INPUTS_READY\', \'scope_manifest_sha256\': report[\'manifest_sha256\'],\n        \'scope\': contract[\'scope\']+\' Corpus child embeddings only; query/search/reranker/submission remain separate.\',\n        \'source_builds\': report[\'source_builds\'], \'source_documents_sum\': report[\'source_documents_sum\'],\n        \'builds\': contract[\'sources\'], \'official_ids\': report[\'official_ids\'], \'input_parts\': len(sources),\n        \'input_rows_before_cross_source_dedup\': report[\'input_rows_before_cross_source_dedup\']}), sources\n',encoding='utf-8')
sys.modules.pop('full_btc_embeddings',None)

from full_btc_embeddings import prepare_full_scope,load_full_batch,PARENT
SCOPE_ROOT,SCOPE=prepare_full_scope(PROJECT,CHECKOUT,WORK_DIR/'metadata',minimum_ids=802028)
BATCH_DIR=SCOPE_ROOT.relative_to(PROJECT).as_posix()
def full_flow(work,*,accounts=2,gpu_count=2,batch_dir=None):
    return pipeline.NewDataKaggle(PROJECT,work,identities,accounts=accounts,gpu_count=gpu_count,
        batch_loader=lambda project:load_full_batch(project,batch_dir or BATCH_DIR,minimum_ids=802028))
print(json.dumps({k:SCOPE[k] for k in ('state','official_ids','official_total','reused_ids','new_chunked_ids',
    'source_builds','source_documents_sum','input_rows_before_cross_source_dedup')},ensure_ascii=False,indent=2),flush=True)
print('Membership report:',SCOPE_ROOT/'scope.json',flush=True)

(WORK_DIR/'full_queue_embeddings.py').write_text('"""Upload all missing text once; issue small, bounded GPU execution plans.\n\nThe immutable input library and the output-capacity plan have different hashes.\nOnly the plan changes after an import. Native encoder and vector-cache formats\nare reused unchanged, and the old bounded export/import history is retained.\n"""\nfrom pathlib import Path\nfrom types import SimpleNamespace\nimport argparse\nimport os\nimport re\nimport subprocess\nimport sys\nimport time\n\nimport pyarrow as pa\nimport pyarrow.parquet as pq\n\nfrom vietmedbridge.artifacts import atomic_json, digest_json, publish_file, read_json, sha256_file, verify_file, utc_now\nfrom vietmedbridge.content_embeddings import ContentBlocks, checked, key, seal\nfrom vietmedbridge.portable_embeddings import kaggle_assignment, relative_path, validate_encoder\nfrom vietmedbridge.scale_vectors import bound\ntry:\n    from . import new_data_kaggle as base\n    from .new_data_embeddings import LazyEncoder, FAMILIES\n    from .new_data_cache_lookup import ResumableContentBlocks\nexcept ImportError:\n    import new_data_kaggle as base\n    from new_data_embeddings import LazyEncoder, FAMILIES\n    from new_data_cache_lookup import ResumableContentBlocks\n\nSOURCE_FILES = (*base.SOURCE_FILES, \'full_queue_embeddings.py\')\nLIBRARY_SCHEMA = \'full-data-text-library-v1\'\nPLAN_SCHEMA = \'full-data-execution-plan-v1\'\nELASTIC_PLAN_SCHEMA = \'full-data-execution-plan-v2\'\nMAX_OUTPUT_BYTES = 16_000_000_000\nELASTIC_OUTPUT_BYTES = 18_000_000_000\nLEGACY_EXECUTOR_FILES = (\'full_queue_embeddings.py\', \'new_data_kaggle.py\')\nEXECUTOR_FILES = (*LEGACY_EXECUTOR_FILES, \'qwen_token_batching.py\')\nQWEN_EXECUTION = \'qwen-token-batches-v2\'\nQWEN_EXECUTIONS = (\'qwen-token-batches-v1\',QWEN_EXECUTION)\n\n\ndef execution_worker(plan, task):\n    return plan[\'task_workers\'][task[\'id\']] if plan[\'schema\'] == ELASTIC_PLAN_SCHEMA else task[\'worker\']\n\n\ndef execution_team(plan, library):\n    return plan[\'team\'] if plan[\'schema\'] == ELASTIC_PLAN_SCHEMA else library[\'team\']\n\n\ndef reuse_worker(candidates,loads,keys,masks,cost):\n    reuse={w:0 for w in candidates}\n    for k in keys:\n        mask=masks.get(k,0)\n        for w in candidates:\n            reuse[w]+=bool(mask & (1<<w))\n    worker=min(candidates,key=lambda w:(loads[w]+(len(keys)-reuse[w])*cost,-reuse[w],w))\n    return worker,reuse[worker]\n\n\ndef verify_executor(plan):\n    if plan[\'schema\'] == ELASTIC_PLAN_SCHEMA:\n        for name, checksum in plan[\'executor_sources\'].items():\n            verify_file(Path(__file__).resolve().parent/name, checksum)\n\n\ndef _without_sha(value):\n    return {k: v for k, v in value.items() if k != \'manifest_sha256\'}\n\n\ndef _value(value):\n    if isinstance(value, dict):\n        return checked(value)\n    path = Path(value)\n    return checked(read_json(path/\'job.json\' if path.is_dir() else path))\n\n\ndef _is_sha(value):\n    return isinstance(value, str) and bool(re.fullmatch(\'[0-9a-f]{64}\', value))\n\n\ndef _validate_library_header(library):\n    library = checked(library)\n    if (library.get(\'schema\') != LIBRARY_SCHEMA or library.get(\'repo_url\') != \'https://github.com/Platypus27-coder/VietMedBridge.git\'\n            or not re.fullmatch(\'[0-9a-f]{40}\', library.get(\'code_commit\', \'\'))\n            or set(library.get(\'worker_sources\', {})) != set(SOURCE_FILES)\n            or set(library.get(\'models\', {})) != set(FAMILIES)):\n        raise ValueError(\'Invalid full-data input library producer.\')\n    assignment = kaggle_assignment(accounts=library[\'team\'][\'accounts\'], account_id=0, gpu_count=library[\'team\'][\'gpu_count\'])\n    assets, seen, occurrences = library.get(\'assets\', {}), set(), {f: 0 for f in FAMILIES}\n    for task in library.get(\'tasks\', []):\n        if (not _is_sha(task.get(\'id\')) or task[\'id\'] in seen or task.get(\'selection_id\') != task[\'id\']\n                or task.get(\'family\') not in FAMILIES or type(task.get(\'worker\')) is not int\n                or not 0 <= task[\'worker\'] < assignment[\'workers\'] or type(task.get(\'rows\')) is not int\n                or not 0 < task[\'rows\'] <= base.TASK_ROWS or type(task.get(\'offset\')) is not int or task[\'offset\'] < 0\n                or type(task.get(\'ordinal\')) is not int or task[\'ordinal\'] < 0\n                or not _is_sha(task.get(\'keys_digest\')) or task.get(\'asset\') not in assets):\n            raise ValueError(\'Invalid full-data task binding.\')\n        asset = assets[task[\'asset\']]\n        if (task[\'asset\'] != f"missing-{task[\'family\']}-{task[\'ordinal\']:06d}.parquet"\n                or task.get(\'sha256\') != asset.get(\'sha256\') or not _is_sha(asset.get(\'sha256\'))\n                or not _is_sha(asset.get(\'keys_sha256\'))\n                or type(asset.get(\'rows\')) is not int or task[\'offset\']+task[\'rows\'] > asset[\'rows\']):\n            raise ValueError(\'Invalid shared text asset range.\')\n        relative_path(asset[\'source_path\'])\n        relative_path(asset[\'keys_path\'])\n        seen.add(task[\'id\'])\n        occurrences[task[\'family\']] += 1\n    if (not seen or library.get(\'scheduled_tasks\') != len(seen) or library.get(\'unscheduled_tasks\') != 0\n            or library.get(\'input_export_complete\') is not True or library.get(\'all_remaining_tasks_scheduled\') is not True\n            or library.get(\'remaining_tasks_before_round\') != occurrences\n            or set(assets) != {t[\'asset\'] for t in library[\'tasks\']}):\n        raise ValueError(\'Full input export does not conserve the pending task queue.\')\n    return library\n\n\ndef validate_library(directory, checkout=None):\n    directory = Path(directory)\n    library = _validate_library_header(read_json(directory/\'job.json\'))\n    for name, checksum in library[\'worker_sources\'].items():\n        verify_file(directory/name, checksum)\n    # Inventory only. Every text payload is hashed when consumed by its worker.\n    missing = [name for name in library[\'assets\'] if not (directory/name).is_file()]\n    if missing:\n        raise FileNotFoundError(f\'Full input library is missing {len(missing)} shared text files: {missing[:3]}\')\n    if checkout is not None:\n        if set(library[\'model_code\']) != {\'retrieval_models.py\', \'qwen_models.py\'}:\n            raise ValueError(\'Unexpected model producer code.\')\n        for name, checksum in library[\'model_code\'].items():\n            verify_file(Path(checkout)/\'src/vietmedbridge\'/name, checksum)\n        for family in FAMILIES:\n            validate_encoder(library, family, library[\'models\'][family][\'encoder\'], library[\'models\'][family][\'dimension\'])\n    return library\n\n\ndef validate_plan(plan, library):\n    plan, library = _value(plan), _validate_library_header(library)\n    elastic = plan.get(\'schema\') == ELASTIC_PLAN_SCHEMA\n    if (plan.get(\'schema\') not in (PLAN_SCHEMA, ELASTIC_PLAN_SCHEMA) or plan.get(\'library_job\') != library[\'manifest_sha256\']\n            or plan.get(\'selection_binding\') != library[\'selection_binding\'] or (not elastic and plan.get(\'team\') != library[\'team\'])\n            or plan.get(\'coverage_scope\') != library.get(\'coverage_scope\', {})):\n        raise ValueError(\'Execution plan belongs to another input library, scope or team.\')\n    team = execution_team(plan, library)\n    assignment = kaggle_assignment(accounts=team[\'accounts\'], account_id=0, gpu_count=team[\'gpu_count\'])\n    expected_sources = EXECUTOR_FILES if plan.get(\'qwen_execution\') in QWEN_EXECUTIONS else LEGACY_EXECUTOR_FILES\n    if plan.get(\'qwen_execution\') not in (None, *QWEN_EXECUTIONS) or (not elastic and plan.get(\'qwen_execution\')):\n        raise ValueError(\'Unknown Qwen execution policy; the native model producer must remain unchanged.\')\n    if elastic and (set(plan.get(\'executor_sources\', {})) != set(expected_sources)\n                    or any(not _is_sha(v) for v in plan[\'executor_sources\'].values())\n                    or plan.get(\'library_team\') != library[\'team\']):\n        raise ValueError(\'Execution scheduler or original library team changed.\')\n    budget, tasks, entries = plan.get(\'output_bytes_per_account\'), plan.get(\'tasks\'), {t[\'id\']: t for t in library[\'tasks\']}\n    if (type(budget) is not int or not 0 < budget <= (ELASTIC_OUTPUT_BYTES if elastic else MAX_OUTPUT_BYTES) or not isinstance(tasks, list)\n            or not tasks or len(set(tasks)) != len(tasks) or not set(tasks) <= set(entries)\n            or plan.get(\'scheduled_tasks\') != len(tasks)):\n        raise ValueError(\'Invalid execution task whitelist/output budget.\')\n    if elastic and (set(plan.get(\'task_workers\', {})) != set(tasks)\n                    or any(type(w) is not int or not 0 <= w < assignment[\'workers\'] for w in plan[\'task_workers\'].values())):\n        raise ValueError(\'Execution task/GPU assignments are incomplete or out of range.\')\n    cached = plan.get(\'cached_keys\', {})\n    if (set(cached) != set(FAMILIES) or any(not isinstance(values, list) or len(set(values)) != len(values)\n            or any(not _is_sha(k) for k in values) for values in cached.values())):\n        raise ValueError(\'Invalid cached-key execution contract.\')\n    estimates = plan.get(\'task_output_bytes\', {})\n    if set(estimates) != set(tasks):\n        raise ValueError(\'Missing bounded task output accounting.\')\n    used = [0]*team[\'accounts\']\n    for task_id in tasks:\n        task = entries[task_id]\n        estimate = estimates[task_id]\n        minimum = library[\'models\'][task[\'family\']][\'dimension\']*4+1024\n        if type(estimate) is not int or not minimum <= estimate <= task[\'rows\']*minimum or estimate % minimum:\n            raise ValueError(\'Execution task output estimate is invalid.\')\n        used[execution_worker(plan, task)//team[\'gpu_count\']] += estimate\n    if plan.get(\'estimated_bytes_per_account\') != used or any(n > budget for n in used):\n        raise ValueError(\'Execution plan exceeds the per-account output budget.\')\n    if (type(plan.get(\'remaining_tasks_after_plan\')) is not int or plan[\'remaining_tasks_after_plan\'] < 0\n            or type(plan.get(\'remaining_tasks_before_plan\')) is not int\n            or plan[\'remaining_tasks_before_plan\'] != len(tasks)+plan[\'remaining_tasks_after_plan\']):\n        raise ValueError(\'Execution plan task accounting changed.\')\n    reserved=plan.get(\'reserved_task_ids\',[])\n    if (not isinstance(reserved,list) or len(set(reserved))!=len(reserved)\n            or not set(reserved)<=set(entries) or set(reserved)&set(tasks)\n            or len(reserved)>plan[\'remaining_tasks_after_plan\']):\n        raise ValueError(\'Reserved pending-account tasks overlap the new execution plan.\')\n    return plan\n\n\nclass FullQueueKaggle(base.NewDataKaggle):\n    def account_receipts(self,library):\n        result=[]\n        for path in (self.root/\'imports\').glob(\'*.json\'):\n            r=checked(read_json(path))\n            if r.get(\'job\')==library[\'manifest_sha256\'] and \'account_id\' in r:\n                if (r.get(\'binding\')!=self.binding or r.get(\'state\')!=\'NEW_DATA_KAGGLE_IMPORTED\'\n                        or not _is_sha(r.get(\'result\'))):\n                    raise ValueError(\'Account import receipt belongs to another binding or is incomplete.\')\n                result.append(r)\n        return result\n\n    def reserve_pending_accounts(self,library,source,team,receipts):\n        if type(team.get(\'accounts\')) is not int or team[\'accounts\'] < 1:\n            raise ValueError(\'Continuation requires at least one execution account.\')\n        matching=[r for r in receipts if r.get(\'execution_plan\')==source[\'manifest_sha256\']]\n        source_accounts=set(range(source[\'team\'][\'accounts\']))\n        released=sorted({r[\'account_id\'] for r in matching})\n        if not set(released)<=source_accounts:\n            raise ValueError(\'Source plan has an import receipt for an unknown account.\')\n        held=sorted(source_accounts-set(released))\n        source_ids=set(source[\'tasks\'])\n        tasks=[t for t in library[\'tasks\'] if t[\'id\'] in source_ids]\n        ids=sorted(t[\'id\'] for t in tasks if execution_worker(source,t)//source[\'team\'][\'gpu_count\'] in held)\n        record={\'schema\':\'pending-account-reservation-v1\',\'binding\':self.binding,\n            \'library_job\':library[\'manifest_sha256\'],\'source_plan\':source[\'manifest_sha256\'],\n            \'released_accounts\':released,\'held_accounts\':held,\'held_task_ids\':ids,\n            \'release_receipts\':{str(a):sorted(r[\'result\'] for r in matching if r[\'account_id\']==a) for a in released}}\n        path=self.root/\'reservations\'/(source[\'manifest_sha256\']+\'.json\')\n        if path.exists():\n            old=checked(read_json(path))\n            if (old.get(\'schema\')!=\'pending-account-reservation-v1\'\n                    or old.get(\'binding\')!=self.binding\n                    or old.get(\'library_job\')!=library[\'manifest_sha256\']\n                    or old.get(\'source_plan\')!=source[\'manifest_sha256\']):\n                raise ValueError(\'Saved pending-account reservation changed.\')\n            # A reservation is immutable once created. Late results from held\n            # accounts release their tasks through the receipt inventory below.\n            self.reserved_tasks(library,receipts)\n            return\n        else:\n            atomic_json(path,seal(record))\n        print(\'CONTINUE_COMPLETED_ACCOUNTS:\',released,\'| keep pending accounts\',held,\n              \'| original assigned tasks\',len(ids),\'| old Outputs remain importable\',flush=True)\n\n    def reserved_tasks(self,library,receipts):\n        held=set(); summaries=[]\n        for path in sorted((self.root/\'reservations\').glob(\'*.json\')):\n            r=checked(read_json(path))\n            if (r.get(\'schema\')!=\'pending-account-reservation-v1\' or r.get(\'binding\')!=self.binding\n                    or r.get(\'library_job\')!=library[\'manifest_sha256\'] or not _is_sha(r.get(\'source_plan\'))):\n                raise ValueError(\'Invalid pending-account reservation binding.\')\n            source=validate_plan(self.saved_plan(r[\'source_plan\']),library)\n            released,accounts=r[\'released_accounts\'],r[\'held_accounts\']\n            source_accounts=set(range(source[\'team\'][\'accounts\']))\n            if (not isinstance(released,list) or not isinstance(accounts,list)\n                    or released!=sorted(set(released)) or accounts!=sorted(set(accounts))\n                    or set(released)&set(accounts) or set(released)|set(accounts)!=source_accounts):\n                raise ValueError(\'Pending-account reservation owner set changed.\')\n            proof=r.get(\'release_receipts\',{})\n            for a in released:\n                valid={x[\'result\'] for x in receipts if x.get(\'execution_plan\')==r[\'source_plan\'] and x[\'account_id\']==a}\n                if not proof.get(str(a)) or not set(proof[str(a)])<=valid:\n                    raise ValueError(\'Continuing account is missing its committed import proof.\')\n            entries={t[\'id\']:t for t in library[\'tasks\']}\n            by_account={a:{tid for tid in source[\'tasks\'] if\n                execution_worker(source,entries[tid])//source[\'team\'][\'gpu_count\']==a} for a in accounts}\n            expected=set().union(*by_account.values())\n            if len(set(r[\'held_task_ids\']))!=len(r[\'held_task_ids\']) or set(r[\'held_task_ids\'])!=expected:\n                raise ValueError(\'Reserved original-account task whitelist changed.\')\n            finished={x[\'account_id\'] for x in receipts if x.get(\'execution_plan\')==r[\'source_plan\']}\n            pending=[a for a in accounts if a not in finished]\n            for a in pending:held.update(by_account[a])\n            summaries.append({\'source_plan\':r[\'source_plan\'],\'pending_accounts\':pending,\n                              \'reserved_tasks\':sum(len(by_account[a]) for a in pending)})\n        return held,summaries\n\n    def export_inputs(self, checkout, *, config, runtime, code_commit, receipt_api=None,\n                      blocks_factory=ResumableContentBlocks):\n        """All pending text assets, independently of any GPU output budget."""\n        pointer = self.root/\'input_library.json\'\n        if pointer.exists():\n            library = _validate_library_header(read_json(pointer))\n            if library[\'selection_binding\'] != self.binding or library[\'models\'] != self.view.identities:\n                raise ValueError(\'Saved input library belongs to another selection/model contract.\')\n            print(\'FULL_INPUT_LIBRARY_REUSED: sealed metadata; no text selection or asset replay\', flush=True)\n            return self.work/\'full-library\'/library[\'manifest_sha256\'][:16], library\n        if not re.fullmatch(\'[0-9a-f]{40}\', code_commit):\n            raise ValueError(\'Pin the full native producer commit.\')\n        selected = self.select(receipt_api=receipt_api, blocks_factory=blocks_factory)\n        done, tasks, assets = self.imported(), [], {}\n        temporary = self.work/\'full-library-assets\'\n        temporary.mkdir(parents=True, exist_ok=True)\n        for family in FAMILIES:\n            for ordinal, entry in sorted(selected[family].items()):\n                pending = [t for t in entry[\'tasks\'] if t[\'selection_id\'] not in done]\n                if not pending:\n                    continue\n                paths = {t[\'path\'] for t in pending}\n                checksums = {t[\'sha256\'] for t in pending}\n                if len(paths) != 1 or len(checksums) != 1:\n                    raise ValueError(\'Shared missing-text input binding changed.\')\n                source_path, checksum = next(iter(paths)), next(iter(checksums))\n                asset = f\'missing-{family}-{ordinal:06d}.parquet\'\n                source = bound(self.root, source_path)\n                local = temporary/asset\n                if not local.is_file():\n                    if publish_file(source, local) != checksum:\n                        raise ValueError(\'Saved missing text changed during input export.\')\n                else:\n                    verify_file(local, checksum)\n                # A small key-only index on Drive lets future plan creation\n                # avoid reopening text payloads or original frozen inputs.\n                keys_path = f\'library_keys/{family}-{ordinal:06d}.parquet\'\n                marker = self.root/(keys_path+\'.done.json\')\n                if marker.exists():\n                    keys_entry = checked(read_json(marker))\n                    if keys_entry.get(\'source_sha256\') != checksum:\n                        raise ValueError(\'Saved library key index source changed.\')\n                else:\n                    table = pq.read_table(local, columns=[\'id\'])\n                    keys_local = temporary/\'keys.parquet\'\n                    pq.write_table(table, keys_local, compression=\'zstd\')\n                    keys_entry = seal({\'path\': keys_path, \'sha256\': publish_file(keys_local, self.root/keys_path),\n                                       \'rows\': len(table), \'source_sha256\': checksum})\n                    atomic_json(marker, keys_entry)\n                assets[asset] = {\'sha256\': checksum, \'rows\': keys_entry[\'rows\'], \'source_path\': source_path,\n                                 \'keys_path\': keys_path, \'keys_sha256\': keys_entry[\'sha256\']}\n                tasks.extend({**t, \'id\': t[\'selection_id\'], \'asset\': asset} for t in pending)\n                if len(assets) % 64 == 0:\n                    print(f\'FULL_INPUT_EXPORT: {len(assets):,} shared files | {len(tasks):,} tasks | no output-cap truncation\', flush=True)\n        if not tasks:\n            return None, {\'state\': \'NO_PENDING_KAGGLE_TASKS\', \'coverage\': self.coverage()}\n        model_code = {n: sha256_file(Path(checkout)/\'src/vietmedbridge\'/n) for n in (\'retrieval_models.py\', \'qwen_models.py\')}\n        directory = Path(__file__).resolve().parent\n        workers = {n: sha256_file(directory/n) for n in SOURCE_FILES}\n        sources = {s[\'build\'][\'build_run\']: s[\'build\'] for s in self.view.sources}\n        library = seal({\'schema\': LIBRARY_SCHEMA, \'selection_binding\': self.binding, \'batch\': self.view.batch[\'manifest_sha256\'],\n            \'source_batch_dir\': self.view.root.relative_to(self.view.project).as_posix(),\n            \'coverage_scope\': {k: self.view.batch[k] for k in (\'official_ids\',\'source_builds\',\'source_documents_sum\',\n                \'input_rows_before_cross_source_dedup\',\'scope_manifest_sha256\') if k in self.view.batch},\n            \'frozen_sources\': [{\'build_run\': b[\'build_run\'], \'candidate_manifest_sha256\': b[\'candidate_manifest_sha256\'],\n                \'index_inputs_manifest_sha256\': b[\'index_inputs_manifest_sha256\'], \'unique_dense_inputs\': b[\'unique_dense_inputs\'],\n                \'inherited_complete_models\': sorted(b.get(\'completed_views\', {}))} for b in sources.values()],\n            \'input_export_complete\': True, \'scheduled_tasks\': len(tasks), \'unscheduled_tasks\': 0,\n            \'all_remaining_tasks_scheduled\': True,\n            \'remaining_tasks_before_round\': {f: sum(t[\'family\'] == f for t in tasks) for f in FAMILIES},\n            \'team\': self.team, \'code_commit\': code_commit, \'repo_url\': \'https://github.com/Platypus27-coder/VietMedBridge.git\',\n            \'models\': self.view.identities, \'config\': config, \'runtime\': runtime, \'model_code\': model_code,\n            \'worker_sources\': workers, \'tasks\': tasks, \'assets\': assets,\n            \'scope\': self.view.batch.get(\'scope\', \'All frozen missing corpus text; no submission/query inference.\')})\n        _validate_library_header(library)\n        target = self.work/\'full-library\'/library[\'manifest_sha256\'][:16]\n        target.mkdir(parents=True, exist_ok=True)\n        # Move within Colab scratch avoids retaining a second full payload copy.\n        for asset, entry in assets.items():\n            destination = target/asset\n            if destination.exists():\n                verify_file(destination, entry[\'sha256\'])\n            else:\n                (temporary/asset).replace(destination)\n        for name, checksum in workers.items():\n            base.copied(directory/name, target/name, checksum)\n        atomic_json(target/\'job.json\', library)\n        atomic_json(self.root/\'exports\'/(\'job-\'+library[\'manifest_sha256\']+\'.json\'), library)\n        atomic_json(pointer, library)\n        print(f\'FULL_INPUT_LIBRARY_READY: {len(tasks):,} tasks; {len(assets):,} shared files; 0 input tasks held by output budget\', flush=True)\n        return target, library\n\n    def create_plan(self, library, *, output_bytes=MAX_OUTPUT_BYTES, receipt_api=None,\n                    blocks_factory=ResumableContentBlocks, supersede_unstarted_legacy=False,\n                    execution_accounts=None, execution_gpu_count=None, replace_unstarted_plan=None,\n                    continue_completed_from=None):\n        library = _validate_library_header(_value(library))\n        if library[\'selection_binding\'] != self.binding or library[\'models\'] != self.view.identities:\n            raise ValueError(\'Input library differs from this frozen selection.\')\n        elastic = execution_accounts is not None\n        if elastic:\n            team = {\'accounts\': execution_accounts,\n                    \'gpu_count\': self.team[\'gpu_count\'] if execution_gpu_count is None else execution_gpu_count}\n        else:\n            if execution_gpu_count is not None:\n                raise ValueError(\'Changing the execution GPU count requires an elastic execution plan.\')\n            team = self.team\n        assignment = kaggle_assignment(accounts=team[\'accounts\'], account_id=0, gpu_count=team[\'gpu_count\'])\n        if type(output_bytes) is not int or not 0 < output_bytes <= (ELASTIC_OUTPUT_BYTES if elastic else MAX_OUTPUT_BYTES):\n            raise ValueError(\'Output budget exceeds the scheduler allowance; leave room below Kaggle saved Outputs capacity.\')\n        active = self.root/\'active_execution_plan.json\'\n        receipts=self.account_receipts(library)\n        reserved,reservations=self.reserved_tasks(library,receipts)\n        if continue_completed_from is not None and not _is_sha(continue_completed_from):\n            raise ValueError(\'Continuation requires the exact source execution-plan SHA.\')\n        if active.exists():\n            record = checked(read_json(active))\n            if record.get(\'binding\') != self.binding or record.get(\'library_job\') != library[\'manifest_sha256\']:\n                raise ValueError(\'Active execution pointer belongs to another library/selection.\')\n            plan = validate_plan(self.saved_plan(record[\'plan\']), library)\n            if any(not (self.root/\'reservations\'/(r[\'source_plan\']+\'.json\')).is_file()\n                   for r in plan.get(\'reservations\',[])):\n                raise ValueError(\'Saved pending-account reservation is missing; no tasks may be reassigned.\')\n            continuing=False\n            if continue_completed_from is not None:\n                if plan[\'manifest_sha256\']==continue_completed_from:\n                    if not elastic:raise ValueError(\'Continuing accounts require an explicit execution team.\')\n                    self.reserve_pending_accounts(library,plan,team,receipts)\n                    reserved,reservations=self.reserved_tasks(library,receipts)\n                    continuing=True\n                elif not (self.root/\'reservations\'/(continue_completed_from+\'.json\')).is_file():\n                    raise ValueError(\'Continuation source differs from the active plan and has no saved reservation.\')\n            if (replace_unstarted_plan != plan[\'manifest_sha256\'] and _is_sha(replace_unstarted_plan)\n                    and (self.root/\'history\'/(\'unstarted-\'+replace_unstarted_plan+\'.json\')).is_file()):\n                replace_unstarted_plan = None\n            imported_accounts = {r[\'account_id\'] for r in receipts if r.get(\'execution_plan\')==plan[\'manifest_sha256\']}\n            if imported_accounts != set(range(plan[\'team\'][\'accounts\'])) and not continuing:\n                if replace_unstarted_plan == plan[\'manifest_sha256\'] and not imported_accounts and elastic:\n                    atomic_json(self.root/\'history\'/(\'unstarted-\'+plan[\'manifest_sha256\']+\'.json\'), seal({\n                        \'state\': \'SUPERSEDED_UNSTARTED\', \'previous\': record, \'unstarted_confirmed\': True,\n                        \'replacement_team\': team, \'updated_at\': utc_now()}))\n                    print(f\'REPLACE_CONFIRMED_UNSTARTED_PLAN: {plan["manifest_sha256"][:12]} | old imports retained\', flush=True)\n                else:\n                    if replace_unstarted_plan is not None:\n                        raise ValueError(\'Only the exact confirmed unstarted plan with no imports may be replaced.\')\n                    if record.get(\'state\') == \'UPLOAD_PENDING\' and not imported_accounts:\n                        if plan[\'team\'] != team:\n                            raise ValueError(\'Saved pending plan has another execution team; retain it or explicitly replace it.\')\n                        return self.stage_plan(plan), plan\n                    return None, {\'state\': \'WAITING_FOR_KAGGLE_OUTPUTS\', \'execution_plan\': plan[\'manifest_sha256\'],\n                        \'library_job\': library[\'manifest_sha256\'], \'dataset_handle\': record.get(\'dataset_handle\'),\n                        \'imported_accounts\': sorted(imported_accounts), \'required_accounts\': list(range(plan[\'team\'][\'accounts\']))}\n        legacy = self.root/\'active_round.json\'\n        if legacy.exists():\n            saved = checked(read_json(legacy))\n            if saved.get(\'binding\') != self.binding:\n                raise ValueError(\'Legacy round pointer belongs to another selection.\')\n            history = self.root/\'history\'/(\'legacy-\'+saved[\'job\']+\'.json\')\n            if not history.exists():\n                imported_accounts = {r[\'account_id\'] for path in (self.root/\'imports\').glob(\'*.json\')\n                    if (r := checked(read_json(path))).get(\'job\') == saved[\'job\'] and \'account_id\' in r}\n                if imported_accounts != set(range(self.team[\'accounts\'])) and not supersede_unstarted_legacy:\n                    return None, {\'state\': \'LEGACY_ROUND_REQUIRES_EXPLICIT_SUPERSEDE\', \'dataset_handle\': saved[\'dataset_handle\'],\n                                  \'instruction\': \'Keep/import any GPU Outputs; explicitly confirm legacy GPU did not run before promoting the full library.\'}\n                atomic_json(history, seal({\'legacy\': saved, \'superseded_by_library\': library[\'manifest_sha256\'],\n                    \'unstarted_confirmed\': supersede_unstarted_legacy, \'updated_at\': utc_now()}))\n        done = self.imported()\n        reserved-=done\n        pending_all=[t for t in library[\'tasks\'] if t[\'id\'] not in done]\n        pending=[t for t in pending_all if t[\'id\'] not in reserved]\n        if reserved:\n            print(\'PENDING_ACCOUNT_TASKS_RESERVED:\',len(reserved),\'| dispatchable\',len(pending),flush=True)\n        used, scheduled, estimates = [0]*team[\'accounts\'], [], {}\n        task_workers, loads, affinity = {}, [0]*assignment[\'workers\'], {}\n        planning_stats={}\n        assets_read = 0\n        cached, cache_only, affected = {f: set() for f in FAMILIES}, [], {f: set() for f in FAMILIES}\n        for family in FAMILIES:\n            family_tasks = [t for t in pending if t[\'family\'] == family]\n            if not family_tasks:\n                continue\n            expected = self.view.identities[family]\n            blocks = blocks_factory(self.view.data, SimpleNamespace(identity=expected[\'encoder\'], dimension=expected[\'dimension\']),\n                                    self.work/family, receipt_api=receipt_api, progress=print)\n            planned_masks={}\n            family_unique,family_worker_rows,family_reuse=0,0,0\n            previous, keys = None, None\n            try:\n                for task in family_tasks:\n                    account = task[\'worker\']//self.team[\'gpu_count\']\n                    unit_bytes = expected[\'dimension\']*4+1024\n                    # No need to read later key files once an account cannot\n                    # fit even one new vector. Those tasks remain in the library.\n                    if (all(n+unit_bytes > output_bytes for n in used) if elastic else used[account]+unit_bytes > output_bytes):\n                        continue\n                    asset = library[\'assets\'][task[\'asset\']]\n                    if previous != task[\'asset\']:\n                        path = bound(self.root, asset[\'keys_path\'])\n                        verify_file(path, asset[\'keys_sha256\'])\n                        keys = pq.read_table(path, columns=[\'id\']).column(\'id\').to_pylist()\n                        if len(keys) != asset[\'rows\']:\n                            raise ValueError(\'Saved task-key index row coverage changed.\')\n                        previous = task[\'asset\']\n                        assets_read += 1\n                        if elastic and assets_read % 32 == 0:\n                            print(f\'PLAN_KEY_METADATA: {assets_read} key indexes | no corpus/text replay; imported tasks excluded\', flush=True)\n                    selected = keys[task[\'offset\']:task[\'offset\']+task[\'rows\']]\n                    if len(selected) != task[\'rows\'] or digest_json(sorted(selected)) != task[\'keys_digest\']:\n                        raise ValueError(\'Saved library task keys changed.\')\n                    known = base.lookup(blocks, selected)\n                    if len(known) == len(selected):\n                        cache_only.append(task[\'id\'])\n                        affected[family].add(task[\'ordinal\'])\n                        continue\n                    estimate = (len(selected)-len(known))*unit_bytes\n                    if estimate > output_bytes:\n                        raise ValueError(\'One missing-text task exceeds the execution budget.\')\n                    if elastic:\n                        group = (family, task[\'keys_digest\'])\n                        candidates = [w for w in range(assignment[\'workers\'])\n                                      if used[w//team[\'gpu_count\']]+estimate <= output_bytes]\n                        if group in affinity:\n                            candidates = [affinity[group]] if affinity[group] in candidates else []\n                        if not candidates:\n                            continue\n                        missing_keys=set(selected)-set(known)\n                        cost=8 if family==\'qwen\' else 1\n                        # Minimize projected work, not raw task counts. Overlap\n                        # reduces work only on the GPU holding those text keys.\n                        worker,reused_rows=reuse_worker(candidates,loads,missing_keys,planned_masks,cost)\n                        affinity[group] = worker\n                        task_workers[task[\'id\']] = worker\n                        new_worker_rows=len(missing_keys)-reused_rows\n                        loads[worker]+=new_worker_rows*cost\n                        family_worker_rows+=new_worker_rows\n                        family_reuse+=reused_rows\n                        for k in missing_keys:\n                            previous_mask=planned_masks.get(k,0)\n                            family_unique+=previous_mask==0\n                            planned_masks[k]=previous_mask | (1<<worker)\n                        account = worker//team[\'gpu_count\']\n                    elif used[account]+estimate > output_bytes:\n                        continue\n                    used[account] += estimate\n                    scheduled.append(task[\'id\'])\n                    estimates[task[\'id\']] = estimate\n                    cached[family].update(known)\n                if affected[family]:\n                    self.complete_maps(family, affected[family], blocks)\n                if elastic:\n                    planning_stats[family]={\'unique_uncached_text_keys\':family_unique,\n                        \'projected_gpu_text_rows\':family_worker_rows,\n                        \'within_gpu_reused_text_rows\':family_reuse,\n                        \'cross_gpu_duplicate_text_rows\':family_worker_rows-family_unique}\n                    print(\'PLAN_TEXT_REUSE:\',family,planning_stats[family],\'| metadata keys only\',flush=True)\n            finally:\n                blocks.close()\n        if cache_only:\n            atomic_json(self.root/\'imports\'/(\'cache-\'+digest_json(cache_only)+\'.json\'), seal({\n                \'binding\': self.binding, \'selection_ids\': cache_only, \'state\': \'REUSED_NEWLY_AVAILABLE_CACHE\'}))\n        if not scheduled:\n            if reserved:\n                return None,{\'state\':\'WAITING_FOR_RESERVED_KAGGLE_OUTPUTS\',\'reserved_tasks\':len(reserved),\n                             \'reservations\':reservations,\'coverage\':self.coverage()}\n            return None, {\'state\': \'NO_PENDING_KAGGLE_TASKS\', \'coverage\': self.coverage()}\n        remaining = len(pending_all)-len(cache_only)\n        extra = {}\n        if elastic:\n            directory = Path(__file__).resolve().parent\n            sources = {n: sha256_file(directory/n) for n in EXECUTOR_FILES}\n            for name, checksum in sources.items():\n                base.copied(directory/name, self.root/\'executors\'/checksum/name, checksum)\n            extra = {\'library_team\': library[\'team\'], \'task_workers\': task_workers, \'executor_sources\': sources,\n                     \'replaces_unstarted_plan\': replace_unstarted_plan, \'qwen_execution\': QWEN_EXECUTION,\n                     \'planning_text_reuse\':planning_stats,\'reserved_task_ids\':sorted(reserved),\n                     \'reservations\':reservations,\'continued_from_plan\':continue_completed_from}\n        plan = seal({\'schema\': ELASTIC_PLAN_SCHEMA if elastic else PLAN_SCHEMA, \'library_job\': library[\'manifest_sha256\'],\n            \'selection_binding\': self.binding, \'team\': team, \'coverage_scope\': library.get(\'coverage_scope\', {}),\n            \'tasks\': scheduled, \'cached_keys\': {f: sorted(v) for f, v in cached.items()},\n            \'task_output_bytes\': estimates, \'output_bytes_per_account\': output_bytes, \'estimated_bytes_per_account\': used,\n            \'scheduled_tasks\': len(scheduled), \'remaining_tasks_before_plan\': remaining,\n            \'remaining_tasks_after_plan\': remaining-len(scheduled), \'created_at\': utc_now(), **extra})\n        validate_plan(plan, library)\n        atomic_json(self.root/\'plans\'/(plan[\'manifest_sha256\']+\'.json\'), plan)\n        # Commit before upload: cold retry resumes exactly this plan rather than\n        # rebuilding a fresh timestamp/whitelist and discarding its checkpoint.\n        atomic_json(active, seal({\'binding\': self.binding, \'plan\': plan[\'manifest_sha256\'],\n                                 \'library_job\': library[\'manifest_sha256\'], \'state\': \'UPLOAD_PENDING\'}))\n        return self.stage_plan(plan), plan\n\n    def stage_plan(self, plan):\n        target = self.work/\'execution-plans\'/plan[\'manifest_sha256\'][:16]\n        target.mkdir(parents=True, exist_ok=True)\n        for name, checksum in plan.get(\'executor_sources\', {}).items():\n            base.copied(self.root/\'executors\'/checksum/name, target/name, checksum)\n        atomic_json(target/\'job.json\', plan)\n        return target\n\n    def saved_plan(self, plan_sha):\n        if not _is_sha(plan_sha):\n            raise ValueError(\'Invalid execution plan hash.\')\n        plan = checked(read_json(self.root/\'plans\'/(plan_sha+\'.json\')))\n        if plan[\'manifest_sha256\'] != plan_sha:\n            raise ValueError(\'Saved execution plan filename/hash binding changed.\')\n        return plan\n\n    def install(self, output, *, receipt_api=None, blocks_factory=ResumableContentBlocks):\n        output = Path(output)\n        library = checked(read_json(output/\'job.json\'))\n        if library.get(\'schema\') != LIBRARY_SCHEMA:\n            return super().install(output, receipt_api=receipt_api, blocks_factory=blocks_factory)\n        saved = checked(read_json(self.root/\'exports\'/(\'job-\'+library[\'manifest_sha256\']+\'.json\')))\n        result = checked(read_json(output/\'new-data-result.json\'))\n        plan = validate_plan(self.saved_plan(result.get(\'execution_plan\')), library)\n        if result.get(\'plan\') != plan or library != saved or result.get(\'job\') != library[\'manifest_sha256\']:\n            raise ValueError(\'GPU Outputs differ from the saved full-library execution plan.\')\n        if result.get(\'state\') not in (\'ROUND_COMPLETE\', \'CHECKPOINTED_PARTIAL\'):\n            raise ValueError(\'Invalid full-library execution result state.\')\n        team = execution_team(plan, library)\n        assignment = kaggle_assignment(accounts=team[\'accounts\'], account_id=result[\'account_id\'], gpu_count=team[\'gpu_count\'])\n        tasks = {t[\'id\']: t for t in library[\'tasks\']}\n        receipts = [r[\'task\'] for r in result[\'tasks\']]\n        allowed = {task for task in plan[\'tasks\'] if execution_worker(plan, tasks[task]) in assignment[\'worker_ids\']}\n        if len(set(receipts)) != len(receipts) or not set(receipts) <= allowed:\n            raise ValueError(\'Output contains duplicate or non-whitelisted execution tasks.\')\n        if result.get(\'state\') == \'ROUND_COMPLETE\' and set(receipts) != allowed:\n            raise ValueError(\'Round completion does not cover this account execution plan.\')\n        # Native importer verifies payload SHA, exact task keys, encoder identity\n        # and skipped cached keys before it can publish a map/coverage receipt.\n        report = super().install(output, receipt_api=receipt_api, blocks_factory=blocks_factory,\n                                 cached_keys=plan[\'cached_keys\'])\n        receipt = self.root/\'imports\'/(result[\'manifest_sha256\']+\'.json\')\n        if report.get(\'execution_plan\') != plan[\'manifest_sha256\']:\n            prior = checked(read_json(receipt))\n            prior = seal({**_without_sha(prior), \'execution_plan\': plan[\'manifest_sha256\'], \'execution_team\': team})\n            atomic_json(receipt, prior)\n            report = prior\n        return report\n\n    def assigned_output_tasks(self, job, result):\n        if result.get(\'plan\', {}).get(\'schema\') != ELASTIC_PLAN_SCHEMA:\n            return super().assigned_output_tasks(job, result)\n        plan = validate_plan(self.saved_plan(result.get(\'execution_plan\')), job)\n        if result.get(\'plan\') != plan:\n            raise ValueError(\'Elastic output scheduler differs from the saved execution plan.\')\n        team = execution_team(plan, job)\n        assignment = kaggle_assignment(accounts=team[\'accounts\'], account_id=result[\'account_id\'], gpu_count=team[\'gpu_count\'])\n        entries = {t[\'id\']: t for t in job[\'tasks\']}\n        return {tid for tid in plan[\'tasks\'] if execution_worker(plan, entries[tid]) in assignment[\'worker_ids\']}\n\n\ndef upload_library(flow, target, library, owner):\n    # Metadata-only reuse after a successful upload. If the first upload was\n    # interrupted by a fresh Colab runtime, rebuild only its missing staging\n    # files from the saved selection; never reopen/chunk the source corpus.\n    library = _validate_library_header(library)\n    saved = checked(read_json(flow.root/\'exports\'/(\'job-\'+library[\'manifest_sha256\']+\'.json\')))\n    if saved != library or library[\'selection_binding\'] != flow.binding:\n        raise ValueError(\'Full-library upload differs from its saved selection contract.\')\n    pointer = flow.root/\'uploads\'/(owner+\'-\'+library[\'manifest_sha256\']+\'.json\')\n    planned = not pointer.exists() or read_json(pointer).get(\'state\') == \'UPLOAD_PLANNED\'\n    if planned:\n        target = Path(target)\n        target.mkdir(parents=True, exist_ok=True)\n        for name, entry in library[\'assets\'].items():\n            if not (target/name).is_file():\n                if publish_file(bound(flow.root, entry[\'source_path\']), target/name) != entry[\'sha256\']:\n                    raise ValueError(\'Saved full-library input changed during upload staging.\')\n        scripts = Path(__file__).resolve().parent\n        for name, checksum in library[\'worker_sources\'].items():\n            base.copied(scripts/name, target/name, checksum)\n        atomic_json(target/\'job.json\', library)\n    return base.upload_job(flow, target, library, owner, activation_filename=\'full_package.json\', dataset_prefix=\'/vmb-full-inputs-\')\n\n\ndef upload_plan(flow, target, plan, owner):\n    plan = checked(plan)\n    library = checked(read_json(flow.root/\'exports\'/(\'job-\'+plan[\'library_job\']+\'.json\')))\n    validate_plan(plan, library)\n    if plan != flow.saved_plan(plan[\'manifest_sha256\']) or plan[\'selection_binding\'] != flow.binding:\n        raise ValueError(\'Execution upload differs from its saved plan/selection.\')\n    if plan[\'schema\'] == ELASTIC_PLAN_SCHEMA:\n        target = flow.stage_plan(plan)\n    handle = base.upload_job(flow, target, plan, owner, activation_filename=\'uploaded_execution_plan.json\', dataset_prefix=\'/vmb-full-plan-\')\n    atomic_json(flow.root/\'active_execution_plan.json\', seal({\'binding\': flow.binding, \'plan\': plan[\'manifest_sha256\'],\n        \'library_job\': plan[\'library_job\'], \'dataset_handle\': handle, \'state\': \'VERIFIED\'}))\n    return handle\n\n\ndef run_worker(directory, plan, output, work, *, worker_id, factories, deadline, batch_sizes=None):\n    directory, output, work = Path(directory), Path(output), Path(work)\n    library = validate_library(directory)\n    plan = validate_plan(plan, library)\n    verify_executor(plan)\n    team = execution_team(plan, library)\n    workers = team[\'accounts\']*team[\'gpu_count\']\n    original_workers = library[\'team\'][\'accounts\']*library[\'team\'][\'gpu_count\']\n    if type(worker_id) is not int or not 0 <= worker_id < workers:\n        raise ValueError(\'Worker ID outside the full-data team.\')\n    output.mkdir(parents=True, exist_ok=True)\n    for name, record in ((\'job.json\', library), (\'execution-plan.json\', plan)):\n        if (output/name).exists() and read_json(output/name) != record:\n            raise ValueError(\'Working output contains another job or execution plan; preserve its Outputs.\')\n        atomic_json(output/name, record)\n    journal = output/f\'worker-{worker_id}.json\'\n    completed = []\n    if journal.exists():\n        saved = checked(read_json(journal))\n        if saved.get(\'job\') != library[\'manifest_sha256\'] or saved.get(\'execution_plan\') != plan[\'manifest_sha256\'] or saved.get(\'worker_id\') != worker_id:\n            raise ValueError(\'Worker journal belongs to another job/plan/worker.\')\n        completed = saved[\'tasks\']\n    entries = {t[\'id\']: t for t in library[\'tasks\']}\n    allowed = {tid for tid in plan[\'tasks\'] if execution_worker(plan, entries[tid]) == worker_id}\n    done = {r[\'task\'] for r in completed}\n    if len(done) != len(completed) or not done <= allowed:\n        raise ValueError(\'Worker journal contains duplicate or non-whitelisted tasks.\')\n    stop_reason = \'ASSIGNMENT_COMPLETE\'\n    cached = {f: set(values) for f, values in plan[\'cached_keys\'].items()}\n    previous, rows = None, None\n    for family in FAMILIES:\n        assigned = [entries[tid] for tid in plan[\'tasks\'] if tid in allowed-done and entries[tid][\'family\'] == family]\n        print(f\'GPU worker {worker_id}: {family}, {len(assigned)} bounded tasks; full input library already present\', flush=True)\n        if not assigned:\n            continue\n        if time.monotonic() >= deadline:\n            stop_reason = \'SESSION_TIME_LIMIT\'\n            continue\n        lazy = LazyEncoder(library[\'models\'][family], factories[family])\n        blocks = ContentBlocks(output, lazy, work/family)\n        try:\n            for task in assigned:\n                if time.monotonic() >= deadline:\n                    stop_reason = \'SESSION_TIME_LIMIT\'\n                    break\n                if previous != task[\'asset\']:\n                    path = directory/task[\'asset\']\n                    verify_file(path, task[\'sha256\'])\n                    rows = pq.read_table(path, columns=[\'id\', \'text\']).to_pylist()\n                    if len(rows) != library[\'assets\'][task[\'asset\']][\'rows\']:\n                        raise ValueError(\'Shared text asset row coverage changed.\')\n                    previous = task[\'asset\']\n                values = rows[task[\'offset\']:task[\'offset\']+task[\'rows\']]\n                keys = [r[\'id\'] for r in values]\n                if (len(values) != task[\'rows\'] or len(set(keys)) != len(keys) or digest_json(sorted(keys)) != task[\'keys_digest\']\n                        or any(k != key(r[\'text\']) or int(k[:8], 16) % original_workers != task[\'worker\'] for k, r in zip(keys, values))):\n                    raise ValueError(\'Full-library task text/key/ownership changed.\')\n                new_values = [r for r in values if r[\'id\'] not in cached[family]]\n                expected_bytes = len(new_values)*(library[\'models\'][family][\'dimension\']*4+1024)\n                if expected_bytes != plan[\'task_output_bytes\'][task[\'id\']]:\n                    raise ValueError(\'Cached-key skip differs from the bounded task estimate.\')\n                defaults = {\'bge\':32,\'qwen\':64 if plan.get(\'qwen_execution\') in QWEN_EXECUTIONS else 16}\n                blocks.refs([r[\'text\'] for r in new_values], batch_size=(batch_sizes or defaults)[family],\n                            encode_missing=True, encode_block_rows=base.TASK_ROWS)\n                completed.append({\'task\': task[\'id\'], \'keys\': keys})\n                atomic_json(journal, seal({\'job\': library[\'manifest_sha256\'], \'execution_plan\': plan[\'manifest_sha256\'],\n                    \'worker_id\': worker_id, \'tasks\': completed}))\n                print(f\'KAGGLE_CHECKPOINT worker {worker_id} | saved tasks {len(completed)} | {family} | plan {plan["manifest_sha256"][:12]}\', flush=True)\n        finally:\n            blocks.close()\n            lazy.close()\n    return seal({\'job\': library[\'manifest_sha256\'], \'execution_plan\': plan[\'manifest_sha256\'], \'worker_id\': worker_id,\n                 \'tasks\': completed, \'stop_reason\': stop_reason})\n\n\ndef finalize(directory, plan, output, account_id):\n    library = validate_library(directory)\n    plan = validate_plan(plan, library)\n    root = Path(output)/\'result\'\n    root.mkdir(parents=True, exist_ok=True)\n    team = execution_team(plan, library)\n    assignment = kaggle_assignment(accounts=team[\'accounts\'], account_id=account_id, gpu_count=team[\'gpu_count\'])\n    entries, tasks = {t[\'id\']: t for t in library[\'tasks\']}, []\n    allowed = {tid for tid in plan[\'tasks\'] if execution_worker(plan, entries[tid]) in assignment[\'worker_ids\']}\n    for worker in assignment[\'worker_ids\']:\n        path = root/f\'worker-{worker}.json\'\n        if not path.exists():\n            continue\n        saved = checked(read_json(path))\n        if saved.get(\'job\') != library[\'manifest_sha256\'] or saved.get(\'execution_plan\') != plan[\'manifest_sha256\'] or saved.get(\'worker_id\') != worker:\n            raise ValueError(\'Worker journal belongs to another job/plan/assignment.\')\n        for receipt in saved[\'tasks\']:\n            if receipt[\'task\'] not in allowed or execution_worker(plan, entries[receipt[\'task\']]) != worker:\n                raise ValueError(\'Worker returned a non-whitelisted execution task.\')\n            tasks.append(receipt)\n    if len({r[\'task\'] for r in tasks}) != len(tasks):\n        raise ValueError(\'Duplicate completed execution task.\')\n    committed = {f: {k for receipt in tasks if entries[receipt[\'task\']][\'family\'] == f for k in receipt[\'keys\']} for f in FAMILIES}\n    blocks = {}\n    for path in root.glob(\'model_cache/*/*.done.json\'):\n        block = checked(read_json(path))\n        family = next((f for f in FAMILIES if block[\'identity\'] == {**library[\'models\'][f], \'format\': \'content-float32-l2-v1\'}), None)\n        if family is not None and set(block[\'keys\']) <= committed[family]:\n            blocks[block[\'path\']] = block\n    atomic_json(root/\'job.json\', library)\n    atomic_json(root/\'execution-plan.json\', plan)\n    result = seal({\'schema\': \'new-data-kaggle-result-v1\', \'job\': library[\'manifest_sha256\'],\n        \'execution_plan\': plan[\'manifest_sha256\'], \'plan\': plan, \'account_id\': account_id,\n        \'tasks\': tasks, \'blocks\': list(blocks.values()), \'updated_at\': utc_now(),\n        \'state\': \'ROUND_COMPLETE\' if {r[\'task\'] for r in tasks} == allowed else \'CHECKPOINTED_PARTIAL\'})\n    atomic_json(root/\'new-data-result.json\', result)\n    return result\n\n\ndef launch(directory, plan, checkout, output, work, *, account_id=0, hours=10.0):\n    import torch\n    from vietmedbridge.portable_embeddings import embedding_runtime\n    library = validate_library(directory, checkout)\n    plan_path = Path(plan)/\'job.json\' if Path(plan).is_dir() else Path(plan)\n    plan = validate_plan(plan_path, library)\n    verify_executor(plan)\n    team = execution_team(plan, library)\n    if embedding_runtime() != library[\'runtime\'] or not torch.cuda.is_available() or torch.cuda.device_count() != team[\'gpu_count\']:\n        raise RuntimeError(\'CUDA/runtime mismatch: select the locked T4 x2 runtime.\')\n    assignment = kaggle_assignment(accounts=team[\'accounts\'], account_id=account_id, gpu_count=team[\'gpu_count\'])\n    if not 0 < hours <= 11.5:\n        raise ValueError(\'Use a save deadline up to 11.5 hours, below the 12-hour session limit.\')\n    deadline, processes = time.monotonic()+hours*3600-180, []\n    try:\n        for gpu, worker in enumerate(assignment[\'worker_ids\']):\n            env = os.environ.copy()\n            env[\'CUDA_VISIBLE_DEVICES\'] = str(gpu)\n            env[\'PYTHONPATH\'] = os.pathsep.join((str(Path(checkout)/\'src\'), str(Path(__file__).resolve().parent), str(Path(directory))))\n            command = [sys.executable, \'-u\', str(Path(__file__).resolve()), \'worker\', \'--job\', str(directory),\n                \'--plan\', str(plan_path), \'--output\', str(Path(output)/\'result\'), \'--work\', str(Path(work)/f\'worker-{worker}\'),\n                \'--worker-id\', str(worker), \'--deadline\', str(deadline)]\n            processes.append(subprocess.Popen(command, env=env))\n        while any(p.poll() is None for p in processes):\n            if any(p.poll() not in (None, 0) for p in processes):\n                raise RuntimeError(\'A GPU worker failed; preserve completed task Outputs.\')\n            time.sleep(2)\n        if any(p.returncode != 0 for p in processes):\n            raise RuntimeError(\'A GPU worker failed; inspect its preceding traceback.\')\n        return finalize(directory, plan, output, account_id)\n    finally:\n        for process in processes:\n            if process.poll() is None:\n                process.terminate()\n        for process in processes:\n            if process.poll() is None:\n                try:\n                    process.wait(timeout=30)\n                except subprocess.TimeoutExpired:\n                    process.kill()\n                    process.wait(timeout=10)\n        finalize(directory, plan, output, account_id)\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'mode\', choices=[\'worker\', \'launch\'])\n    for name in (\'job\', \'plan\', \'output\', \'work\'):\n        parser.add_argument(\'--\'+name, required=True)\n    parser.add_argument(\'--checkout\')\n    parser.add_argument(\'--worker-id\', type=int)\n    parser.add_argument(\'--deadline\', type=float)\n    parser.add_argument(\'--account-id\', type=int, default=0)\n    parser.add_argument(\'--hours\', type=float, default=10.0)\n    args = parser.parse_args()\n    if args.mode == \'launch\':\n        result = launch(args.job, args.plan, args.checkout, args.output, args.work, account_id=args.account_id, hours=args.hours)\n    else:\n        import torch\n        from vietmedbridge.portable_embeddings import embedding_runtime\n        from vietmedbridge.retrieval_models import TorchDenseEncoder\n        from vietmedbridge.qwen_models import TorchQwenEncoder\n        library = validate_library(args.job)\n        if not torch.cuda.is_available() or embedding_runtime() != library[\'runtime\']:\n            raise RuntimeError(\'GPU worker runtime differs from the paid vector producer.\')\n        plan = validate_plan(args.plan,library)\n        def qwen_factory():\n            owner = TorchQwenEncoder(library[\'config\'][\'second_dense\'])\n            if plan.get(\'qwen_execution\') in QWEN_EXECUTIONS:\n                from qwen_token_batching import TokenBatchOwner\n                return TokenBatchOwner(owner,Path(args.output)/\'qwen-tuning\'/f\'worker-{args.worker_id}.json\',\n                    deadline=args.deadline,attention_tuning=plan.get(\'qwen_execution\')==QWEN_EXECUTION)\n            return owner\n        result = run_worker(args.job, plan, args.output, args.work, worker_id=args.worker_id, deadline=args.deadline,\n            factories={\'bge\': lambda: TorchDenseEncoder(library[\'config\'][\'dense\']), \'qwen\': qwen_factory})\n    print(\'GPU_WORKER_RESULT:\', base.json.dumps({k: v for k, v in result.items()\n          if k not in (\'tasks\', \'blocks\', \'plan\')}, ensure_ascii=False), flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n',encoding='utf-8')

(WORK_DIR/'qwen_token_batching.py').write_text('"""Measured execution-only batching around the immutable corpus encoder.\n\nNo weights, tokenizer options, text, pooling, vector dimensions or cache identity\nare changed. The native forward is used, with a bounded on-device canary before\nselecting any faster batching order. There is no local model inference here.\n"""\nfrom contextlib import contextmanager\nimport gc\nimport time\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom vietmedbridge.artifacts import atomic_json\nfrom vietmedbridge.embeddings import validate_vectors\n\n\nPOLICY = \'qwen-token-batches-v2\'\n\n\ndef token_groups(lengths, cap, budget):\n    order = sorted(range(len(lengths)), key=lambda i: (lengths[i], i))\n    group = []\n    for i in order:\n        if group and (len(group) >= cap or (len(group)+1)*lengths[i] > budget):\n            yield group\n            group = []\n        group.append(i)\n    if group:\n        yield group\n\n\ndef vector_agreement(reference, candidate):\n    if reference.shape != candidate.shape or not np.isfinite(candidate).all():\n        return False, {\'reason\':\'shape_or_finite\'}\n    cosine = np.sum(reference*candidate, axis=1) / np.maximum(\n        np.linalg.norm(reference,axis=1)*np.linalg.norm(candidate,axis=1), 1e-12)\n    minimum = float(cosine.min())\n    maximum = float(np.max(np.abs(reference-candidate)))\n    return minimum >= 0.99999 and maximum <= 0.001, {\'min_cosine\':minimum,\'max_abs_error\':maximum}\n\n\nclass TokenBatchEncoder:\n    def __init__(self, native, report_path, *, deadline=None, clock=time.perf_counter, report=print,\n                 attention_tuning=False):\n        self.native = native\n        self.encoder = getattr(native,\'encoder\',native)\n        self.identity, self.dimension = native.identity, native.dimension\n        self.path = Path(report_path)\n        self.deadline, self.clock, self.report = deadline, clock, report\n        self.selected, self.safe_batch_size = None, 16\n        self.selected_kernel,self.selected_budget = \'native\',24_576\n        self.attention_tuning = attention_tuning\n        self.caps, self.ooms = {}, 0\n        self.attempted = False\n\n    def _attention_api(self):\n        backend=getattr(getattr(self.encoder.torch,\'backends\',None),\'cuda\',None)\n        setter=getattr(backend,\'allow_fp16_bf16_reduction_math_sdp\',None)\n        getter=getattr(backend,\'fp16_bf16_reduction_math_sdp_allowed\',None)\n        return (setter,getter) if callable(setter) and callable(getter) else (None,None)\n\n    @contextmanager\n    def _attention(self,kernel):\n        if kernel==\'native\':\n            yield\n            return\n        setter,getter=self._attention_api()\n        if kernel!=\'fp16_math\' or setter is None:\n            raise ValueError(\'Unsupported attention execution policy.\')\n        previous=getter()\n        try:\n            setter(True)\n            yield\n        finally:\n            setter(previous)\n\n    def lengths(self, texts):\n        # Exactly the native tokenizer\'s options, without batch padding.\n        ids = self.encoder.tokenizer(list(texts),padding=False,truncation=False)[\'input_ids\']\n        lengths = [len(row) for row in ids]\n        if any(n > self.encoder.spec[\'max_length\'] for n in lengths):\n            raise ValueError(\'Qwen input exceeds the frozen max_length; no text was truncated.\')\n        return lengths\n\n    def _forward(self, texts, indices, lengths, kernel):\n        bucket = max(lengths[i] for i in indices).bit_length()\n        cap = self.caps.get((kernel,bucket),len(indices))\n        if len(indices) > cap:\n            return np.concatenate([self._forward(texts,indices[i:i+cap],lengths,kernel)\n                                   for i in range(0,len(indices),cap)],axis=0)\n        try:\n            self.encoder.input_role = \'corpus\'\n            return np.asarray(self.encoder._batch([texts[i] for i in indices]),dtype=np.float32)\n        except self.encoder.torch.OutOfMemoryError:\n            if len(indices) == 1:\n                raise\n            self.ooms += 1\n        # Drop the exception/traceback before freeing CUDA allocations.\n        cap = max(1,len(indices)//2)\n        self.caps[kernel,bucket] = cap\n        gc.collect()\n        self.encoder.torch.cuda.empty_cache()\n        self.report(f\'QWEN_BATCH_OOM: token bucket {bucket}; retry same texts with batch {cap}\',flush=True)\n        return np.concatenate([self._forward(texts,indices[i:i+cap],lengths,kernel)\n                               for i in range(0,len(indices),cap)],axis=0)\n\n    def _run(self, texts, lengths, cap, kernel=\'native\', budget=24_576):\n        result = np.empty((len(texts),self.dimension),np.float32)\n        with self._attention(kernel):\n            for indices in token_groups(lengths,cap,budget):\n                values = self._forward(texts,indices,lengths,kernel)\n                validate_vectors(values,len(indices),self.dimension)\n                result[indices] = values\n        return result\n\n    def _calibrate(self, texts, lengths):\n        self.attempted = True\n        started = self.clock()\n        order = sorted(range(len(texts)),key=lambda i:lengths[i])\n        indices = sorted({order[int(i)] for i in np.linspace(0,len(order)-1,min(64,len(order)))})\n        sample = [texts[i] for i in indices]\n        sample_lengths = [lengths[i] for i in indices]\n        self.report(f\'QWEN_TUNING: {len(sample)} existing task texts; native batch 16 versus token batches 16/32/64\',flush=True)\n        # Warm up only existing task texts. These vectors are never published.\n        self.native.encode(sample[:4],batch_size=16)\n        tick = self.clock()\n        reference = np.asarray(self.native.encode(sample,batch_size=16),dtype=np.float32)\n        baseline = max(self.clock()-tick,1e-9)\n        validate_vectors(reference,len(sample),self.dimension)\n        best, trials = baseline, []\n        candidates=[(16,\'native\',24_576),(32,\'native\',24_576),(64,\'native\',24_576)]\n        if self.attention_tuning:\n            candidates.append((64,\'native\',49_152))\n            setter,getter=self._attention_api()\n            if setter is not None and not getter():\n                candidates.insert(1,(32,\'fp16_math\',24_576))\n                candidates.insert(2,(64,\'fp16_math\',49_152))\n        for cap,kernel,budget in candidates:\n            if self.clock()-started > (120 if self.attention_tuning else 90) or (self.deadline is not None and time.monotonic()+180 >= self.deadline):\n                break\n            tick = self.clock()\n            # Include token-length preparation in the candidate measurement.\n            self.lengths(sample)\n            try:\n                values = self._run(sample,sample_lengths,cap,kernel,budget)\n            except ValueError:\n                trials.append({\'batch_cap\':cap,\'kernel\':kernel,\'compatible\':False,\'reason\':\'vector_validation_failed\'})\n                continue\n            seconds = max(self.clock()-tick,1e-9)\n            compatible, agreement = vector_agreement(reference,values)\n            trial = {\'batch_cap\':cap,\'kernel\':kernel,\'max_padded_tokens\':budget,\n                     \'seconds\':seconds,\'compatible\':compatible,**agreement}\n            trials.append(trial)\n            self.report(f\'QWEN_TUNING_TRIAL: {kernel} cap {cap} | {seconds:.2f}s | compatible {compatible} | native {baseline:.2f}s\',flush=True)\n            if compatible and seconds < best and seconds <= baseline/1.05:\n                best, self.selected = seconds, cap\n                self.selected_kernel,self.selected_budget=kernel,budget\n        confirmation=None\n        if self.selected is not None and self.attention_tuning:\n            # A second comparison prevents choosing a single noisy timing win.\n            tick=self.clock()\n            ref2=np.asarray(self.native.encode(sample,batch_size=16),dtype=np.float32)\n            baseline2=max(self.clock()-tick,1e-9)\n            tick=self.clock()\n            self.lengths(sample)\n            try:\n                values=self._run(sample,sample_lengths,self.selected,self.selected_kernel,self.selected_budget)\n                seconds2=max(self.clock()-tick,1e-9)\n                compatible,agreement=vector_agreement(ref2,values)\n            except ValueError:\n                compatible,agreement,seconds2=False,{\'reason\':\'vector_validation_failed\'},baseline2\n            confirmation={\'compatible\':compatible,\'native_seconds\':baseline2,\'selected_seconds\':seconds2,**agreement}\n            if not compatible or seconds2>baseline2/1.05:\n                self.selected,self.selected_kernel,self.selected_budget=None,\'native\',24_576\n                best=baseline\n            else:\n                baseline,best=baseline2,seconds2\n        self.safe_batch_size = self.selected or 16\n        record = {\'policy\':POLICY,\'encoder\':self.identity,\'mode\':\'token_batches\' if self.selected else \'native\',\n            \'sample_texts\':len(sample),\'baseline_seconds\':baseline,\'selected_seconds\':best,\n            \'batch_cap\':self.safe_batch_size,\'kernel\':self.selected_kernel,\n            \'max_padded_tokens\':self.selected_budget,\'trials\':trials,\'confirmation\':confirmation,\n            \'sample_tokens\':sum(sample_lengths),\'sample_token_range\':[min(sample_lengths),max(sample_lengths)],\n            \'measured_sample_speedup\':baseline/best,\'oom_backoffs\':self.ooms,\n            \'baseline_texts_per_second\':len(sample)/baseline,\'selected_texts_per_second\':len(sample)/best,\n            \'scope\':\'Startup canary, not a full-corpus quality or throughput certification.\'}\n        atomic_json(self.path,record)\n        self.report(f\'QWEN_TUNING_SELECTED: {record["mode"]} {self.selected_kernel} | cap {self.safe_batch_size} | measured sample speedup {baseline/best:.2f}x | {len(sample)/best:.2f} texts/s | report {self.path}\',flush=True)\n\n    def encode(self, texts, *, batch_size=16):\n        if not texts:\n            return np.empty((0,self.dimension),np.float32)\n        if not self.attempted and len(texts) >= 32:\n            self._calibrate(texts,self.lengths(texts))\n        if self.selected is None:\n            return self.native.encode(texts,batch_size=16)\n        lengths=self.lengths(texts)\n        try:\n            return self._run(texts,lengths,self.selected,self.selected_kernel,self.selected_budget)\n        except ValueError:\n            # Only the current uncommitted task is retried if later inputs expose\n            # a numerical issue; previous vector blocks remain unchanged.\n            self.selected,self.selected_kernel,self.safe_batch_size=None,\'native\',16\n            from vietmedbridge.artifacts import read_json\n            if self.path.is_file():\n                saved=read_json(self.path)\n                saved.update(active_mode=\'native\',runtime_fallback=\'vector_validation_failed\')\n                atomic_json(self.path,saved)\n            self.report(\'QWEN_TUNING_FALLBACK: native batch 16 for current uncommitted texts; previous checkpoints retained.\',flush=True)\n            return self.native.encode(texts,batch_size=16)\n\n\nclass TokenBatchOwner:\n    def __init__(self, owner, report_path, *, deadline=None, attention_tuning=False):\n        self.owner, self.path, self.deadline = owner, report_path, deadline\n        self.attention_tuning=attention_tuning\n\n    def for_role(self, role):\n        native = self.owner.for_role(role)\n        if role != \'corpus\' or native.identity.get(\'model_id\') != \'Qwen/Qwen3-Embedding-8B\':\n            return native\n        return TokenBatchEncoder(native,self.path,deadline=self.deadline,attention_tuning=self.attention_tuning)\n\n    def close(self):\n        self.owner.close()\n\n\ndef benchmark_texts(directory,library):\n    """Sample three bounded Qwen task slices; never walk corpus/vector files."""\n    import pyarrow.parquet as pq\n    from vietmedbridge.artifacts import digest_json,verify_file\n    from vietmedbridge.content_embeddings import key\n    tasks=[t for t in library[\'tasks\'] if t[\'family\']==\'qwen\']\n    if not tasks:\n        raise ValueError(\'Library has no Qwen task to benchmark.\')\n    selected=[tasks[i] for i in sorted({0,len(tasks)//2,len(tasks)-1})]\n    values=[]\n    for task in selected:\n        if Path(task[\'asset\']).name!=task[\'asset\']:\n            raise ValueError(\'Unsafe benchmark text asset path.\')\n        path=Path(directory)/task[\'asset\']\n        verify_file(path,task[\'sha256\'])\n        table=pq.read_table(path,columns=[\'id\',\'text\'])\n        if len(table)!=library[\'assets\'][task[\'asset\']][\'rows\']:\n            raise ValueError(\'Benchmark asset row count changed.\')\n        rows=table.slice(task[\'offset\'],task[\'rows\']).to_pylist()\n        if (len(rows)!=task[\'rows\'] or digest_json(sorted(r[\'id\'] for r in rows))!=task[\'keys_digest\']\n                or any(r[\'id\']!=key(r[\'text\']) for r in rows)):\n            raise ValueError(\'Benchmark task text/key binding changed.\')\n        indices=sorted({int(i) for i in np.linspace(0,len(rows)-1,min(24,len(rows)))})\n        values.extend(rows[i][\'text\'] for i in indices)\n    return values\n\n\ndef benchmark(directory,checkout,output,worker_id):\n    import torch\n    from full_queue_embeddings import validate_library\n    from vietmedbridge.artifacts import read_json\n    from vietmedbridge.portable_embeddings import embedding_runtime\n    from vietmedbridge.qwen_models import TorchQwenEncoder\n    from vietmedbridge.qwen_execution import configure_embedding_execution\n    job=validate_library(directory,checkout=checkout)\n    if not torch.cuda.is_available() or embedding_runtime()!=job[\'runtime\']:\n        raise RuntimeError(\'Benchmark must use the original producer CUDA runtime.\')\n    texts=benchmark_texts(directory,job)\n    owner=TorchQwenEncoder(job[\'config\'][\'second_dense\'])\n    try:\n        native=owner.for_role(\'corpus\')\n        expected=job[\'models\'][\'qwen\']\n        if native.identity!=expected[\'encoder\'] or native.dimension!=expected[\'dimension\']:\n            raise ValueError(\'Benchmark encoder differs from the existing vector producer.\')\n        configure_embedding_execution(native)\n        output=Path(output)\n        path=output/f\'worker-{worker_id}.json\'\n        tuned=TokenBatchEncoder(native,path,attention_tuning=True,deadline=time.monotonic()+900)\n        result=tuned.encode(texts,batch_size=64)\n        validate_vectors(result,len(texts),native.dimension)\n        report=read_json(path) if path.is_file() else {\'mode\':\'native\',\'reason\':\'not_enough_sample_texts\'}\n        operators=[]\n        try:\n            with torch.profiler.profile(activities=[torch.profiler.ProfilerActivity.CPU]) as profile:\n                native.encode([texts[0],texts[-1]],batch_size=2)\n            operators=sorted({item.key for item in profile.key_averages() if \'scaled_dot_product\' in item.key})\n        except RuntimeError:\n            operators=[\'profiler_unavailable\']\n        report.update(state=\'BENCHMARK_COMPLETE_NO_TASKS_COMMITTED\',job=job[\'manifest_sha256\'],\n            gpu=torch.cuda.get_device_name(0),worker_id=worker_id,validated_texts=len(texts),\n            tasks_committed=0,native_sdpa_operators=operators)\n        atomic_json(path,report)\n        import json\n        print(\'QWEN_BENCHMARK_RESULT:\',json.dumps({k:v for k,v in report.items() if k!=\'encoder\'},ensure_ascii=False),flush=True)\n        return report\n    finally:\n        owner.close()\n\n\nif __name__==\'__main__\':\n    import argparse\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'--job\',required=True)\n    parser.add_argument(\'--checkout\',required=True)\n    parser.add_argument(\'--output\',required=True)\n    parser.add_argument(\'--worker-id\',type=int,required=True)\n    args=parser.parse_args()\n    benchmark(args.job,args.checkout,args.output,args.worker_id)\n',encoding='utf-8')

(WORK_DIR/'kaggle_output_transfer.py').write_text('"""Fetch pinned notebook Outputs without enumerating every output-file page."""\nfrom concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait\nfrom contextlib import contextmanager, nullcontext\nfrom email.utils import parsedate_to_datetime\nimport os\nfrom pathlib import Path, PurePosixPath\nimport time\n\nfrom requests.exceptions import ChunkedEncodingError, ConnectionError, SSLError, Timeout\n\nfrom vietmedbridge.artifacts import atomic_json, digest_json\n\n\ndef _request():\n    from kagglesdk.kernels.types.kernels_api_service import ApiGetKernelRequest\n    return ApiGetKernelRequest()\n\n\ndef retryable_tls_alert(error):\n    """Retry the observed protocol alert, never certificate validation failures."""\n    if not isinstance(error, SSLError):\n        return False\n    reason = str(error).upper()\n    if any(value in reason for value in (\'CERTIFICATE_VERIFY_FAILED\', \'CERTIFICATE VERIFICATION\',\n                                        \'HOSTNAME MISMATCH\', \'CERTIFICATE EXPIRED\')):\n        return False\n    return \'TLSV1_ALERT_DECODE_ERROR\' in reason\n\n\ndef retry_kaggle(operation, *, report=print, attempts=6, sleep=time.sleep):\n    """Retry a pinned transfer after throttling, server or interrupted-stream errors.\n\n    The Hub downloader retains the partial file. Calling the same operation with\n    the same cache lets it use HTTP Range when the server advertises support.\n    Do not remove partial payloads or mark them complete here.\n    """\n    for attempt in range(attempts):\n        try:\n            return operation()\n        except Exception as error:\n            response = getattr(error, \'response\', None)\n            status = getattr(response, \'status_code\', None)\n            interrupted = (status is None and isinstance(error, (ChunkedEncodingError, ConnectionError, Timeout))\n                           and (not isinstance(error, SSLError) or retryable_tls_alert(error)))\n            if (not interrupted and status not in (429, 500, 502, 503, 504)) or attempt == attempts-1:\n                raise\n            value = (getattr(response, \'headers\', {}) or {}).get(\'Retry-After\')\n            delay = min(15 * 2**attempt, 120)\n            if value:\n                try:\n                    delay = max(1, float(value))\n                except (TypeError, ValueError):\n                    try:\n                        delay = max(1, parsedate_to_datetime(value).timestamp()-time.time())\n                    except (TypeError, ValueError, OverflowError):\n                        pass\n            if delay > 300:\n                raise RuntimeError(\'Kaggle requests a cooldown longer than five minutes; \'\n                                   \'retain this runtime/cache and retry after the server cooldown.\') from error\n            reason = f\'network {type(error).__name__}\' if interrupted else f\'HTTP {status}\'\n            report(f\'KAGGLE_RETRY {reason} | same pinned resource; partial cache retained | wait {delay:.0f}s \'\n                   f\'| attempt {attempt+1}/{attempts-1}\')\n            while delay > 0:\n                step = min(30, delay)\n                sleep(step)\n                delay -= step\n                if delay > 0:\n                    report(f\'KAGGLE_COOLDOWN: {delay:.0f}s remaining; no new API request yet\')\n\n\nretry_kaggle._handles_tls_decode = True\n\n\n@contextmanager\ndef _cache_at(path):\n    """Keep large temporary downloads on Colab, without copying a second payload."""\n    name = \'KAGGLEHUB_CACHE\'\n    prior = os.environ.get(name)\n    os.environ[name] = str(path)\n    try:\n        yield\n    finally:\n        if prior is None:\n            os.environ.pop(name, None)\n        else:\n            os.environ[name] = prior\n\n\nclass SavedNotebookOutputs:\n    def __init__(self, api, handle, work, *, project=None, report=print):\n        self.api, self.handle, self.report = api, handle, report\n        owner, slug, *version = handle.split(\'/\')\n        if version:\n            number = int(version[0])\n        else:\n            def metadata():\n                request = _request()\n                request.user_name, request.kernel_slug = owner, slug\n                with api.build_kaggle_client() as client:\n                    return client.kernels.kernels_api_client.get_kernel(request)\n            response = retry_kaggle(metadata, report=report)\n            number = response.metadata.current_version_number\n        if type(number) is not int or number <= 0:\n            raise ValueError(\'Kaggle did not return a valid saved notebook version; no output download started.\')\n        self.pinned = f\'{owner}/{slug}/versions/{number}\'\n        self.cache = Path(work)/\'kagglehub-output-cache\'\n        root = Path(project)/\'data_staging\' if project is not None else Path(work)\n        self.record = root/\'kaggle_output_downloads\'/(digest_json(self.pinned)[:20]+\'.json\')\n        self._saved(\'VERSION_PINNED\')\n        report(f\'KAGGLE_OUTPUT_VERSION: {self.pinned} | direct manifest, no output-page scan\')\n\n    def _saved(self, state, **details):\n        atomic_json(self.record, {\'handle\': self.handle, \'pinned_handle\': self.pinned,\n                                 \'state\': state, **details})\n\n    def _download(self, *, path=None, force=False, cache_active=False):\n        import kagglehub\n        def transfer():\n            with (nullcontext() if cache_active else _cache_at(self.cache)):\n                # The public Hub resolver downloads a single named file, or the\n                # full archived bundle. It does not list output pages for these.\n                return Path(kagglehub.notebook_output_download(self.pinned, path=path,\n                                                               force_download=force))\n        return retry_kaggle(transfer, report=self.report)\n\n    def manifest(self, relative):\n        name = PurePosixPath(relative)\n        if (not relative or name.is_absolute() or \'..\' in name.parts or \'\\\\\' in relative\n                or \':\' in relative or name.as_posix() != relative):\n            raise ValueError(\'Unsafe saved manifest path.\')\n        self._saved(\'FETCHING_MANIFEST\', manifest_path=relative)\n        path = self._download(path=relative, force=True)\n        if not path.is_file():\n            raise RuntimeError(\'Kaggle did not return the saved result manifest; keep the failed version.\')\n        self._saved(\'MANIFEST_DOWNLOADED\', manifest_path=relative)\n        return path\n\n    def all(self, relative, manifest):\n        # A large dynamically generated archive can repeatedly lose its stream\n        # without advertising Range support. The manifest already names every\n        # asset needed by native install, so neither an archive nor a listing\n        # request is necessary. Completed named files retain Hub cache markers.\n        self._saved(\'FETCHING_NAMED_FILES\', manifest_sha256=manifest[\'manifest_sha256\'])\n        target = self._download(path=relative)\n        root = target.parent.resolve()\n        required = [\'job.json\']\n        if manifest.get(\'execution_plan\'):\n            required.append(\'execution-plan.json\')\n        required.extend(block[\'path\'] for block in manifest[\'blocks\'])\n        required = list(dict.fromkeys(required))\n        # Validate every path before starting additional remote requests.\n        for name in required:\n            pure = PurePosixPath(name)\n            if (not name or pure.is_absolute() or \'..\' in pure.parts or \'\\\\\' in name\n                    or \':\' in name or pure.as_posix() != name):\n                raise ValueError(\'Unsafe saved vector path.\')\n            path = (root/name).resolve()\n            if not path.is_relative_to(root):\n                raise ValueError(\'Saved vector path escapes the result directory.\')\n        remote_root = PurePosixPath(relative).parent\n        finished = 0\n        total = len(required)\n        def fetch(name):\n            local = self._download(path=(remote_root/name).as_posix(), cache_active=True)\n            if not local.is_file() or local.resolve() != (root/name).resolve():\n                raise RuntimeError(f\'Kaggle did not return required output file: {name}\')\n            return name\n        self.report(f\'KAGGLE_OUTPUT_FILES: 0/{total:,} | two named transfers; no archive or output-page scan\')\n        # Set the process-wide cache once, outside the pool. Per-thread changes\n        # of KAGGLEHUB_CACHE could restore each other\'s environment mid-transfer.\n        try:\n            with _cache_at(self.cache), ThreadPoolExecutor(max_workers=2) as pool:\n                names = iter(required)\n                pending = {pool.submit(fetch, name) for name in [next(names, None), next(names, None)]\n                           if name is not None}\n                while pending:\n                    done, pending = wait(pending, return_when=FIRST_COMPLETED)\n                    for future in done:\n                        future.result()\n                        finished += 1\n                        if finished % 32 == 0 or finished == total:\n                            self.report(f\'KAGGLE_OUTPUT_FILES: {finished:,}/{total:,} ready in cache \'\n                                        \'| native checksums checked during import\')\n                        if finished % 128 == 0:\n                            self._saved(\'FETCHING_NAMED_FILES\', completed_files=finished, total_files=total,\n                                        manifest_sha256=manifest[\'manifest_sha256\'])\n                    for _ in done:\n                        name = next(names, None)\n                        if name is not None:\n                            pending.add(pool.submit(fetch, name))\n        except Exception:\n            self._saved(\'NAMED_FILES_INTERRUPTED\', completed_files=finished, total_files=total,\n                        manifest_sha256=manifest[\'manifest_sha256\'])\n            raise\n        self._saved(\'BUNDLE_READY_FOR_NATIVE_VALIDATION\', manifest_sha256=manifest[\'manifest_sha256\'])\n        return target\n',encoding='utf-8')

(WORK_DIR/'kaggle_output_paged.py').write_text('"""Resume cached outputs via paged signed URLs, avoiding one API call per asset."""\nfrom concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait\nfrom pathlib import Path, PurePosixPath\nimport os\nimport re\nimport time\nfrom urllib.parse import urlsplit\n\nimport requests\n\nfrom vietmedbridge.artifacts import atomic_json, digest_json, read_json, sha256_file\n\n\ndef _page_request():\n    from kagglesdk.kernels.types.kernels_api_service import ApiListKernelSessionOutputRequest\n    return ApiListKernelSessionOutputRequest()\n\n\ndef _asset(url, destination, checksum):\n    """Publish one complete file; keep a resumable partial on network failure."""\n    if urlsplit(url).scheme != \'https\' or not urlsplit(url).netloc:\n        raise ValueError(\'Kaggle returned an unsafe download URL.\')\n    destination = Path(destination)\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    partial = destination.with_name(destination.name+\'.vmb-part\')\n    start = partial.stat().st_size if partial.is_file() else 0\n    headers = {\'Accept-Encoding\':\'identity\'}\n    if start:\n        headers[\'Range\'] = f\'bytes={start}-\'\n    with requests.get(url, stream=True, timeout=(15,60), headers=headers) as response:\n        response.raise_for_status()\n        if start and response.status_code == 206:\n            match = re.fullmatch(r\'bytes (\\d+)-(\\d+)/(\\d+)\', response.headers.get(\'Content-Range\',\'\'))\n            if not match or int(match[1]) != start or int(match[2])+1 != int(match[3]):\n                raise ValueError(\'Invalid Range response; partial output was retained.\')\n            expected = int(match[3])\n            mode = \'ab\'\n        elif response.status_code == 200:\n            # A server that ignores Range must replace only the temporary file.\n            start, mode = 0, \'wb\'\n            length = response.headers.get(\'Content-Length\')\n            expected = int(length) if length and length.isdigit() else None\n        else:\n            raise ValueError(\'Unexpected download response; no checkpoint published.\')\n        with partial.open(mode) as stream:\n            for block in response.iter_content(1024*1024):\n                if block:\n                    stream.write(block)\n        if expected is not None and partial.stat().st_size != expected:\n            raise requests.exceptions.ChunkedEncodingError(\'Output stream ended before the declared size\')\n    if checksum and sha256_file(partial) != checksum:\n        raise ValueError(\'Downloaded vector checksum differs; no checkpoint published.\')\n    os.replace(partial,destination)\n    return destination.stat().st_size\n\n\ndef download_paged_outputs(self, relative, manifest):\n    from kagglehub.cache import Cache\n    from kagglehub.handle import parse_notebook_handle\n    try:\n        from .kaggle_output_transfer import _cache_at, retry_kaggle\n    except ImportError:\n        from kaggle_output_transfer import _cache_at, retry_kaggle\n    owner,slug,_,version = self.pinned.split(\'/\')\n    handle = parse_notebook_handle(self.pinned)\n    with _cache_at(self.cache):\n        cache = Cache()\n        target = cache.load_from_cache(handle,relative)\n        target = Path(target) if target else self._download(path=relative)\n        root = target.parent.resolve()\n        remote_root = PurePosixPath(relative).parent\n        required = {\'job.json\':None}\n        if manifest.get(\'execution_plan\'):\n            required[\'execution-plan.json\'] = None\n        for block in manifest[\'blocks\']:\n            name,checksum = block[\'path\'],block[\'sha256\']\n            if name in required and required[name] != checksum:\n                raise ValueError(\'Conflicting output paths in manifest.\')\n            required[name] = checksum\n        for name in required:\n            pure = PurePosixPath(name)\n            if (not name or pure.is_absolute() or \'..\' in pure.parts or \'\\\\\' in name or \':\' in name\n                    or pure.as_posix() != name or not (root/name).resolve().is_relative_to(root)):\n                raise ValueError(\'Unsafe saved output path.\')\n        remote = {(remote_root/name).as_posix():name for name in required}\n        ready = {name for rel,name in remote.items() if cache.load_from_cache(handle,rel)}\n        if any(Path(cache.get_path(handle,(remote_root/name).as_posix())).resolve() != (root/name).resolve()\n               for name in ready):\n            raise ValueError(\'Output cache location differs from the pinned manifest.\')\n        total,initial = len(required),len(ready)\n        pending = set(required)-ready\n        started = time.monotonic()\n        last_report = started\n        cursor_path = Path(self.cache)/\'vmb-output-pages\'/(digest_json([self.pinned,manifest[\'manifest_sha256\']])[:20]+\'.json\')\n        cursor = read_json(cursor_path) if cursor_path.is_file() else {}\n        valid_cursor = (cursor.get(\'pinned\') == self.pinned and cursor.get(\'manifest\') == manifest[\'manifest_sha256\']\n                        and set(cursor.get(\'processed\',[])) <= ready)\n        token = cursor.get(\'next_token\',\'\') if valid_cursor else \'\'\n        processed = set(cursor.get(\'processed\',[])) if valid_cursor else set()\n        pages = int(cursor.get(\'pages\',0)) if valid_cursor else 0\n        seen = set()\n\n        def progress(force=False):\n            nonlocal last_report\n            now = time.monotonic()\n            if not force and now-last_report < 15:\n                return\n            done = len(ready)-initial\n            eta = f\'{(total-len(ready))*(now-started)/done/60:.1f} min\' if done >= 8 else \'measuring\'\n            self.report(f\'KAGGLE_TRANSFER: {len(ready):,}/{total:,} ready | cached at start {initial:,} \'\n                        f\'| missing {len(pending):,} | download ETA {eta}; Drive import follows\')\n            self._saved(\'FETCHING_PAGED_FILES\',completed_files=len(ready),total_files=total,\n                        missing_files=len(pending),cached_at_start=initial,pages=pages,\n                        manifest_sha256=manifest[\'manifest_sha256\'])\n            last_report = now\n\n        progress(True)\n        if pending:\n            while True:\n                def page():\n                    request = _page_request()\n                    request.user_name,request.kernel_slug = owner,slug\n                    request.page_size,request.page_token,request.version_label = 200,token,\'v\'+version\n                    with self.api.build_kaggle_client() as client:\n                        return client.kernels.kernels_api_client.list_kernel_session_output(request)\n                response = retry_kaggle(page,report=self.report)\n                pages += 1\n                batch = {}\n                for item in response.files or []:\n                    name = remote.get(item.file_name)\n                    if name in pending:\n                        batch[name] = item.url\n                    if name is not None:\n                        processed.add(name)\n                self.report(f\'KAGGLE_URL_PAGE: {pages} | page size 200 | {len(batch)} missing assets on this page\')\n                with ThreadPoolExecutor(max_workers=4) as pool:\n                    assets = iter(batch.items())\n                    jobs = {}\n                    def enqueue():\n                        item = next(assets,None)\n                        if item is not None:\n                            name,url = item\n                            future = pool.submit(retry_kaggle,\n                                lambda:_asset(url,root/name,required[name]),report=self.report)\n                            jobs[future] = name\n                    for _ in range(4):\n                        enqueue()\n                    while jobs:\n                        done,_ = wait(jobs,timeout=15,return_when=FIRST_COMPLETED)\n                        for future in done:\n                            name = jobs.pop(future)\n                            future.result()\n                            cache.mark_as_complete(handle,(remote_root/name).as_posix())\n                            ready.add(name)\n                            pending.remove(name)\n                        for _ in done:\n                            enqueue()\n                        progress()\n                following = response.next_page_token or \'\'\n                if following and (not isinstance(following,str) or following == token or following in seen):\n                    raise RuntimeError(\'Kaggle repeated an output-page token; cached files retained.\')\n                seen.add(token)\n                atomic_json(cursor_path,{\'pinned\':self.pinned,\'manifest\':manifest[\'manifest_sha256\'],\n                    \'next_token\':following,\'pages\':pages,\'processed\':sorted(processed)})\n                progress(True)\n                if not pending:\n                    break\n                if not following:\n                    self._saved(\'PAGED_OUTPUTS_MISSING\',missing_files=len(pending),total_files=total,\n                                manifest_sha256=manifest[\'manifest_sha256\'])\n                    raise RuntimeError(f\'{len(pending)} required files absent from saved Outputs; no import started.\')\n                token = following\n        self._saved(\'BUNDLE_READY_FOR_NATIVE_VALIDATION\',completed_files=total,total_files=total,\n                    manifest_sha256=manifest[\'manifest_sha256\'])\n        self.report(f\'KAGGLE_TRANSFER_COMPLETE: {total:,}/{total:,}; native validation and Drive import next\')\n        return target\n\n\ndef install_paged_output_transfer():\n    try:\n        from . import kaggle_output_transfer as transport\n    except ImportError:\n        import kaggle_output_transfer as transport\n    transport.SavedNotebookOutputs.all = download_paged_outputs\n\n\nif __name__ == \'__main__\':\n    install_paged_output_transfer()\n    print(\'Đã bật tải theo trang URL, giữ cache. Chạy lại riêng cell 3 trong cùng runtime.\',flush=True)\n',encoding='utf-8')

sys.modules.pop('full_queue_embeddings',None)
sys.modules.pop('kaggle_output_transfer',None)
sys.modules.pop('kaggle_output_paged',None)
for module_name in tuple(sys.modules):
    if module_name == 'kagglehub' or module_name.startswith('kagglehub.'):
        sys.modules.pop(module_name,None)
importlib.invalidate_caches()
from kaggle_output_paged import install_paged_output_transfer
install_paged_output_transfer()
import full_queue_embeddings as queue
def queue_flow(work,*,accounts=2,gpu_count=2,batch_dir=None):
    return queue.FullQueueKaggle(PROJECT,work,identities,accounts=accounts,gpu_count=gpu_count,
        batch_loader=lambda project:load_full_batch(project,batch_dir or BATCH_DIR,minimum_ids=802028))


In [ ]:
#@title 3. Tạo plan Colab 4×1 GPU; giữ riêng task Kaggle acc 2 đang chạy
CONTINUE_FROM_PLAN = "63b6fe57cbf42fbf355f4d1c476a1fa68801c4b91d03c65855274f35861dec15"
EXPECTED_LIBRARY_JOB = "d6ab9a432ac64e5aadb39a826b7dc2176094cc66d4f4e3d9e17c121e071c0711"
EXECUTION_ACCOUNTS = 4
EXECUTION_GPUS_PER_ACCOUNT = 1

if KAGGLE_NOTEBOOK_OUTPUT.strip():
    from kaggle.api.kaggle_api_extended import KaggleApi
    from vietmedbridge.kaggle_outputs import output_handles
    api=KaggleApi()
    api.authenticate()
    roots=[p.relative_to(PROJECT).as_posix() for p in (PROJECT/PARENT).glob('*/embeddings_v1')]
    download_work=PROJECT/'data_staging/kaggle_output_transfers'
    download_work.mkdir(parents=True,exist_ok=True)
    for handle in output_handles(KAGGLE_NOTEBOOK_OUTPUT):
        output=pipeline.download_outputs(api,handle,download_work,project=PROJECT,import_roots=roots,
            allow_failed=True,manifest_path='vmb_full_queue/result/new-data-result.json')
        saved=pipeline.checked(pipeline.read_json(output/'job.json'))
        result=pipeline.checked(pipeline.read_json(output/'new-data-result.json'))
        if result.get('execution_plan')!=CONTINUE_FROM_PLAN or result.get('account_id')!=2:
            raise ValueError('Chỉ nhập Outputs acc 2 của plan đang chạy 63b6; không nhập nhầm lượt khác.')
        if saved.get('manifest_sha256')!=EXPECTED_LIBRARY_JOB:
            raise ValueError('Outputs thuộc input library khác; giữ nguyên checkpoint.')
        old_flow=queue_flow(WORK_DIR/'cpu',accounts=4,gpu_count=2,batch_dir=saved['source_batch_dir'])
        print(json.dumps(old_flow.install(output,receipt_api=api_source),ensure_ascii=False,indent=2),flush=True)

flow=queue_flow(WORK_DIR/'cpu',accounts=4,gpu_count=2,batch_dir=BATCH_DIR)
pointer=flow.root/'input_library.json'
if not pointer.is_file():
    raise FileNotFoundError('Drive chưa có full input library đã freeze; dừng để không tạo lại từ corpus.')
library=queue._validate_library_header(queue.read_json(pointer))
if library.get('manifest_sha256')!=EXPECTED_LIBRARY_JOB:
    raise ValueError('Drive full input library khác job đã chốt; không tạo plan mới.')
active_path=flow.root/'active_execution_plan.json'
active=queue.checked(queue.read_json(active_path))
if active.get('plan')==CONTINUE_FROM_PLAN:
    target,plan=flow.create_plan(library,receipt_api=api_source,
        execution_accounts=EXECUTION_ACCOUNTS,execution_gpu_count=EXECUTION_GPUS_PER_ACCOUNT,
        continue_completed_from=CONTINUE_FROM_PLAN)
else:
    plan=flow.saved_plan(active.get('plan'))
    if (plan.get('team')!={'accounts':EXECUTION_ACCOUNTS,'gpu_count':EXECUTION_GPUS_PER_ACCOUNT}
            or plan.get('continued_from_plan')!=CONTINUE_FROM_PLAN):
        raise RuntimeError('Drive đang có plan khác. Không thay plan sau khi worker có thể đã chạy.')
    target=flow.stage_plan(plan)
    print('COLAB_PLAN_REUSED: không tạo task assignment mới.',flush=True)
print('COLAB_PLAN_READY:',plan['manifest_sha256'],flush=True)
print('TEAM:',plan['team'],'| tasks:',plan['scheduled_tasks'],'| task IDs còn khóa Kaggle:',
    len(plan.get('reserved_task_ids',[])),'| source pending accounts:',plan.get('reservations',[]),flush=True)
print('PLAN_ON_DRIVE:',flow.root/'plans'/(plan['manifest_sha256']+'.json'),flush=True)
print('ACCOUNT_ID trên 4 Colab: 0, 1, 2, 3. Mỗi ID dùng đúng một GPU và một checkpoint folder.',flush=True)